# NeoOLAF DocRED — resume ONLY failed documents + final micro evaluation (FULLY self-contained V3)

This notebook resumes the **existing** DocRED v6.2 full-dev experiment under the exact original batch root:

`examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming`

It does **not** start a new experiment.

## Intended starting state

The previous full run requested **998 DocRED dev records**:

- **978 completed**
- **20 failed**

This notebook:

1. rebuilds the current aggregate **offline** from the existing saved artifacts;
2. lists the currently failed records;
3. verifies that every non-failed document is already completed/resumable;
4. reruns the v6.2 streaming scheduler with:
   - `resume_completed=True`
   - `retry_failed_documents=True`
5. therefore **completed documents are skipped** and only failed/unresolved documents are eligible for paid rerun;
6. uses the original frozen DocRED execution settings:
   - model `openai/gpt-oss-20b`
   - `DOCUMENT_WORKERS=4`
   - `LAYER_WORKERS=16`
   - frozen v5.1 scientific profile/guidance/evaluator
7. rebuilds the final aggregate from disk afterward;
8. reports the definitive DocRED relation/entity/endpoint micro metrics and runtime.

If some failed documents remain after one invocation, rerun this notebook: successfully recovered documents are then skipped and only the remaining failures are retried.


### V2 import hotfix
This version embeds and restores the missing `docred_native_batch_v6_2_dev_streaming.py` helper automatically. It does not modify `src/neoolaf`.

### V3 dependency fix

This version embeds the **entire exact frozen DocRED experiment helper chain** from the original v6.2 bundle: `docred_native_ablation.py`, v3, v4, v5, `docred_native_batch_v6.py`, v6.1, and v6.2. Missing helpers are restored under `examples/RAGTreeDatasets/tools`; differently-versioned local copies are backed up first. Nothing under `src/neoolaf` is modified.


In [2]:
from __future__ import annotations

import os
import sys
import json
import hashlib
import multiprocessing as mp
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents, Path(r"C:\Users\galencarmedeiro\NeoOLAF")]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the NeoOLAF repository "
        "or set the working directory to the NeoOLAF project root."
    )


def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else PROJECT_ROOT / candidate
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" + "\n".join(str(path) for path in checked)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
TOOLS_DIR.mkdir(parents=True, exist_ok=True)

for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR, NOTEBOOK_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# ---------------------------------------------------------------------------
# FULLY SELF-CONTAINED FROZEN DocRED v6.2 TOOLCHAIN
# ---------------------------------------------------------------------------
# These are the exact experiment helper files from the original
# NeoOLAF_DocRED_DevStreaming_v6_2 bundle. They are experiment-only files
# under examples/RAGTreeDatasets/tools; NOTHING under src/neoolaf is changed.
#
# This fixes the previous two failures:
#   1) missing docred_native_batch_v6_2_dev_streaming.py
#   2) missing docred_native_batch_v6_1.py
#
# It also restores the frozen v5 Layer-0--12 execution dependency chain.
FROZEN_HELPER_SOURCES = {'docred_native_ablation.py': 'from __future__ import annotations\n\n"""Notebook support for the native DocRED NeoOLAF layer ablation.\n\nThis module orchestrates existing NeoOLAF layers only. It does not add a second\nLLM extraction task, alter src/neoolaf, expose gold entities/relations to the\npipeline, or add closure facts. Gold data is loaded only after the run for\nanalysis.\n"""\n\nfrom contextlib import redirect_stderr, redirect_stdout\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any, Iterable\nimport csv\nimport json\nimport re\nimport sys\nimport threading\nimport time\nimport traceback\n\nfrom neoolaf.core.pipeline import Pipeline\nfrom neoolaf.core.pipeline_state import PipelineState\nfrom neoolaf.core.runner import Runner\nfrom neoolaf.domain.documents import Document\nfrom neoolaf.grounding.rag.base import RAGRequest, RAGResult\nfrom neoolaf.grounding.rag.types import GroundingRequest, GroundingResult, RetrievedItem\nfrom neoolaf.grounding.rag.spaces.ontology_space import OntologySpace\nfrom neoolaf.ontology.loader import SeedOntologyLoader\nfrom neoolaf.profiles.profile_loader import load_document_profile\n\nfrom neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer\nfrom neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer\nfrom neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer\nfrom neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer\nfrom neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer\nfrom neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer\nfrom neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer\nfrom neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer\nfrom neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer\nfrom neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer\nfrom neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer\nfrom neoolaf.layers.layer11_inference_completion.component import InferenceCompletionLayer\nfrom neoolaf.layers.layer12_serialization.component import SerializationLayer\n\nfrom experiments.methods.run_neoolaf import (\n    OpenAICompatibleBackend,\n    OfflineWikipediaSource,\n    OfflineWikidataSource,\n    OfflineWebSearchSource,\n    load_user_guidance,\n)\n\n\nLAYER_NAMES = [\n    "layer00_preprocessing",\n    "layer01_linguistic_expression_extraction",\n    "layer02_candidate_enrichment",\n    "layer03_candidate_typing_resolution",\n    "layer04_candidate_relation_extraction",\n    "layer05_candidate_triple_generation",\n    "layer06_concept_relation_induction",\n    "layer07_hierarchisation",\n    "layer08_axiom_schemata_extraction",\n    "layer09_general_axiom_extraction",\n    "layer10_validation_reasoning",\n    "layer11_inference_completion",\n    "layer12_serialization",\n]\n\n\ndef read_json(path: str | Path) -> Any:\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef read_jsonl(path: str | Path) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    with Path(path).open("r", encoding="utf-8") as handle:\n        for line in handle:\n            if line.strip():\n                rows.append(json.loads(line))\n    return rows\n\n\ndef write_json(path: str | Path, value: Any) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\\n", encoding="utf-8")\n    return path\n\n\ndef append_jsonl(path: str | Path, value: dict[str, Any], lock: threading.Lock | None = None) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    line = json.dumps(value, ensure_ascii=False, default=str) + "\\n"\n    if lock is None:\n        with path.open("a", encoding="utf-8") as handle:\n            handle.write(line)\n        return\n    with lock:\n        with path.open("a", encoding="utf-8") as handle:\n            handle.write(line)\n\n\ndef safe_name(value: str) -> str:\n    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")\n    return cleaned[:100] or "document"\n\n\nclass Tee:\n    def __init__(self, *streams: Any) -> None:\n        self.streams = streams\n\n    def write(self, data: str) -> int:\n        for stream in self.streams:\n            stream.write(data)\n            stream.flush()\n        return len(data)\n\n    def flush(self) -> None:\n        for stream in self.streams:\n            stream.flush()\n\n\nclass LoggedBackend:\n    """Thread-safe logger around the existing OpenAI-compatible backend."""\n\n    def __init__(self, backend: OpenAICompatibleBackend, log_dir: str | Path) -> None:\n        self.backend = backend\n        self.log_dir = Path(log_dir)\n        self.log_dir.mkdir(parents=True, exist_ok=True)\n        (self.log_dir / "responses").mkdir(parents=True, exist_ok=True)\n        self.calls_path = self.log_dir / "llm_calls.jsonl"\n        self.errors_path = self.log_dir / "llm_errors.jsonl"\n        self.lock = threading.Lock()\n        self.call_index = 0\n\n    def chat(self, model: str, messages: list[dict[str, str]], temperature: float = 0.0, **_: Any) -> str:\n        with self.lock:\n            self.call_index += 1\n            call_index = self.call_index\n        started = time.time()\n        meta = {\n            "call_index": call_index,\n            "model": model,\n            "temperature": temperature,\n            "message_count": len(messages),\n            "system_chars": sum(len(m.get("content", "")) for m in messages if m.get("role") == "system"),\n            "user_chars": sum(len(m.get("content", "")) for m in messages if m.get("role") != "system"),\n            "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n        }\n        try:\n            response = self.backend.chat(model=model, messages=messages, temperature=temperature)\n            response_path = self.log_dir / "responses" / f"response_{call_index:04d}.txt"\n            response_path.write_text(response, encoding="utf-8")\n            append_jsonl(self.calls_path, {\n                **meta,\n                "status": "ok",\n                "elapsed_seconds": round(time.time() - started, 3),\n                "response_chars": len(response),\n                "response_path": str(response_path),\n            }, self.lock)\n            return response\n        except Exception as exc:\n            append_jsonl(self.errors_path, {\n                **meta,\n                "status": "error",\n                "elapsed_seconds": round(time.time() - started, 3),\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n            }, self.lock)\n            raise\n\n    @staticmethod\n    def extract_json(text: str) -> Any:\n        return OpenAICompatibleBackend.extract_json(text)\n\n\nclass OntologyOnlyRAGAdapter:\n    """Balanced deterministic retrieval over the supplied DocRED ontology.\n\n    The repository\'s ``OntologySpace.retrieve`` currently appends classes before\n    properties and then truncates the combined list to ``top_k``. With a normal\n    ontology this can hide every property from relation-oriented prompts. This\n    notebook adapter keeps the same NeoOLAF grounding contracts but retrieves\n    classes and properties separately, guaranteeing property evidence for\n    Layers 2-4 without changing ``src/neoolaf`` or adding another LLM task.\n    """\n\n    name = "balanced_ontology_only"\n\n    def __init__(\n        self,\n        seed_ontology: Any,\n        log_path: str | Path,\n        top_k: int = 8,\n        query_expansions: dict[str, list[str]] | None = None,\n    ) -> None:\n        self.seed_ontology = seed_ontology\n        self.space = OntologySpace(seed_ontology)\n        self.log_path = Path(log_path)\n        self.top_k = max(2, int(top_k))\n        self.query_expansions = {\n            str(key).lower().strip(): [str(value) for value in values]\n            for key, values in (query_expansions or {}).items()\n            if key and isinstance(values, list)\n        }\n        self.lock = threading.Lock()\n\n    def _expanded_queries(self, query: str) -> list[str]:\n        original = str(query or "").strip()\n        lowered = original.lower()\n        values: list[str] = []\n        # Put profile-provided ontology labels first so exact schema matches are\n        # not displaced by weak lexical neighbors of the surface phrase.\n        for trigger, expansions in self.query_expansions.items():\n            if trigger and trigger in lowered:\n                values.extend(expansions)\n        values.append(original)\n        return list(dict.fromkeys(value for value in values if value))\n\n    @staticmethod\n    def _class_item(cls: Any) -> RetrievedItem:\n        return RetrievedItem(\n            source="ontology",\n            content=f"Class: {cls.label}. {cls.description or \'\'}".strip(),\n            metadata={\n                "type": "class",\n                "uri": cls.uri,\n                "label": cls.label,\n                "alt_labels": getattr(cls, "alt_labels", []),\n                "parents": cls.parent_uris,\n                "children": cls.child_uris,\n            },\n            reference=cls.uri,\n        )\n\n    @staticmethod\n    def _property_item(prop: Any) -> RetrievedItem:\n        property_id = str(prop.uri).rstrip("/").split("/")[-1]\n        return RetrievedItem(\n            source="ontology",\n            content=(\n                f"Property: {property_id} : {prop.label}. {prop.description or \'\'} "\n                f"Domain URIs: {\', \'.join(prop.domain_uris or []) or \'unspecified\'}. "\n                f"Range URIs: {\', \'.join(prop.range_uris or []) or \'unspecified\'}."\n            ).strip(),\n            metadata={\n                "type": "property",\n                "uri": prop.uri,\n                "property_id": property_id,\n                "label": prop.label,\n                "alt_labels": getattr(prop, "alt_labels", []),\n                "domain_uris": prop.domain_uris,\n                "range_uris": prop.range_uris,\n                "parents": prop.parent_uris,\n                "children": prop.child_uris,\n            },\n            reference=prop.uri,\n        )\n\n    def _items(\n        self,\n        query: str,\n        top_k: int | None = None,\n        layer_name: str | None = None,\n    ) -> tuple[list[RetrievedItem], list[str]]:\n        top_k = max(2, int(top_k or self.top_k))\n        relation_focused = str(layer_name or "").endswith(\n            ("candidate_relation_extraction", "concept_relation_induction")\n        )\n        property_budget = max(1, int(round(top_k * (0.75 if relation_focused else 0.5))))\n        class_budget = max(1, top_k - property_budget)\n\n        expanded = self._expanded_queries(query)\n        retriever = self.space.retriever\n        if retriever is None:\n            return [], expanded\n\n        classes: list[Any] = []\n        properties: list[Any] = []\n        # First collect exact matches for every profile expansion.\n        for expanded_query in expanded:\n            normalized_query = expanded_query.lower().strip()\n            for uri in self.seed_ontology.property_uris_by_label.get(normalized_query, []):\n                prop = self.seed_ontology.properties_by_uri.get(uri)\n                if prop is not None:\n                    properties.append(prop)\n            for uri in self.seed_ontology.class_uris_by_label.get(normalized_query, []):\n                cls = self.seed_ontology.classes_by_uri.get(uri)\n                if cls is not None:\n                    classes.append(cls)\n        # Then fill remaining slots with fuzzy lexical neighbors.\n        for expanded_query in expanded:\n            properties.extend(retriever.nearest_properties(expanded_query, top_k=property_budget))\n            classes.extend(retriever.nearest_classes(expanded_query, top_k=class_budget))\n\n        def dedup(values: list[Any], budget: int) -> list[Any]:\n            result = []\n            seen = set()\n            for value in values:\n                uri = str(value.uri)\n                if uri in seen:\n                    continue\n                seen.add(uri)\n                result.append(value)\n                if len(result) >= budget:\n                    break\n            return result\n\n        selected_properties = dedup(properties, property_budget)\n        selected_classes = dedup(classes, class_budget)\n        items = [self._property_item(prop) for prop in selected_properties]\n        items.extend(self._class_item(cls) for cls in selected_classes)\n        return items[:top_k], expanded\n\n    def _log(\n        self,\n        *,\n        layer_name: str | None,\n        query: str,\n        expanded_queries: list[str],\n        top_k: int,\n        items: list[RetrievedItem],\n    ) -> None:\n        append_jsonl(self.log_path, {\n            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n            "layer_name": layer_name,\n            "query": query,\n            "expanded_queries": expanded_queries,\n            "top_k": top_k,\n            "result_count": len(items),\n            "result_types": [item.metadata.get("type") for item in items],\n            "property_ids": [\n                item.metadata.get("property_id")\n                for item in items\n                if item.metadata.get("type") == "property"\n            ],\n            "references": [item.reference for item in items],\n            "labels": [item.metadata.get("label") for item in items],\n        }, self.lock)\n\n    def retrieve(self, request: RAGRequest | GroundingRequest) -> RAGResult:\n        query = str(getattr(request, "query", "") or "")\n        top_k = int(getattr(request, "top_k", self.top_k) or self.top_k)\n        layer_name = getattr(request, "layer_name", None)\n        items, expanded = self._items(query, top_k, layer_name)\n        context = "\\n".join(item.content for item in items)\n        sources = [\n            {\n                "space": item.source,\n                "text": item.content,\n                "score": item.score,\n                "reference": item.reference,\n                "metadata": item.metadata,\n            }\n            for item in items\n        ]\n        self._log(\n            layer_name=layer_name,\n            query=query,\n            expanded_queries=expanded,\n            top_k=top_k,\n            items=items,\n        )\n        return RAGResult(context=context, sources=sources, metadata={"backend": self.name})\n\n    def ground(self, request: GroundingRequest | RAGRequest) -> GroundingResult:\n        if isinstance(request, GroundingRequest):\n            grounding_request = request\n        else:\n            payload = dict(getattr(request, "metadata", {}) or {})\n            if getattr(request, "document_id", None):\n                payload.setdefault("document_id", request.document_id)\n            grounding_request = GroundingRequest(\n                layer_name=getattr(request, "layer_name", "unknown_layer"),\n                query=getattr(request, "query", ""),\n                payload=payload,\n                preferred_sources=list(getattr(request, "allowed_spaces", []) or ["ontology"]),\n                top_k=int(getattr(request, "top_k", self.top_k) or self.top_k),\n            )\n        items, expanded = self._items(\n            grounding_request.query,\n            grounding_request.top_k,\n            grounding_request.layer_name,\n        )\n        self._log(\n            layer_name=grounding_request.layer_name,\n            query=grounding_request.query,\n            expanded_queries=expanded,\n            top_k=grounding_request.top_k,\n            items=items,\n        )\n        return GroundingResult(\n            request=grounding_request,\n            selected_sources=["ontology"] if items else [],\n            retrieved_items=items,\n            grounding_summary="\\n".join(item.content for item in items),\n            merged_context={"backend": self.name, "expanded_queries": expanded},\n        )\n\ndef choose_chunk_size(text: str, max_safe_chars: int = 24000) -> int:\n    """Prefer one whole-document chunk while retaining a safety ceiling."""\n    length = len(text)\n    if length <= max_safe_chars:\n        return max(4096, length + 512)\n    return max_safe_chars\n\n\ndef build_document(record: dict[str, Any], source_path: str | Path) -> Document:\n    return Document(\n        doc_id=str(record["document_id"]),\n        source_path=str(source_path),\n        raw_text=str(record["text"]),\n    )\n\n\ndef build_pipeline(\n    *,\n    backend: LoggedBackend,\n    rag_adapter: OntologyOnlyRAGAdapter,\n    profile_config: dict[str, Any],\n    chunk_size: int,\n    workers: int = 4,\n    retry_failed_calls: int = 2,\n    retry_sleep_seconds: float = 2.0,\n    verbose: bool = True,\n) -> Pipeline:\n    """Build the standard 13-layer sequence from existing NeoOLAF components."""\n    workers = max(1, int(workers))\n    layers = [\n        PreprocessingLayer(\n            chunk_size=chunk_size,\n            overlap=0,\n            enable_chunking=True,\n            translate=False,\n            save_intermediate=True,\n            verbose=verbose,\n            profile_config=profile_config,\n        ),\n        LinguisticExpressionExtractionLayer(\n            backend,\n            max_chunks=1,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_backend=rag_adapter,\n            max_concurrency=1,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n            rag_enabled=False,\n        ),\n        CandidateEnrichmentLayer(\n            backend,\n            wikipedia_source=OfflineWikipediaSource(),\n            wikidata_source=OfflineWikidataSource(),\n            web_search_source=OfflineWebSearchSource(),\n            max_expressions=None,\n            use_web_search=False,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n        ),\n        CandidateTypingResolutionLayer(\n            backend,\n            max_expressions=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n        ),\n        CandidateRelationExtractionLayer(\n            backend,\n            max_relation_mentions=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n        ),\n        CandidateTripleGenerationLayer(\n            max_assertions=None,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n            save_intermediate=True,\n            verbose=verbose,\n        ),\n        ConceptRelationInductionLayer(\n            backend,\n            max_concept_inputs=None,\n            max_relation_inputs=None,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n        ),\n        HierarchisationLayer(\n            backend,\n            max_concept_pairs=None,\n            max_relation_pairs=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n        ),\n        AxiomSchemataExtractionLayer(\n            backend,\n            max_relation_schema_inputs=None,\n            max_subclass_inputs=None,\n            temperature=0.0,\n            rag_adapter=rag_adapter,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n        ),\n        GeneralAxiomExtractionLayer(\n            backend,\n            max_schema_inputs=None,\n            max_description_inputs=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n        ),\n        ValidationReasoningLayer(\n            max_triples=None,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n        ),\n        InferenceCompletionLayer(\n            max_inferred_triples=None,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_failed_calls,\n            retry_sleep_seconds=retry_sleep_seconds,\n        ),\n        SerializationLayer(\n            output_subdir="exports",\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n        ),\n    ]\n    return Pipeline(layers=layers, verbose=verbose, continue_from_last=False)\n\n\ndef run_native_pipeline(\n    *,\n    project_root: str | Path,\n    input_jsonl: str | Path,\n    ontology_path: str | Path,\n    profile_path: str | Path,\n    guidance_path: str | Path,\n    run_dir: str | Path,\n    model_name: str,\n    api_key: str,\n    host: str = "https://openrouter.ai/api/v1",\n    workers: int = 4,\n    max_tokens: int = 8192,\n    request_timeout: int = 600,\n    reasoning_effort: str = "minimal",\n    verbose: bool = True,\n) -> PipelineState:\n    """Run one document through all 13 native NeoOLAF layers."""\n    project_root = Path(project_root).resolve()\n    input_jsonl = Path(input_jsonl).resolve()\n    ontology_path = Path(ontology_path).resolve()\n    profile_path = Path(profile_path).resolve()\n    guidance_path = Path(guidance_path).resolve()\n    run_dir = Path(run_dir).resolve()\n    run_dir.mkdir(parents=True, exist_ok=True)\n    logs_dir = run_dir / "run_logs"\n    logs_dir.mkdir(parents=True, exist_ok=True)\n\n    records = read_jsonl(input_jsonl)\n    if len(records) != 1:\n        raise ValueError(f"This notebook expects exactly one input document, found {len(records)}")\n    record = records[0]\n    if "entities" in record or "relations" in record:\n        raise ValueError("Pipeline input must not contain gold entities or relations.")\n    if not api_key:\n        raise ValueError("OPENROUTER_API_KEY is not set. The key is read from the environment and is never written to artifacts.")\n\n    profile = load_document_profile(profile_path=profile_path)\n    guidance = load_user_guidance(str(guidance_path))\n    seed_ontology = SeedOntologyLoader().load(str(ontology_path))\n    if len(seed_ontology.properties_by_uri) < 90:\n        raise RuntimeError(\n            f"Only {len(seed_ontology.properties_by_uri)} ontology properties were loaded. "\n            "Use docred_redocred_neoolaf_compatible.ttl, which exposes all 96 rdf:Property predicates to the current loader."\n        )\n\n    chunk_size = choose_chunk_size(record["text"], int(profile.get("chunking.max_safe_chunk_chars", 24000)))\n    core_backend = OpenAICompatibleBackend(\n        backend_name="openrouter",\n        host=host,\n        api_key=api_key,\n        timeout=request_timeout,\n        max_tokens=max_tokens,\n        reasoning_effort=reasoning_effort,\n        exclude_reasoning=True,\n    )\n    backend = LoggedBackend(core_backend, logs_dir)\n    rag_adapter = OntologyOnlyRAGAdapter(\n        seed_ontology,\n        log_path=logs_dir / "ontology_retrieval.jsonl",\n        top_k=int(profile.get("rag.top_k", 8)),\n        query_expansions=profile.get("rag.query_expansions", {}) or {},\n    )\n    pipeline = build_pipeline(\n        backend=backend,\n        rag_adapter=rag_adapter,\n        profile_config=profile.to_state_dict(),\n        chunk_size=chunk_size,\n        workers=workers,\n        verbose=verbose,\n    )\n    state = PipelineState(\n        document=build_document(record, input_jsonl),\n        llm_model=model_name,\n        user_guidance=guidance,\n        seed_ontology=seed_ontology,\n        artifact_dir=str(run_dir),\n        profile_name=profile.name,\n        profile_config=profile.to_state_dict(),\n    )\n    runner = Runner(\n        pipeline=pipeline,\n        runs_root=str(run_dir.parent),\n        verbose=verbose,\n        max_workers=workers,\n        enable_checkpoints=True,\n        save_chunk_checkpoints=False,\n    )\n\n    manifest = {\n        "document_id": record["document_id"],\n        "title": record.get("title"),\n        "model_name": model_name,\n        "profile_name": profile.name,\n        "profile_path": str(profile_path),\n        "guidance_path": str(guidance_path),\n        "ontology_path": str(ontology_path),\n        "ontology_classes": len(seed_ontology.classes_by_uri),\n        "ontology_properties": len(seed_ontology.properties_by_uri),\n        "input_has_gold": False,\n        "chunk_size": chunk_size,\n        "whole_document_single_chunk_expected": len(record["text"]) <= chunk_size,\n        "workers": workers,\n        "max_tokens": max_tokens,\n        "anti_cheating": profile.get("anti_cheating", {}),\n        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(run_dir / "run_manifest.json", manifest)\n\n    console_log = logs_dir / "console.log"\n    errors_path = logs_dir / "pipeline_errors.jsonl"\n    started = time.time()\n    with console_log.open("w", encoding="utf-8") as log_handle:\n        tee_out = Tee(sys.stdout, log_handle)\n        tee_err = Tee(sys.stderr, log_handle)\n        try:\n            with redirect_stdout(tee_out), redirect_stderr(tee_err):\n                final_state = runner.run(state, from_layer=0, to_layer=12, run_dir=run_dir)\n        except Exception as exc:\n            append_jsonl(errors_path, {\n                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n            })\n            raise\n\n    manifest.update({\n        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "elapsed_seconds": round(time.time() - started, 3),\n        "final_counts": state_counts(final_state),\n    })\n    write_json(run_dir / "run_manifest.json", manifest)\n    return final_state\n\n\ndef state_counts(state: PipelineState) -> dict[str, int]:\n    fields = [\n        "linguistic_expressions", "enriched_expressions", "entity_candidates",\n        "relation_candidates", "attribute_candidates", "event_candidates",\n        "candidate_relation_assertions", "candidate_triples", "concept_candidates",\n        "ontology_relation_candidates", "concept_hierarchy_links", "relation_hierarchy_links",\n        "axiom_schema_candidates", "general_axiom_candidates", "completion_candidates",\n    ]\n    counts = {name: len(getattr(state, name, []) or []) for name in fields}\n    counts["validation_issues"] = len(getattr(getattr(state, "validation_report", None), "issues", []) or [])\n    counts["reasoning_inferred_triples"] = len(getattr(getattr(state, "reasoning_report", None), "inferred_triples", []) or [])\n    return counts\n\n\ndef load_layer_states(run_dir: str | Path) -> list[tuple[int, str, PipelineState]]:\n    run_dir = Path(run_dir)\n    states: list[tuple[int, str, PipelineState]] = []\n    for index, name in enumerate(LAYER_NAMES):\n        path = run_dir / name / "state.json"\n        if path.is_file():\n            states.append((index, name, PipelineState.load_json(str(path))))\n    return states\n\n\ndef write_layer_summary(run_dir: str | Path) -> list[dict[str, Any]]:\n    run_dir = Path(run_dir)\n    rows: list[dict[str, Any]] = []\n    for index, name, state in load_layer_states(run_dir):\n        metadata_path = run_dir / name / "metadata.json"\n        metadata = read_json(metadata_path) if metadata_path.is_file() else {}\n        rows.append({\n            "layer_index": index,\n            "layer_name": name,\n            "elapsed_seconds": metadata.get("elapsed_seconds"),\n            **state_counts(state),\n        })\n    path = run_dir / "analysis" / "layer_summary.csv"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if rows:\n        with path.open("w", encoding="utf-8", newline="") as handle:\n            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n            writer.writeheader(); writer.writerows(rows)\n    write_json(run_dir / "analysis" / "layer_summary.json", rows)\n    return rows\n\n\ndef norm(text: Any) -> str:\n    return re.sub(r"[^a-z0-9]+", "", str(text or "").lower())\n\n\ndef gold_entity_aliases(gold: dict[str, Any]) -> dict[str, set[str]]:\n    return {\n        entity_id: {norm(m.get("trigger_word")) for m in payload.get("mentions", []) if m.get("trigger_word")}\n        for entity_id, payload in gold.get("entities", {}).items()\n    }\n\n\ndef candidate_aliases(candidate: Any) -> set[str]:\n    values = {getattr(candidate, "canonical_label", "")}\n    values.update(getattr(candidate, "aliases", []) or [])\n    values.update(getattr(candidate, "synonyms", []) or [])\n    values.update(getattr(candidate, "lexical_variants", []) or [])\n    for mention in getattr(candidate, "mentions", []) or []:\n        values.add(getattr(mention, "text", ""))\n    return {norm(value) for value in values if norm(value)}\n\n\ndef align_candidate(candidate: Any, aliases: dict[str, set[str]]) -> str | None:\n    c_aliases = candidate_aliases(candidate)\n    exact = [entity_id for entity_id, gold_alias in aliases.items() if c_aliases.intersection(gold_alias)]\n    if len(exact) == 1:\n        return exact[0]\n    # Conservative containment fallback for aliases such as "Athens metropolitan area".\n    containment = []\n    for entity_id, gold_alias in aliases.items():\n        if any(a and b and (a in b or b in a) for a in c_aliases for b in gold_alias):\n            containment.append(entity_id)\n    return containment[0] if len(set(containment)) == 1 else None\n\n\ndef align_text_values(values: Iterable[str], aliases: dict[str, set[str]]) -> set[str]:\n    """Align raw Layer 1/2 surface strings to gold IDs for diagnostics only."""\n    normalized_values = {norm(value) for value in values if norm(value)}\n    result: set[str] = set()\n    for entity_id, gold_aliases in aliases.items():\n        if normalized_values.intersection(gold_aliases):\n            result.add(entity_id)\n            continue\n        if any(a and b and (a in b or b in a) for a in normalized_values for b in gold_aliases):\n            result.add(entity_id)\n    return result\n\n\ndef relation_lookup(catalog_path: str | Path, aliases_path: str | Path) -> tuple[dict[str, str], dict[str, str]]:\n    catalog = read_json(catalog_path)["relations"]\n    aliases = read_json(aliases_path)\n    id_to_label = {item["relation_id"]: item["label"] for item in catalog}\n    label_to_id: dict[str, str] = {}\n    for relation_id, labels in aliases.items():\n        label_to_id[norm(relation_id)] = relation_id\n        for label in labels:\n            label_to_id[norm(label)] = relation_id\n            label_to_id[norm(f"{relation_id} : {label}")] = relation_id\n    return id_to_label, label_to_id\n\n\ndef map_predicate(label: str, hints: Iterable[str], label_to_id: dict[str, str]) -> str | None:\n    for value in [label, *list(hints or [])]:\n        match = re.search(r"\\bP\\d+\\b", str(value), re.IGNORECASE)\n        if match:\n            return match.group(0).upper()\n    return label_to_id.get(norm(label))\n\n\ndef native_predictions(state: PipelineState, gold: dict[str, Any], catalog_path: str | Path, aliases_path: str | Path) -> list[dict[str, Any]]:\n    aliases = gold_entity_aliases(gold)\n    _, label_to_id = relation_lookup(catalog_path, aliases_path)\n    candidate_by_id = {\n        candidate.candidate_id: candidate\n        for candidate in [\n            *(state.entity_candidates or []), *(state.event_candidates or []),\n            *(state.attribute_candidates or []), *(state.relation_candidates or []),\n        ]\n    }\n    relation_by_id = {candidate.candidate_id: candidate for candidate in state.relation_candidates or []}\n    rows: list[dict[str, Any]] = []\n    for triple in state.candidate_triples or []:\n        subject = candidate_by_id.get(triple.subject_id)\n        obj = candidate_by_id.get(triple.object_id)\n        relation_candidate = relation_by_id.get(triple.predicate_id)\n        relation_id = map_predicate(\n            triple.predicate_label,\n            getattr(relation_candidate, "ontology_hints", []) if relation_candidate else [],\n            label_to_id,\n        )\n        rows.append({\n            "triple_id": triple.triple_id,\n            "subject_label": triple.subject_label,\n            "predicate_label": triple.predicate_label,\n            "object_label": triple.object_label,\n            "head_id": align_candidate(subject, aliases) if subject else None,\n            "relation_id": relation_id,\n            "tail_id": align_candidate(obj, aliases) if obj else None,\n            "fully_mapped": bool(subject and obj and relation_id and align_candidate(subject, aliases) and align_candidate(obj, aliases)),\n            "confidence": triple.confidence,\n            "justification": triple.justification,\n        })\n    return rows\n\n\ndef gold_triples(gold: dict[str, Any]) -> set[tuple[str, str, str]]:\n    triples: set[tuple[str, str, str]] = set()\n    for relation_key, pairs in gold.get("relations", {}).items():\n        relation_id = relation_key.split(":", 1)[0].strip()\n        triples.update((head, relation_id, tail) for head, tail in pairs)\n    return triples\n\n\ndef strict_evaluate(predictions: list[dict[str, Any]], gold: dict[str, Any]) -> dict[str, Any]:\n    predicted = {\n        (row["head_id"], row["relation_id"], row["tail_id"])\n        for row in predictions\n        if row.get("fully_mapped")\n    }\n    expected = gold_triples(gold)\n    tp = predicted.intersection(expected)\n    fp = predicted.difference(expected)\n    fn = expected.difference(predicted)\n    precision = len(tp) / len(predicted) if predicted else 0.0\n    recall = len(tp) / len(expected) if expected else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {\n        "predicted": len(predicted), "gold": len(expected), "true_positive": len(tp),\n        "false_positive": len(fp), "false_negative": len(fn),\n        "precision": precision, "recall": recall, "f1": f1,\n        "tp": sorted(tp), "fp": sorted(fp), "fn": sorted(fn),\n    }\n\n\ndef relation_candidates_by_id(state: PipelineState, label_to_id: dict[str, str]) -> set[str]:\n    result = set()\n    for candidate in state.relation_candidates or []:\n        mapped = map_predicate(candidate.canonical_label, candidate.ontology_hints, label_to_id)\n        if mapped:\n            result.add(mapped)\n    return result\n\n\ndef write_gold_trace(run_dir: str | Path, gold: dict[str, Any], catalog_path: str | Path, aliases_path: str | Path) -> list[dict[str, Any]]:\n    run_dir = Path(run_dir)\n    aliases = gold_entity_aliases(gold)\n    id_to_label, label_to_id = relation_lookup(catalog_path, aliases_path)\n    states = {index: state for index, _, state in load_layer_states(run_dir)}\n    final_state = states[max(states)]\n    rows: list[dict[str, Any]] = []\n\n    layer1 = states.get(1, final_state)\n    layer2 = states.get(2, final_state)\n    layer3 = states.get(3, final_state)\n\n    layer1_nodes = align_text_values(\n        [expr.text for expr in (layer1.linguistic_expressions or [])], aliases\n    )\n    layer2_nodes = align_text_values(\n        [item.base_expression.text for item in (layer2.enriched_expressions or [])], aliases\n    )\n\n    node_candidates = [\n        *(layer3.entity_candidates or []),\n        *(layer3.event_candidates or []),\n        *(layer3.attribute_candidates or []),\n    ]\n    aligned_nodes = {align_candidate(candidate, aliases) for candidate in node_candidates}\n    aligned_nodes.discard(None)\n    available_predicates = relation_candidates_by_id(layer3, label_to_id)\n\n    layer4 = states.get(4, final_state)\n    layer5 = states.get(5, final_state)\n    pred4 = []\n    cand4 = {\n        c.candidate_id: c\n        for c in [\n            *(layer4.entity_candidates or []),\n            *(layer4.event_candidates or []),\n            *(layer4.attribute_candidates or []),\n        ]\n    }\n    rel4 = {c.candidate_id: c for c in layer4.relation_candidates or []}\n    for assertion in layer4.candidate_relation_assertions or []:\n        src = cand4.get(assertion.source_candidate_id)\n        dst = cand4.get(assertion.target_candidate_id)\n        rel = rel4.get(assertion.relation_candidate_id)\n        pred4.append((\n            align_candidate(src, aliases) if src else None,\n            map_predicate(\n                assertion.relation_label,\n                getattr(rel, "ontology_hints", []) if rel else [],\n                label_to_id,\n            ),\n            align_candidate(dst, aliases) if dst else None,\n        ))\n    pred5_rows = native_predictions(layer5, gold, catalog_path, aliases_path)\n    pred5 = {\n        (row["head_id"], row["relation_id"], row["tail_id"])\n        for row in pred5_rows\n        if row["fully_mapped"]\n    }\n\n    for head, relation_id, tail in sorted(gold_triples(gold)):\n        head_l1 = head in layer1_nodes\n        tail_l1 = tail in layer1_nodes\n        head_l2 = head in layer2_nodes\n        tail_l2 = tail in layer2_nodes\n        head_l3 = head in aligned_nodes\n        tail_l3 = tail in aligned_nodes\n        predicate_found = relation_id in available_predicates\n        assertion_found = (head, relation_id, tail) in pred4\n        triple_found = (head, relation_id, tail) in pred5\n\n        if not head_l1 or not tail_l1:\n            first_failure = "layer01_expression_extraction"\n        elif not head_l2 or not tail_l2:\n            first_failure = "layer02_enrichment_survival"\n        elif not head_l3 or not tail_l3:\n            first_failure = "layer03_endpoint_resolution"\n        elif not predicate_found:\n            first_failure = "layer03_predicate_typing_or_ontology_linking"\n        elif not assertion_found:\n            first_failure = "layer04_endpoint_assignment"\n        elif not triple_found:\n            first_failure = "layer05_triple_materialization_or_mapping"\n        else:\n            first_failure = "survived_to_layer05"\n\n        rows.append({\n            "head_id": head,\n            "relation_id": relation_id,\n            "relation_label": id_to_label.get(relation_id),\n            "tail_id": tail,\n            "head_available_layer01": head_l1,\n            "tail_available_layer01": tail_l1,\n            "head_available_layer02": head_l2,\n            "tail_available_layer02": tail_l2,\n            "head_available_layer03": head_l3,\n            "tail_available_layer03": tail_l3,\n            "predicate_available_layer03": predicate_found,\n            "assertion_found_layer04": assertion_found,\n            "triple_found_layer05": triple_found,\n            "first_failure": first_failure,\n        })\n\n    analysis_dir = run_dir / "analysis"\n    analysis_dir.mkdir(parents=True, exist_ok=True)\n    with (analysis_dir / "gold_relation_trace.csv").open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    write_json(analysis_dir / "gold_relation_trace.json", rows)\n    return rows\n\ndef analyze_run(*, run_dir: str | Path, gold_jsonl: str | Path, catalog_path: str | Path, aliases_path: str | Path) -> dict[str, Any]:\n    run_dir=Path(run_dir)\n    gold_rows=read_jsonl(gold_jsonl)\n    if len(gold_rows)!=1:\n        raise ValueError(\'Analysis expects exactly one gold document.\')\n    gold=gold_rows[0]\n    layer_rows=write_layer_summary(run_dir)\n    states=load_layer_states(run_dir)\n    if not states:\n        raise FileNotFoundError(f\'No layer state artifacts found in {run_dir}\')\n    final_state=states[-1][2]\n    predictions=native_predictions(final_state,gold,catalog_path,aliases_path)\n    analysis_dir=run_dir/\'analysis\'; analysis_dir.mkdir(parents=True,exist_ok=True)\n    write_json(analysis_dir/\'native_mapped_predictions.json\',predictions)\n    with (analysis_dir/\'native_mapped_predictions.jsonl\').open(\'w\',encoding=\'utf-8\') as handle:\n        for row in predictions: handle.write(json.dumps(row,ensure_ascii=False,default=str)+\'\\n\')\n    evaluation=strict_evaluate(predictions,gold)\n    write_json(analysis_dir/\'strict_docred_evaluation.json\',evaluation)\n    trace=write_gold_trace(run_dir,gold,catalog_path,aliases_path)\n    summary={\n        \'document_id\':gold[\'document_id\'],\n        \'layer_summary\':layer_rows,\n        \'strict_evaluation\':evaluation,\n        \'mapped_native_triples\':sum(1 for row in predictions if row.get(\'fully_mapped\')),\n        \'unmapped_native_triples\':sum(1 for row in predictions if not row.get(\'fully_mapped\')),\n        \'failure_counts\':{},\n    }\n    for row in trace:\n        summary[\'failure_counts\'][row[\'first_failure\']]=summary[\'failure_counts\'].get(row[\'first_failure\'],0)+1\n    write_json(analysis_dir/\'analysis_summary.json\',summary)\n    return summary\n', 'docred_native_ablation_v3.py': 'from __future__ import annotations\n\n"""Fast, profile-guided native DocRED ablation support.\n\nThis module changes no file under ``src/neoolaf``. It orchestrates the existing\n13 NeoOLAF layers, adds input-level UserGuidance metadata, uses separate\nper-layer token/time budgets, parallelizes the existing Layer 4 task, and adds\nmore detailed diagnostics. It does not add a second relation extraction task,\nsource-entity anchoring, closure rules, or gold-derived facts.\n"""\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom contextlib import redirect_stderr, redirect_stdout\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any, Iterable\nimport csv\nimport json\nimport re\nimport shutil\nimport sys\nimport threading\nimport time\nimport traceback\n\nimport docred_native_ablation as v2\n\nfrom neoolaf.core.pipeline import Pipeline\nfrom neoolaf.core.pipeline_state import PipelineState\nfrom neoolaf.core.runner import Runner\nfrom neoolaf.domain.candidates import RelationCandidate\nfrom neoolaf.domain.documents import Document\nfrom neoolaf.domain.relation_assertion import CandidateRelationAssertion\nfrom neoolaf.domain.user_guidance import (\n    NegativeExample,\n    RelationExample,\n    TypingExample,\n    UserGuidance,\n)\nfrom neoolaf.grounding.rag.formatting import build_grounding_context\nfrom neoolaf.grounding.rag.types import GroundingRequest, RetrievedItem\nfrom neoolaf.ontology.loader import SeedOntologyLoader\nfrom neoolaf.profiles.profile_loader import load_document_profile\n\nfrom neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer\nfrom neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer\nfrom neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer\nfrom neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer\nfrom neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer\nfrom neoolaf.layers.layer04_candidate_relation_extraction.prompt import build_system_prompt as build_l4_system_prompt\nfrom neoolaf.layers.layer04_candidate_relation_extraction.prompt import build_user_prompt as build_l4_user_prompt\nfrom neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer\nfrom neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer\nfrom neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer\nfrom neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer\nfrom neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer\nfrom neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer\nfrom neoolaf.layers.layer11_inference_completion.component import InferenceCompletionLayer\nfrom neoolaf.layers.layer12_serialization.component import SerializationLayer\n\nfrom experiments.methods.run_neoolaf import (\n    OfflineWebSearchSource,\n    OfflineWikipediaSource,\n    OfflineWikidataSource,\n    OpenAICompatibleBackend,\n    load_user_guidance,\n)\n\n# Re-export notebook helpers from v2.\nread_json = v2.read_json\nread_jsonl = v2.read_jsonl\nwrite_json = v2.write_json\nappend_jsonl = v2.append_jsonl\nload_layer_states = v2.load_layer_states\nstate_counts = v2.state_counts\nsafe_name = v2.safe_name\nTee = v2.Tee\nLAYER_NAMES = v2.LAYER_NAMES\n\n\ndef _dedup(values: Iterable[Any]) -> list[str]:\n    result: list[str] = []\n    seen: set[str] = set()\n    for value in values:\n        if value is None:\n            continue\n        text = str(value).strip()\n        if not text or text in seen:\n            continue\n        seen.add(text)\n        result.append(text)\n    return result\n\n\nclass SharedCallLogger:\n    """One thread-safe logger shared by all per-layer backends."""\n\n    def __init__(self, log_dir: str | Path) -> None:\n        self.log_dir = Path(log_dir)\n        self.log_dir.mkdir(parents=True, exist_ok=True)\n        (self.log_dir / "responses").mkdir(parents=True, exist_ok=True)\n        self.calls_path = self.log_dir / "llm_calls.jsonl"\n        self.errors_path = self.log_dir / "llm_errors.jsonl"\n        self.parse_errors_path = self.log_dir / "llm_parse_errors.jsonl"\n        self.cap_errors_path = self.log_dir / "llm_response_cap_errors.jsonl"\n        self.lock = threading.Lock()\n        self.call_index = 0\n\n    def next_index(self) -> int:\n        with self.lock:\n            self.call_index += 1\n            return self.call_index\n\n\nclass TaggedLoggedBackend:\n    """Logger and output guard around one existing OpenAI-compatible backend."""\n\n    def __init__(\n        self,\n        backend: OpenAICompatibleBackend,\n        logger: SharedCallLogger,\n        *,\n        layer_tag: str,\n        response_hard_cap_chars: int | None = None,\n    ) -> None:\n        self.backend = backend\n        self.logger = logger\n        self.layer_tag = layer_tag\n        self.response_hard_cap_chars = int(response_hard_cap_chars or 0) or None\n\n    def chat(self, model: str, messages: list[dict[str, str]], temperature: float = 0.0, **_: Any) -> str:\n        call_index = self.logger.next_index()\n        started = time.time()\n        meta = {\n            "call_index": call_index,\n            "layer_tag": self.layer_tag,\n            "model": model,\n            "temperature": temperature,\n            "message_count": len(messages),\n            "system_chars": sum(len(m.get("content", "")) for m in messages if m.get("role") == "system"),\n            "user_chars": sum(len(m.get("content", "")) for m in messages if m.get("role") != "system"),\n            "max_tokens": getattr(self.backend, "max_tokens", None),\n            "request_timeout": getattr(self.backend, "timeout", None),\n            "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n        }\n        try:\n            response = self.backend.chat(model=model, messages=messages, temperature=temperature)\n            response_path = (\n                self.logger.log_dir / "responses" /\n                f"{call_index:04d}_{safe_name(self.layer_tag)}.txt"\n            )\n            response_path.write_text(response, encoding="utf-8")\n\n            parse_ok = True\n            parse_error = None\n            parsed_type = None\n            try:\n                parsed = OpenAICompatibleBackend.extract_json(response)\n                parsed_type = type(parsed).__name__\n            except Exception as exc:  # The layer may retry; log before it does.\n                parse_ok = False\n                parse_error = f"{type(exc).__name__}: {exc}"\n                append_jsonl(self.logger.parse_errors_path, {\n                    **meta,\n                    "elapsed_seconds": round(time.time() - started, 3),\n                    "response_chars": len(response),\n                    "response_path": str(response_path),\n                    "error": parse_error,\n                }, self.logger.lock)\n\n            if self.response_hard_cap_chars and len(response) > self.response_hard_cap_chars:\n                message = (\n                    f"{self.layer_tag}: response length {len(response)} exceeds hard cap "\n                    f"{self.response_hard_cap_chars}; retrying with the same native task."\n                )\n                append_jsonl(self.logger.cap_errors_path, {\n                    **meta,\n                    "elapsed_seconds": round(time.time() - started, 3),\n                    "response_chars": len(response),\n                    "response_path": str(response_path),\n                    "error": message,\n                }, self.logger.lock)\n                raise RuntimeError(message)\n\n            append_jsonl(self.logger.calls_path, {\n                **meta,\n                "status": "ok",\n                "elapsed_seconds": round(time.time() - started, 3),\n                "response_chars": len(response),\n                "response_path": str(response_path),\n                "json_parse_ok": parse_ok,\n                "parsed_type": parsed_type,\n                "parse_error": parse_error,\n            }, self.logger.lock)\n            return response\n        except Exception as exc:\n            append_jsonl(self.logger.errors_path, {\n                **meta,\n                "status": "error",\n                "elapsed_seconds": round(time.time() - started, 3),\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n            }, self.logger.lock)\n            raise\n\n    @staticmethod\n    def extract_json(text: str) -> Any:\n        return OpenAICompatibleBackend.extract_json(text)\n\n\nclass PriorityOntologyRAGAdapter(v2.OntologyOnlyRAGAdapter):\n    """Property-first retrieval with relation aliases and priority property IDs."""\n\n    name = "priority_property_first_ontology_only"\n\n    def __init__(\n        self,\n        seed_ontology: Any,\n        log_path: str | Path,\n        *,\n        top_k: int = 10,\n        query_expansions: dict[str, list[str]] | None = None,\n        relation_aliases: dict[str, list[str]] | None = None,\n        priority_property_ids: list[str] | None = None,\n    ) -> None:\n        super().__init__(\n            seed_ontology,\n            log_path=log_path,\n            top_k=top_k,\n            query_expansions=query_expansions,\n        )\n        self.relation_aliases = relation_aliases or {}\n        self.priority_property_ids = [str(x).upper() for x in (priority_property_ids or [])]\n        self.alias_to_ids: dict[str, list[str]] = {}\n        for relation_id, aliases in self.relation_aliases.items():\n            for alias in [relation_id, *list(aliases or [])]:\n                key = re.sub(r"\\s+", " ", str(alias).lower()).strip()\n                if key:\n                    self.alias_to_ids.setdefault(key, []).append(str(relation_id).upper())\n\n    def _expanded_queries(self, query: str) -> list[str]:\n        values = super()._expanded_queries(query)\n        lowered = re.sub(r"\\s+", " ", str(query or "").lower()).strip()\n        for alias, relation_ids in self.alias_to_ids.items():\n            if alias and (alias == lowered or alias in lowered or lowered in alias):\n                for relation_id in relation_ids:\n                    prop = self.seed_ontology.properties_by_uri.get(\n                        f"http://www.wikidata.org/prop/direct/{relation_id}"\n                    )\n                    values.extend([relation_id, getattr(prop, "label", None)])\n        return _dedup(values)\n\n    def _items(self, query: str, top_k: int | None = None, layer_name: str | None = None):\n        top_k = max(3, int(top_k or self.top_k))\n        layer_name = str(layer_name or "")\n        relation_focused = layer_name.endswith((\n            "candidate_relation_extraction",\n            "concept_relation_induction",\n        ))\n        query_l = str(query or "").lower()\n        relation_like = relation_focused or any(\n            token in query_l\n            for token in [" based ", "part of", "member of", "owned", "subsidiary", "country", "located", "born", "died", "founded", "established", "performed", "released"]\n        )\n        property_budget = top_k - 1 if relation_focused else max(2, int(round(top_k * (0.8 if relation_like else 0.6))))\n        class_budget = max(1, top_k - property_budget)\n\n        expanded = self._expanded_queries(query)\n        retriever = self.space.retriever\n        if retriever is None:\n            return [], expanded\n\n        properties: list[Any] = []\n        classes: list[Any] = []\n        for expanded_query in expanded:\n            normalized = str(expanded_query).lower().strip()\n            property_id = re.fullmatch(r"p\\d+", normalized)\n            if property_id:\n                prop = self.seed_ontology.properties_by_uri.get(\n                    f"http://www.wikidata.org/prop/direct/{normalized.upper()}"\n                )\n                if prop is not None:\n                    properties.append(prop)\n            for uri in self.seed_ontology.property_uris_by_label.get(normalized, []):\n                prop = self.seed_ontology.properties_by_uri.get(uri)\n                if prop is not None:\n                    properties.append(prop)\n            for uri in self.seed_ontology.class_uris_by_label.get(normalized, []):\n                cls = self.seed_ontology.classes_by_uri.get(uri)\n                if cls is not None:\n                    classes.append(cls)\n\n        for expanded_query in expanded:\n            properties.extend(retriever.nearest_properties(expanded_query, top_k=property_budget))\n            classes.extend(retriever.nearest_classes(expanded_query, top_k=class_budget))\n\n        # Priority properties are tie-breakers, not unconditional evidence.\n        if relation_like:\n            for relation_id in self.priority_property_ids:\n                if relation_id.lower() in query_l:\n                    prop = self.seed_ontology.properties_by_uri.get(\n                        f"http://www.wikidata.org/prop/direct/{relation_id}"\n                    )\n                    if prop is not None:\n                        properties.insert(0, prop)\n\n        def dedup_objects(values: list[Any], budget: int) -> list[Any]:\n            result: list[Any] = []\n            seen: set[str] = set()\n            for value in values:\n                uri = str(value.uri)\n                if uri in seen:\n                    continue\n                seen.add(uri)\n                result.append(value)\n                if len(result) >= budget:\n                    break\n            return result\n\n        selected_properties = dedup_objects(properties, property_budget)\n        selected_classes = dedup_objects(classes, class_budget)\n        items = [self._property_item(prop) for prop in selected_properties]\n        items.extend(self._class_item(cls) for cls in selected_classes)\n        return items[:top_k], expanded\n\n\nclass CanonicalizingCandidateTypingResolutionLayer(CandidateTypingResolutionLayer):\n    """Run native Layer 3, then normalize relation labels from its own hints.\n\n    This deterministic step does not infer a relation. It only converts a\n    relation candidate already linked by Layer 2 to its canonical ontology ID,\n    which is the benchmark-allowed native-to-ontology mapping stage.\n    """\n\n    def __init__(self, *args: Any, relation_catalog_path: str | Path, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        catalog = read_json(relation_catalog_path)["relations"]\n        self.id_to_label = {item["relation_id"]: item["label"] for item in catalog}\n        self.relation_metadata = {item["relation_id"]: item for item in catalog}\n\n    @staticmethod\n    def _relation_id_from_hints(hints: Iterable[str]) -> str | None:\n        controlled: list[str] = []\n        other: list[str] = []\n        for hint in hints or []:\n            text = str(hint)\n            match = re.search(r"\\bP\\d+\\b", text, re.IGNORECASE)\n            if not match:\n                continue\n            relation_id = match.group(0).upper()\n            if text.lower().startswith("controlled_relation:"):\n                controlled.append(relation_id)\n            else:\n                other.append(relation_id)\n        return (controlled or other or [None])[0]\n\n    def _run(self, state: PipelineState) -> PipelineState:\n        state = super()._run(state)\n        diagnostics: list[dict[str, Any]] = []\n        for candidate in state.relation_candidates or []:\n            original = candidate.canonical_label\n            relation_id = self._relation_id_from_hints(candidate.ontology_hints)\n            if relation_id and relation_id in self.id_to_label:\n                canonical = f"{relation_id} : {self.id_to_label[relation_id]}"\n                candidate.aliases = _dedup([original, *list(candidate.aliases or [])])\n                candidate.canonical_label = canonical\n                candidate.normalized_label = self._normalize_label(canonical)\n                metadata = self.relation_metadata.get(relation_id, {})\n                candidate.ontology_hints = _dedup([\n                    f"controlled_relation:{canonical}",\n                    "promote_to_ontology:true",\n                    metadata.get("uri"),\n                    metadata.get("label"),\n                    f"domain:{\', \'.join(metadata.get(\'domain_uris\') or [])}" if metadata.get("domain_uris") else None,\n                    f"range:{\', \'.join(metadata.get(\'range_uris\') or [])}" if metadata.get("range_uris") else None,\n                    *list(candidate.ontology_hints or []),\n                ])\n            diagnostics.append({\n                "candidate_id": candidate.candidate_id,\n                "original_label": original,\n                "canonical_label": candidate.canonical_label,\n                "has_mentions": bool(candidate.mentions),\n                "relation_id": relation_id,\n                "controlled_hint_present": any(\n                    str(h).lower().startswith("controlled_relation:")\n                    for h in candidate.ontology_hints or []\n                ),\n                "ontology_hints": list(candidate.ontology_hints or []),\n            })\n        if state.artifact_dir:\n            path = Path(state.artifact_dir) / self.name / "relation_canonicalization.json"\n            write_json(path, diagnostics)\n        return state\n\n\nclass ParallelCandidateRelationExtractionLayer(CandidateRelationExtractionLayer):\n    """Parallel orchestration of the existing native Layer 4 task."""\n\n    def __init__(\n        self,\n        *args: Any,\n        max_attempts_per_relation: int = 2,\n        retry_wait_seconds: float = 1.0,\n        failure_log_path: str | Path | None = None,\n        **kwargs: Any,\n    ) -> None:\n        super().__init__(*args, **kwargs)\n        self.max_attempts_per_relation = max(1, int(max_attempts_per_relation))\n        self.relation_retry_wait_seconds = float(retry_wait_seconds)\n        self.failure_log_path = Path(failure_log_path) if failure_log_path else None\n        self.failure_lock = threading.Lock()\n\n    def _call_model_with_retries(self, state: PipelineState, messages, max_attempts=5, retry_wait_seconds=3.0):\n        return super()._call_model_with_retries(\n            state,\n            messages,\n            max_attempts=self.max_attempts_per_relation,\n            retry_wait_seconds=self.relation_retry_wait_seconds,\n        )\n\n    def _process_relation_mention(\n        self,\n        *,\n        state: PipelineState,\n        relation_mention: dict[str, Any],\n        chunk_to_local_candidates: dict[str, list[dict[str, Any]]],\n    ) -> CandidateRelationAssertion | None:\n        chunk_id = relation_mention["chunk_id"]\n        relation_candidate = relation_mention["relation_candidate"]\n        relation_evidence = relation_mention["evidence"]\n        chunk = self._get_chunk_by_id(state, chunk_id)\n        if chunk is None:\n            return None\n        local_candidates = chunk_to_local_candidates.get(chunk_id, [])\n        if len(local_candidates) < 2:\n            return None\n\n        relation_payload = {\n            "candidate_id": relation_candidate.candidate_id,\n            "canonical_label": relation_candidate.canonical_label,\n            "candidate_type": relation_candidate.candidate_type,\n            "ontology_hints": list(relation_candidate.ontology_hints or []),\n            "aliases": list(relation_candidate.aliases or []),\n        }\n        local_candidate_payload = [\n            {\n                "candidate_id": item["candidate"].candidate_id,\n                "canonical_label": item["candidate"].canonical_label,\n                "candidate_type": item["candidate"].candidate_type,\n                "ontology_hints": list(item["candidate"].ontology_hints or []),\n            }\n            for item in local_candidates\n        ]\n        grounding_context = ""\n        if self.rag_adapter is not None:\n            grounding = self.rag_adapter.ground(GroundingRequest(\n                layer_name="layer04_candidate_relation_extraction",\n                query=relation_candidate.canonical_label,\n                payload={\n                    "relation_candidate": relation_candidate.canonical_label,\n                    "chunk_text": chunk.text,\n                    "local_candidates": local_candidate_payload,\n                },\n                preferred_sources=["ontology"],\n                top_k=8,\n            ))\n            grounding_context = build_grounding_context(grounding)\n\n        messages = [\n            {"role": "system", "content": build_l4_system_prompt()},\n            {"role": "user", "content": build_l4_user_prompt(\n                chunk_text=chunk.text,\n                chunk_id=chunk_id,\n                relation_candidate=relation_payload,\n                local_candidates=local_candidate_payload,\n                guidance=state.user_guidance,\n                grounding_context=grounding_context,\n            )},\n        ]\n        parsed = self._call_model_with_retries(state, messages)\n        if not parsed.get("found", False):\n            return None\n        source = self._find_candidate_by_id(state, parsed.get("source_candidate_id"))\n        target = self._find_candidate_by_id(state, parsed.get("target_candidate_id"))\n        if source is None or target is None:\n            return None\n        return CandidateRelationAssertion(\n            assertion_id="pending",\n            relation_candidate_id=relation_candidate.candidate_id,\n            relation_label=relation_candidate.canonical_label,\n            source_candidate_id=source.candidate_id,\n            source_candidate_label=source.canonical_label,\n            source_candidate_type=source.candidate_type,\n            target_candidate_id=target.candidate_id,\n            target_candidate_label=target.canonical_label,\n            target_candidate_type=target.candidate_type,\n            chunk_id=chunk_id,\n            justification=str(parsed.get("justification") or "").strip(),\n            confidence=parsed.get("confidence"),\n            evidence=relation_evidence,\n        )\n\n    def _run(self, state: PipelineState) -> PipelineState:\n        self._relation_strategy = self._strategy(state)\n        if self.verbose:\n            print(\n                f"[NeoOLAF][Layer 4] strategy={self._relation_strategy}; "\n                f"parallel_workers={self.max_concurrency}; attempts={self.max_attempts_per_relation}"\n            )\n        if self._is_record_aware_strategy(self._relation_strategy):\n            return self._run_record_aware_ontology(state)\n\n        chunk_to_local_candidates = self._index_local_entity_event_candidates(state)\n        relation_mentions = self._index_relation_mentions(state)\n        if self.max_relation_mentions is not None:\n            relation_mentions = relation_mentions[: self.max_relation_mentions]\n\n        results: dict[int, CandidateRelationAssertion] = {}\n        failures: list[dict[str, Any]] = []\n        with ThreadPoolExecutor(max_workers=self.max_concurrency) as executor:\n            future_to_index = {\n                executor.submit(\n                    self._process_relation_mention,\n                    state=state,\n                    relation_mention=mention,\n                    chunk_to_local_candidates=chunk_to_local_candidates,\n                ): index\n                for index, mention in enumerate(relation_mentions)\n            }\n            for future in as_completed(future_to_index):\n                index = future_to_index[future]\n                try:\n                    assertion = future.result()\n                    if assertion is not None:\n                        results[index] = assertion\n                except Exception as exc:\n                    mention = relation_mentions[index]\n                    failure = {\n                        "index": index,\n                        "relation_candidate_id": mention["relation_candidate"].candidate_id,\n                        "relation_label": mention["relation_candidate"].canonical_label,\n                        "chunk_id": mention["chunk_id"],\n                        "error_type": type(exc).__name__,\n                        "error": str(exc),\n                        "traceback": traceback.format_exc(),\n                    }\n                    failures.append(failure)\n                    if self.failure_log_path:\n                        append_jsonl(self.failure_log_path, failure, self.failure_lock)\n\n        assertions: list[CandidateRelationAssertion] = []\n        seen: set[tuple[str, str, str, str]] = set()\n        for index in sorted(results):\n            assertion = results[index]\n            key = (\n                assertion.relation_candidate_id,\n                assertion.source_candidate_id,\n                assertion.target_candidate_id,\n                assertion.chunk_id,\n            )\n            if key in seen:\n                continue\n            seen.add(key)\n            assertion.assertion_id = f"rel_assert_{len(assertions):05d}"\n            assertions.append(assertion)\n\n        state.candidate_relation_assertions = assertions\n        state.log(\n            f"[{self.name}] parallel native extraction; mentions={len(relation_mentions)}; "\n            f"assertions={len(assertions)}; failed={len(failures)}"\n        )\n        return state\n\n\ndef merge_input_task_guidance(guidance: UserGuidance, record: dict[str, Any]) -> UserGuidance:\n    """Merge non-gold per-document task metadata into fields NeoOLAF consumes."""\n    task = record.get("task_guidance")\n    if not isinstance(task, dict):\n        return guidance\n\n    allowed_ids = [str(x) for x in task.get("allowed_relation_ids") or []]\n    specs = task.get("relation_specs") or []\n    spec_lines = []\n    for spec in specs:\n        if not isinstance(spec, dict):\n            continue\n        rid = spec.get("relation_id")\n        label = spec.get("label")\n        direction = spec.get("direction")\n        if rid and label:\n            spec_lines.append(f"{rid} : {label} — {direction or \'\'}".strip())\n    guidance.priority_relations = _dedup([\n        *list(guidance.priority_relations or []),\n        *[\n            line.split(" — ", 1)[0]\n            for line in spec_lines\n        ],\n    ])\n    if spec_lines:\n        guidance.domain_focus = (\n            (guidance.domain_focus or "")\n            + " INPUT RELATION SPECIFICATION: "\n            + "; ".join(spec_lines)\n        ).strip()\n    guidance.population_policy = " ".join(filter(None, [\n        guidance.population_policy,\n        task.get("canonical_hint_contract"),\n        task.get("inference_policy"),\n        task.get("relation_instance_policy"),\n    ]))\n\n    existing_examples = {\n        (e.text, e.relation_label, e.source_label, e.target_label)\n        for e in guidance.relation_examples\n    }\n    for item in task.get("relation_examples") or []:\n        if not isinstance(item, dict):\n            continue\n        key = (\n            str(item.get("text") or ""),\n            str(item.get("relation_label") or ""),\n            str(item.get("source_label") or ""),\n            str(item.get("target_label") or ""),\n        )\n        if all(key) and key not in existing_examples:\n            # ``task_guidance.relation_examples`` may carry experiment-only\n            # metadata (for example ``candidate_relation_ids``) used by the\n            # compact DocRED prompt builder.  ``RelationExample`` intentionally\n            # contains only the stable NeoOLAF guidance contract, so construct\n            # it explicitly instead of forwarding every input key.\n            guidance.relation_examples.append(\n                RelationExample(\n                    text=key[0],\n                    relation_label=key[1],\n                    source_label=key[2],\n                    target_label=key[3],\n                    explanation=(\n                        str(item.get("explanation"))\n                        if item.get("explanation") is not None\n                        else None\n                    ),\n                )\n            )\n            existing_examples.add(key)\n    return guidance\n\n\ndef guidance_to_dict(guidance: UserGuidance) -> dict[str, Any]:\n    return asdict(guidance)\n\n\ndef _layer_cfg(profile: dict[str, Any], layer_name: str) -> dict[str, Any]:\n    return dict((profile.get("layers") or {}).get(layer_name) or {})\n\n\ndef _make_backend(\n    *,\n    logger: SharedCallLogger,\n    layer_tag: str,\n    model_host: str,\n    api_key: str,\n    cfg: dict[str, Any],\n    fallback_max_tokens: int,\n    fallback_timeout: int,\n    reasoning_effort: str,\n) -> TaggedLoggedBackend:\n    core = OpenAICompatibleBackend(\n        backend_name="openrouter",\n        host=model_host,\n        api_key=api_key,\n        timeout=int(cfg.get("request_timeout_seconds", fallback_timeout)),\n        max_tokens=int(cfg.get("max_output_tokens", fallback_max_tokens)),\n        reasoning_effort=reasoning_effort,\n        exclude_reasoning=True,\n    )\n    return TaggedLoggedBackend(\n        core,\n        logger,\n        layer_tag=layer_tag,\n        response_hard_cap_chars=cfg.get("response_hard_cap_chars"),\n    )\n\n\ndef choose_chunk_size(text: str, max_safe_chars: int = 24000) -> int:\n    return v2.choose_chunk_size(text, max_safe_chars)\n\n\ndef build_document(record: dict[str, Any], source_path: str | Path) -> Document:\n    return v2.build_document(record, source_path)\n\n\ndef build_pipeline(\n    *,\n    backends: dict[str, TaggedLoggedBackend],\n    rag_adapter: PriorityOntologyRAGAdapter,\n    profile_config: dict[str, Any],\n    relation_catalog_path: str | Path,\n    chunk_size: int,\n    run_dir: str | Path,\n    workers: int = 12,\n    verbose: bool = True,\n) -> Pipeline:\n    workers = max(1, int(workers))\n    l2_cfg = _layer_cfg(profile_config, "layer02_candidate_enrichment")\n    l4_cfg = _layer_cfg(profile_config, "layer04_candidate_relation_extraction")\n    retry_default = int((profile_config.get("orchestration") or {}).get("retry_failed_calls", 1))\n    sleep_default = float((profile_config.get("orchestration") or {}).get("retry_sleep_seconds", 1.0))\n    l2_workers = int(l2_cfg.get("max_concurrency", workers))\n    l4_workers = int(l4_cfg.get("max_concurrency", min(workers, 8)))\n\n    layers = [\n        PreprocessingLayer(\n            chunk_size=chunk_size,\n            overlap=0,\n            enable_chunking=True,\n            translate=False,\n            save_intermediate=True,\n            verbose=verbose,\n            profile_config=profile_config,\n        ),\n        LinguisticExpressionExtractionLayer(\n            backends["layer01"],\n            max_chunks=1,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_backend=rag_adapter,\n            max_concurrency=1,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n            rag_enabled=False,\n        ),\n        CandidateEnrichmentLayer(\n            backends["layer02"],\n            wikipedia_source=OfflineWikipediaSource(),\n            wikidata_source=OfflineWikidataSource(),\n            web_search_source=OfflineWebSearchSource(),\n            max_expressions=None,\n            use_web_search=False,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=l2_workers,\n            retry_failed_calls=int(l2_cfg.get("retry_failed_calls", retry_default)),\n            retry_sleep_seconds=sleep_default,\n        ),\n        CanonicalizingCandidateTypingResolutionLayer(\n            backends["other"],\n            relation_catalog_path=relation_catalog_path,\n            max_expressions=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        ParallelCandidateRelationExtractionLayer(\n            backends["layer04"],\n            max_relation_mentions=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=l4_workers,\n            retry_failed_calls=int(l4_cfg.get("retry_failed_calls", retry_default)),\n            retry_sleep_seconds=sleep_default,\n            max_attempts_per_relation=int(l4_cfg.get("max_attempts_per_relation", 2)),\n            retry_wait_seconds=float(l4_cfg.get("retry_wait_seconds", 1.0)),\n            failure_log_path=Path(run_dir) / "run_logs/layer04_relation_errors.jsonl",\n        ),\n        CandidateTripleGenerationLayer(\n            max_assertions=None,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n            save_intermediate=True,\n            verbose=verbose,\n        ),\n        ConceptRelationInductionLayer(\n            backends["other"],\n            max_concept_inputs=None,\n            max_relation_inputs=None,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n        ),\n        HierarchisationLayer(\n            backends["other"],\n            max_concept_pairs=None,\n            max_relation_pairs=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        AxiomSchemataExtractionLayer(\n            backends["other"],\n            max_relation_schema_inputs=None,\n            max_subclass_inputs=None,\n            temperature=0.0,\n            rag_adapter=rag_adapter,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        GeneralAxiomExtractionLayer(\n            backends["other"],\n            max_schema_inputs=None,\n            max_description_inputs=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        ValidationReasoningLayer(\n            max_triples=None,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        InferenceCompletionLayer(\n            max_inferred_triples=None,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        SerializationLayer(\n            output_subdir="exports",\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n        ),\n    ]\n    return Pipeline(layers=layers, verbose=verbose, continue_from_last=False)\n\n\ndef run_native_pipeline(\n    *,\n    project_root: str | Path,\n    input_jsonl: str | Path,\n    ontology_path: str | Path,\n    profile_path: str | Path,\n    guidance_path: str | Path,\n    relation_catalog_path: str | Path,\n    relation_aliases_path: str | Path,\n    run_dir: str | Path,\n    model_name: str,\n    api_key: str,\n    host: str = "https://openrouter.ai/api/v1",\n    workers: int = 12,\n    max_tokens: int = 4096,\n    request_timeout: int = 180,\n    reasoning_effort: str = "minimal",\n    verbose: bool = True,\n    clean_run_dir: bool = True,\n) -> PipelineState:\n    project_root = Path(project_root).resolve()\n    input_jsonl = Path(input_jsonl).resolve()\n    ontology_path = Path(ontology_path).resolve()\n    profile_path = Path(profile_path).resolve()\n    guidance_path = Path(guidance_path).resolve()\n    relation_catalog_path = Path(relation_catalog_path).resolve()\n    relation_aliases_path = Path(relation_aliases_path).resolve()\n    run_dir = Path(run_dir).resolve()\n    if clean_run_dir and run_dir.exists():\n        shutil.rmtree(run_dir)\n    run_dir.mkdir(parents=True, exist_ok=True)\n    logs_dir = run_dir / "run_logs"\n    logs_dir.mkdir(parents=True, exist_ok=True)\n\n    records = read_jsonl(input_jsonl)\n    if len(records) != 1:\n        raise ValueError(f"This notebook expects exactly one input document, found {len(records)}")\n    record = records[0]\n    if "entities" in record or "relations" in record:\n        raise ValueError("Pipeline input must not contain gold entities or relations.")\n    if not api_key:\n        raise ValueError("OPENROUTER_API_KEY is not set.")\n\n    profile = load_document_profile(profile_path=profile_path)\n    profile_dict = profile.to_state_dict()\n    guidance = load_user_guidance(str(guidance_path))\n    if guidance is None:\n        guidance = UserGuidance()\n    guidance = merge_input_task_guidance(guidance, record)\n    write_json(run_dir / "input_task_guidance.json", record.get("task_guidance") or {})\n    write_json(run_dir / "effective_user_guidance.json", guidance_to_dict(guidance))\n\n    seed_ontology = SeedOntologyLoader().load(str(ontology_path))\n    if len(seed_ontology.properties_by_uri) < 90:\n        raise RuntimeError(\n            f"Only {len(seed_ontology.properties_by_uri)} ontology properties were loaded."\n        )\n\n    chunk_size = choose_chunk_size(\n        record["text"],\n        int(profile.get("chunking.max_safe_chunk_chars", 24000)),\n    )\n    logger = SharedCallLogger(logs_dir)\n    backends = {\n        "layer01": _make_backend(\n            logger=logger, layer_tag="layer01", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer01_linguistic_expression_extraction"),\n            fallback_max_tokens=max_tokens, fallback_timeout=request_timeout,\n            reasoning_effort=reasoning_effort,\n        ),\n        "layer02": _make_backend(\n            logger=logger, layer_tag="layer02", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer02_candidate_enrichment"),\n            fallback_max_tokens=768, fallback_timeout=90,\n            reasoning_effort=reasoning_effort,\n        ),\n        "layer04": _make_backend(\n            logger=logger, layer_tag="layer04", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer04_candidate_relation_extraction"),\n            fallback_max_tokens=512, fallback_timeout=90,\n            reasoning_effort=reasoning_effort,\n        ),\n        "other": _make_backend(\n            logger=logger, layer_tag="other", model_host=host, api_key=api_key,\n            cfg={}, fallback_max_tokens=1024, fallback_timeout=120,\n            reasoning_effort=reasoning_effort,\n        ),\n    }\n    aliases = read_json(relation_aliases_path)\n    rag_adapter = PriorityOntologyRAGAdapter(\n        seed_ontology,\n        log_path=logs_dir / "ontology_retrieval.jsonl",\n        top_k=int(profile.get("rag.top_k", 10)),\n        query_expansions=profile.get("rag.query_expansions", {}) or {},\n        relation_aliases=aliases,\n        priority_property_ids=profile.get("rag.priority_property_ids", []) or [],\n    )\n    pipeline = build_pipeline(\n        backends=backends,\n        rag_adapter=rag_adapter,\n        profile_config=profile_dict,\n        relation_catalog_path=relation_catalog_path,\n        chunk_size=chunk_size,\n        run_dir=run_dir,\n        workers=workers,\n        verbose=verbose,\n    )\n    state = PipelineState(\n        document=build_document(record, input_jsonl),\n        llm_model=model_name,\n        user_guidance=guidance,\n        seed_ontology=seed_ontology,\n        artifact_dir=str(run_dir),\n        profile_name=profile.name,\n        profile_config=profile_dict,\n    )\n    runner = Runner(\n        pipeline=pipeline,\n        runs_root=str(run_dir.parent),\n        verbose=verbose,\n        max_workers=workers,\n        enable_checkpoints=True,\n        save_chunk_checkpoints=False,\n    )\n    manifest = {\n        "document_id": record["document_id"],\n        "title": record.get("title"),\n        "model_name": model_name,\n        "profile_name": profile.name,\n        "profile_path": str(profile_path),\n        "guidance_path": str(guidance_path),\n        "input_task_guidance_present": bool(record.get("task_guidance")),\n        "ontology_path": str(ontology_path),\n        "ontology_classes": len(seed_ontology.classes_by_uri),\n        "ontology_properties": len(seed_ontology.properties_by_uri),\n        "input_has_gold": False,\n        "chunk_size": chunk_size,\n        "whole_document_single_chunk_expected": len(record["text"]) <= chunk_size,\n        "workers": workers,\n        "per_layer_limits": {\n            name: _layer_cfg(profile_dict, name)\n            for name in [\n                "layer01_linguistic_expression_extraction",\n                "layer02_candidate_enrichment",\n                "layer04_candidate_relation_extraction",\n            ]\n        },\n        "anti_cheating": profile.get("anti_cheating", {}),\n        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(run_dir / "run_manifest.json", manifest)\n\n    console_log = logs_dir / "console.log"\n    errors_path = logs_dir / "pipeline_errors.jsonl"\n    started = time.time()\n    with console_log.open("w", encoding="utf-8") as log_handle:\n        tee_out = Tee(sys.stdout, log_handle)\n        tee_err = Tee(sys.stderr, log_handle)\n        try:\n            with redirect_stdout(tee_out), redirect_stderr(tee_err):\n                final_state = runner.run(state, from_layer=0, to_layer=12, run_dir=run_dir)\n        except Exception as exc:\n            append_jsonl(errors_path, {\n                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n            })\n            raise\n\n    manifest.update({\n        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "elapsed_seconds": round(time.time() - started, 3),\n        "final_counts": state_counts(final_state),\n    })\n    write_json(run_dir / "run_manifest.json", manifest)\n    return final_state\n\n\ndef _assertion_predictions(\n    state: PipelineState,\n    gold: dict[str, Any],\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> list[dict[str, Any]]:\n    entity_aliases = v2.gold_entity_aliases(gold)\n    _, label_to_id = v2.relation_lookup(catalog_path, aliases_path)\n    candidates = {\n        c.candidate_id: c\n        for c in [\n            *(state.entity_candidates or []),\n            *(state.event_candidates or []),\n            *(state.attribute_candidates or []),\n        ]\n    }\n    relations = {c.candidate_id: c for c in state.relation_candidates or []}\n    rows = []\n    for assertion in state.candidate_relation_assertions or []:\n        src = candidates.get(assertion.source_candidate_id)\n        dst = candidates.get(assertion.target_candidate_id)\n        rel = relations.get(assertion.relation_candidate_id)\n        relation_id = v2.map_predicate(\n            assertion.relation_label,\n            getattr(rel, "ontology_hints", []) if rel else [],\n            label_to_id,\n        )\n        head = v2.align_candidate(src, entity_aliases) if src else None\n        tail = v2.align_candidate(dst, entity_aliases) if dst else None\n        rows.append({\n            "assertion_id": assertion.assertion_id,\n            "head_id": head,\n            "relation_id": relation_id,\n            "tail_id": tail,\n            "fully_mapped": bool(head and relation_id and tail),\n            "source_label": assertion.source_candidate_label,\n            "predicate_label": assertion.relation_label,\n            "target_label": assertion.target_candidate_label,\n        })\n    return rows\n\n\ndef write_cumulative_evaluation(\n    run_dir: str | Path,\n    gold: dict[str, Any],\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> list[dict[str, Any]]:\n    rows = []\n    for index, name, state in load_layer_states(run_dir):\n        if index < 4:\n            predictions = []\n        elif index == 4:\n            predictions = _assertion_predictions(state, gold, catalog_path, aliases_path)\n        else:\n            predictions = v2.native_predictions(state, gold, catalog_path, aliases_path)\n        evaluation = v2.strict_evaluate(predictions, gold)\n        rows.append({\n            "layer_index": index,\n            "layer_name": name,\n            "mapped_predictions": sum(1 for p in predictions if p.get("fully_mapped")),\n            **{\n                key: evaluation[key]\n                for key in [\n                    "predicted", "gold", "true_positive", "false_positive",\n                    "false_negative", "precision", "recall", "f1",\n                ]\n            },\n        })\n    analysis_dir = Path(run_dir) / "analysis"\n    analysis_dir.mkdir(parents=True, exist_ok=True)\n    write_json(analysis_dir / "cumulative_strict_evaluation.json", rows)\n    if rows:\n        with (analysis_dir / "cumulative_strict_evaluation.csv").open("w", encoding="utf-8", newline="") as handle:\n            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n            writer.writeheader(); writer.writerows(rows)\n    return rows\n\n\ndef write_relation_trace_v3(\n    run_dir: str | Path,\n    gold: dict[str, Any],\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> list[dict[str, Any]]:\n    run_dir = Path(run_dir)\n    states = {index: state for index, _, state in load_layer_states(run_dir)}\n    final_state = states[max(states)]\n    aliases = v2.gold_entity_aliases(gold)\n    id_to_label, label_to_id = v2.relation_lookup(catalog_path, aliases_path)\n\n    l1 = states.get(1, final_state)\n    l2 = states.get(2, final_state)\n    l3 = states.get(3, final_state)\n    l4 = states.get(4, final_state)\n    l5 = states.get(5, final_state)\n\n    l1_nodes = v2.align_text_values([x.text for x in l1.linguistic_expressions or []], aliases)\n    l2_nodes = v2.align_text_values([x.base_expression.text for x in l2.enriched_expressions or []], aliases)\n    l3_node_candidates = [*(l3.entity_candidates or []), *(l3.event_candidates or []), *(l3.attribute_candidates or [])]\n    l3_nodes = {v2.align_candidate(c, aliases) for c in l3_node_candidates}\n    l3_nodes.discard(None)\n\n    schema_relations: set[str] = set()\n    mention_relations: dict[str, list[str]] = {}\n    for candidate in l3.relation_candidates or []:\n        rid = v2.map_predicate(candidate.canonical_label, candidate.ontology_hints, label_to_id)\n        if not rid:\n            continue\n        schema_relations.add(rid)\n        if candidate.mentions:\n            mention_relations.setdefault(rid, []).append(candidate.candidate_id)\n\n    assertion_rows = _assertion_predictions(l4, gold, catalog_path, aliases_path)\n    exact_assertions = {\n        (row["head_id"], row["relation_id"], row["tail_id"]): row["assertion_id"]\n        for row in assertion_rows if row.get("fully_mapped")\n    }\n    triple_rows = v2.native_predictions(l5, gold, catalog_path, aliases_path)\n    exact_triples = {\n        (row["head_id"], row["relation_id"], row["tail_id"]): row["triple_id"]\n        for row in triple_rows if row.get("fully_mapped")\n    }\n\n    rows = []\n    for head, relation_id, tail in sorted(v2.gold_triples(gold)):\n        head_l1, tail_l1 = head in l1_nodes, tail in l1_nodes\n        head_l2, tail_l2 = head in l2_nodes, tail in l2_nodes\n        head_l3, tail_l3 = head in l3_nodes, tail in l3_nodes\n        schema_available = relation_id in schema_relations\n        mention_ids = mention_relations.get(relation_id, [])\n        mention_available = bool(mention_ids)\n        assertion_id = exact_assertions.get((head, relation_id, tail))\n        triple_id = exact_triples.get((head, relation_id, tail))\n        if not head_l1 or not tail_l1:\n            first_failure = "layer01_expression_extraction"\n        elif not head_l2 or not tail_l2:\n            first_failure = "layer02_enrichment_survival"\n        elif not head_l3 or not tail_l3:\n            first_failure = "layer03_endpoint_resolution"\n        elif not schema_available:\n            first_failure = "layer03_schema_property_unavailable"\n        elif not mention_available:\n            first_failure = "layer03_no_relation_mention_linked_to_property"\n        elif not assertion_id:\n            first_failure = "layer04_exact_endpoint_assignment"\n        elif not triple_id:\n            first_failure = "layer05_triple_materialization_or_mapping"\n        else:\n            first_failure = "survived_to_layer05"\n        rows.append({\n            "head_id": head,\n            "relation_id": relation_id,\n            "relation_label": id_to_label.get(relation_id),\n            "tail_id": tail,\n            "head_available_layer01": head_l1,\n            "tail_available_layer01": tail_l1,\n            "head_available_layer02": head_l2,\n            "tail_available_layer02": tail_l2,\n            "head_available_layer03": head_l3,\n            "tail_available_layer03": tail_l3,\n            "schema_property_available_layer03": schema_available,\n            "predicate_mention_available_layer03": mention_available,\n            "matching_relation_candidate_ids": "|".join(mention_ids),\n            "exact_assertion_id_layer04": assertion_id,\n            "exact_triple_id_layer05": triple_id,\n            "first_failure": first_failure,\n        })\n    analysis_dir = run_dir / "analysis"\n    write_json(analysis_dir / "gold_relation_trace_v3.json", rows)\n    if rows:\n        with (analysis_dir / "gold_relation_trace_v3.csv").open("w", encoding="utf-8", newline="") as handle:\n            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n            writer.writeheader(); writer.writerows(rows)\n    return rows\n\n\ndef write_native_views(\n    run_dir: str | Path,\n    state: PipelineState,\n    gold: dict[str, Any],\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> dict[str, list[dict[str, Any]]]:\n    analysis_dir = Path(run_dir) / "analysis"\n    analysis_dir.mkdir(parents=True, exist_ok=True)\n    mapped = v2.native_predictions(state, gold, catalog_path, aliases_path)\n    relations = {c.candidate_id: c for c in state.relation_candidates or []}\n    lexical = []\n    canonical = []\n    for triple, mapped_row in zip(state.candidate_triples or [], mapped):\n        rel = relations.get(triple.predicate_id)\n        row = {\n            "triple_id": triple.triple_id,\n            "subject_label": triple.subject_label,\n            "predicate_label": triple.predicate_label,\n            "object_label": triple.object_label,\n            "relation_candidate_id": triple.predicate_id,\n            "relation_aliases": list(getattr(rel, "aliases", []) or []) if rel else [],\n            "ontology_hints": list(getattr(rel, "ontology_hints", []) or []) if rel else [],\n            "confidence": triple.confidence,\n            "justification": triple.justification,\n        }\n        lexical.append(row)\n        if mapped_row.get("relation_id"):\n            canonical.append({**row, **{\n                "head_id": mapped_row.get("head_id"),\n                "relation_id": mapped_row.get("relation_id"),\n                "tail_id": mapped_row.get("tail_id"),\n                "fully_mapped": mapped_row.get("fully_mapped"),\n            }})\n    gold_set = v2.gold_triples(gold)\n    not_in_gold = [\n        {**row, "manual_review_required": True}\n        for row in mapped\n        if row.get("fully_mapped") and (row["head_id"], row["relation_id"], row["tail_id"]) not in gold_set\n    ]\n    files = {\n        "native_lexical_triples": lexical,\n        "ontology_canonical_triples": canonical,\n        "strict_docred_predictions": mapped,\n        "predictions_not_in_gold_manual_review": not_in_gold,\n    }\n    for name, rows in files.items():\n        write_json(analysis_dir / f"{name}.json", rows)\n        if rows:\n            with (analysis_dir / f"{name}.csv").open("w", encoding="utf-8", newline="") as handle:\n                writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n                writer.writeheader(); writer.writerows(rows)\n    return files\n\n\ndef analyze_run(\n    *,\n    run_dir: str | Path,\n    gold_jsonl: str | Path,\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> dict[str, Any]:\n    summary = v2.analyze_run(\n        run_dir=run_dir,\n        gold_jsonl=gold_jsonl,\n        catalog_path=catalog_path,\n        aliases_path=aliases_path,\n    )\n    gold_rows = read_jsonl(gold_jsonl)\n    gold = gold_rows[0]\n    states = load_layer_states(run_dir)\n    final_state = states[-1][2]\n    cumulative = write_cumulative_evaluation(run_dir, gold, catalog_path, aliases_path)\n    trace = write_relation_trace_v3(run_dir, gold, catalog_path, aliases_path)\n    views = write_native_views(run_dir, final_state, gold, catalog_path, aliases_path)\n    summary["cumulative_strict_evaluation"] = cumulative\n    summary["failure_counts_v3"] = {}\n    for row in trace:\n        key = row["first_failure"]\n        summary["failure_counts_v3"][key] = summary["failure_counts_v3"].get(key, 0) + 1\n    summary["native_views"] = {key: len(value) for key, value in views.items()}\n    write_json(Path(run_dir) / "analysis/analysis_summary_v3.json", summary)\n    return summary\n', 'docred_native_ablation_v4.py': 'from __future__ import annotations\n\n"""NeoOLAF native DocRED one-document ablation support, v4.\n\nThis experiment module changes no file under ``src/neoolaf``. It keeps the full\nLayer 0--12 NeoOLAF pipeline while improving only profile/guidance-driven\norchestration for DocRED:\n\n* one whole-document Layer 1 call emits named endpoints and one structured\n  relation instance per source/predicate/target pair;\n* Layer 2 deterministically preserves entity/date expressions and sends only\n  relation instances to a contrastive ontology-property classifier;\n* Layer 3 uses NeoOLAF\'s existing role-based typing and promotes only\n  document-supported relation candidates (no empty vocabulary candidates);\n* Layer 4 resolves exact structured endpoints deterministically when possible,\n  otherwise falls back to NeoOLAF\'s existing parallel endpoint-selection task;\n* ontology/domain-range and coarse DocRED type constraints reject impossible\n  assertions such as publication-date -> location;\n* every decision, rejection, prompt, response, retrieval, layer state, log and\n  error remains inspectable.\n\nNo direct DocRED relation extractor, source-entity anchoring, gold leakage,\nclosure rule, or post-hoc relation invention is used.\n"""\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom contextlib import redirect_stderr, redirect_stdout\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any, Iterable\nimport json\nimport re\nimport shutil\nimport sys\nimport threading\nimport time\nimport traceback\n\nimport docred_native_ablation as v2\nimport docred_native_ablation_v3 as v3\n\nfrom neoolaf.core.pipeline import Pipeline\nfrom neoolaf.core.pipeline_state import PipelineState\nfrom neoolaf.core.runner import Runner\nfrom neoolaf.domain.documents import Document\nfrom neoolaf.domain.enriched_expression import EnrichedExpression, EnrichmentEvidence\nfrom neoolaf.domain.linguistic_expression import Evidence, LinguisticExpression\nfrom neoolaf.domain.relation_assertion import CandidateRelationAssertion\nfrom neoolaf.domain.user_guidance import UserGuidance\nfrom neoolaf.grounding.rag.formatting import build_grounding_context\nfrom neoolaf.grounding.rag.types import GroundingRequest\nfrom neoolaf.ontology.loader import SeedOntologyLoader\nfrom neoolaf.profiles.profile_loader import load_document_profile\n\nfrom neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer\nfrom neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer\nfrom neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer\nfrom neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer\nfrom neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer\nfrom neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer\nfrom neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer\nfrom neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer\nfrom neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer\nfrom neoolaf.layers.layer11_inference_completion.component import InferenceCompletionLayer\nfrom neoolaf.layers.layer12_serialization.component import SerializationLayer\n\nfrom experiments.methods.run_neoolaf import (\n    OfflineWebSearchSource,\n    OfflineWikipediaSource,\n    OfflineWikidataSource,\n    OpenAICompatibleBackend,\n    load_user_guidance,\n)\n\n# Re-export notebook helpers.\nread_json = v3.read_json\nread_jsonl = v3.read_jsonl\nwrite_json = v3.write_json\nappend_jsonl = v3.append_jsonl\nload_layer_states = v3.load_layer_states\nstate_counts = v3.state_counts\nsafe_name = v3.safe_name\nTee = v3.Tee\nLAYER_NAMES = v3.LAYER_NAMES\nSharedCallLogger = v3.SharedCallLogger\nTaggedLoggedBackend = v3.TaggedLoggedBackend\nPriorityOntologyRAGAdapter = v3.PriorityOntologyRAGAdapter\nCanonicalizingCandidateTypingResolutionLayer = v3.CanonicalizingCandidateTypingResolutionLayer\n\n\ndef _dedup(values: Iterable[Any]) -> list[str]:\n    result: list[str] = []\n    seen: set[str] = set()\n    for value in values:\n        if value is None:\n            continue\n        text = str(value).strip()\n        if not text or text in seen:\n            continue\n        seen.add(text)\n        result.append(text)\n    return result\n\n\ndef _norm(text: str | None) -> str:\n    value = re.sub(r"\\s+", " ", str(text or "").lower()).strip()\n    value = re.sub(r"[^\\w\\s\\-]", "", value)\n    return value\n\n\ndef _relation_id(text: str | None) -> str | None:\n    match = re.search(r"\\bP\\d+\\b", str(text or ""), re.IGNORECASE)\n    return match.group(0).upper() if match else None\n\n\ndef _parse_relation_instance(text: str | None) -> tuple[str, str, str] | None:\n    parts = [part.strip() for part in str(text or "").split("||")]\n    if len(parts) != 3 or not all(parts):\n        return None\n    return parts[0], parts[1], parts[2]\n\n\ndef _json_block(value: Any, max_chars: int = 14000) -> str:\n    text = json.dumps(value, ensure_ascii=False, indent=2)\n    return text if len(text) <= max_chars else text[:max_chars] + "\\n... [truncated]"\n\n\nclass DocREDRelationInstanceExtractionLayer(LinguisticExpressionExtractionLayer):\n    """Layer 1 with a DocRED-specific output contract expressed via guidance.\n\n    This is still one native Layer 1 call over the whole document. It does not\n    predict canonical ontology properties; it only extracts endpoint nodes and\n    source/predicate/target relation instances for Layer 2 to classify.\n    """\n\n    def __init__(self, *args: Any, decision_log_path: str | Path, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self.decision_log_path = Path(decision_log_path)\n\n    def _prompt(self, state: PipelineState, chunk_text: str) -> list[dict[str, str]]:\n        task = (state.profile_config or {}).get("_input_task_guidance", {}) or {}\n        layer_guidance = (task.get("layer_guidance") or {}).get("layer01", {})\n        examples = task.get("relation_instance_examples") or task.get("relation_examples") or []\n        priority_specs = task.get("priority_relation_specs") or task.get("relation_specs") or []\n        system = """\nYou are NeoOLAF Layer 1 for document-level DocRED linguistic-expression extraction.\n\nExtract two kinds of expressions only:\n1. named or literal relation endpoints;\n2. relation instances grounded in this document.\n\nEndpoint labels must be one of:\n- entity_org\n- entity_per\n- entity_loc\n- entity_time\n- entity_num\n- entity_misc\n\nEvery relation instance MUST use this exact text form:\nSOURCE || LEXICAL RELATION PHRASE || TARGET\nand the label relation_instance.\n\nRules:\n- create one relation instance for each independently supported source-target pair;\n- never collapse a date relation and a location relation into one expression;\n- preserve exact endpoint surface forms whenever possible;\n- relation instances may be explicit or controlled high-confidence inferences;\n- mark inferred relations clearly in justification;\n- use demonyms, aliases, coreference, “the country”, and common city-country\n  containment only when confidence is high and evidence is explained;\n- do not choose P-identifiers here; Layer 2 performs ontology classification;\n- do not output topical phrases that are neither endpoints nor relations;\n- do not expose or use gold annotations.\n\nReturn JSON only:\n{\n  "expressions": [\n    {"text": "Northstar TV", "label": "entity_org", "justification": "Named organization endpoint."},\n    {"text": "Northstar TV || is based in || Harbor City", "label": "relation_instance", "justification": "Explicit ORG->LOC statement; source_type=ORG; target_type=LOC."}\n  ]\n}\n""".strip()\n        user = f"""\nLayer-specific guidance:\n{_json_block(layer_guidance, 5000)}\n\nPriority relation definitions and direction rules (use them only to notice relevant relation instances; do not select the final P-ID here):\n{_json_block(priority_specs, 9000)}\n\nSynthetic examples:\n{_json_block(examples, 8000)}\n\nDocument:\n\\"\\"\\"\n{chunk_text}\n\\"\\"\\"\n\nExtract endpoint expressions and separate structured relation instances. Return JSON only.\n""".strip()\n        return [{"role": "system", "content": system}, {"role": "user", "content": user}]\n\n    def _run(self, state: PipelineState) -> PipelineState:\n        chunks = list(state.document.chunks)\n        if self.max_chunks is not None:\n            chunks = chunks[: self.max_chunks]\n        expressions: list[LinguisticExpression] = []\n        decisions: list[dict[str, Any]] = []\n        expr_counter = 0\n\n        for chunk in chunks:\n            messages = self._prompt(state, chunk.text)\n            raw = self.ollama_backend.chat(\n                model=state.llm_model,\n                messages=messages,\n                temperature=self.temperature,\n            )\n            parsed = self._safe_extract_json(\n                raw_response=raw,\n                state=state,\n                chunk_id=chunk.chunk_id,\n                messages=messages,\n            )\n            if not isinstance(parsed, dict):\n                continue\n            for item in parsed.get("expressions", []) or []:\n                if not isinstance(item, dict):\n                    continue\n                text = str(item.get("text") or "").strip()\n                label = str(item.get("label") or "").strip().lower()\n                justification = str(item.get("justification") or "").strip()\n                if not text:\n                    continue\n                if label == "relation_instance":\n                    parsed_instance = _parse_relation_instance(text)\n                    if parsed_instance is None:\n                        decisions.append({\n                            "status": "rejected",\n                            "reason": "relation_instance_not_source_pipe_predicate_pipe_target",\n                            "text": text,\n                            "label": label,\n                        })\n                        continue\n                elif label not in {\n                    "entity_org", "entity_per", "entity_loc", "entity_time",\n                    "entity_num", "entity_misc",\n                }:\n                    decisions.append({\n                        "status": "rejected",\n                        "reason": "unsupported_layer01_label",\n                        "text": text,\n                        "label": label,\n                    })\n                    continue\n\n                match_span = self._find_expression_span(text, chunk.text)\n                if match_span is not None:\n                    chunk_start, chunk_end = match_span\n                    doc_start = chunk.start_char + chunk_start\n                    doc_end = chunk.start_char + chunk_end\n                    snippet = self._build_snippet(chunk.text, chunk_start, chunk_end)\n                else:\n                    chunk_start = chunk_end = doc_start = doc_end = -1\n                    snippet = chunk.text[:1000]\n\n                expressions.append(LinguisticExpression(\n                    expr_id=f"expr_{expr_counter:05d}",\n                    text=text,\n                    label=label,\n                    justification=justification,\n                    evidence=[Evidence(\n                        chunk_id=chunk.chunk_id,\n                        chunk_start_char=chunk_start,\n                        chunk_end_char=chunk_end,\n                        doc_start_char=doc_start,\n                        doc_end_char=doc_end,\n                        snippet=snippet,\n                    )],\n                ))\n                decisions.append({\n                    "status": "accepted",\n                    "expr_id": f"expr_{expr_counter:05d}",\n                    "text": text,\n                    "label": label,\n                    "relation_instance": _parse_relation_instance(text),\n                    "justification": justification,\n                })\n                expr_counter += 1\n\n        dedup: dict[tuple[str, str], LinguisticExpression] = {}\n        for expr in expressions:\n            dedup.setdefault((_norm(expr.text), expr.label), expr)\n        state.linguistic_expressions = list(dedup.values())\n        self.decision_log_path.parent.mkdir(parents=True, exist_ok=True)\n        write_json(self.decision_log_path, decisions)\n        state.log(\n            f"[{self.name}] DocRED structured extraction; expressions={len(state.linguistic_expressions)}; "\n            f"relation_instances={sum(1 for x in state.linguistic_expressions if x.label == \'relation_instance\')}"\n        )\n        return state\n\n\nclass SelectiveContrastiveCandidateEnrichmentLayer(CandidateEnrichmentLayer):\n    """Layer 2: deterministic nodes, parallel contrastive relation linking."""\n\n    def __init__(\n        self,\n        *args: Any,\n        relation_catalog_path: str | Path,\n        decision_log_path: str | Path,\n        **kwargs: Any,\n    ) -> None:\n        super().__init__(*args, **kwargs)\n        catalog = read_json(relation_catalog_path)["relations"]\n        self.catalog = {item["relation_id"].upper(): item for item in catalog}\n        self.decision_log_path = Path(decision_log_path)\n        self._decision_lock = threading.Lock()\n        self._decisions: list[dict[str, Any]] = []\n\n    def _is_relation(self, expr: LinguisticExpression) -> bool:\n        return expr.label == "relation_instance" or _parse_relation_instance(expr.text) is not None\n\n    def _compact_catalog(self, relation_ids: Iterable[str]) -> list[dict[str, Any]]:\n        rows: list[dict[str, Any]] = []\n        for relation_id in _dedup(relation_ids):\n            item = self.catalog.get(relation_id.upper())\n            if not item:\n                continue\n            rows.append({\n                "relation_id": item["relation_id"],\n                "label": item["label"],\n                "definition": item.get("comment") or "",\n                "domain_uris": item.get("domain_uris") or [],\n                "range_uris": item.get("range_uris") or [],\n            })\n        return rows\n\n    def _candidate_ids_from_grounding(self, grounding_text: str) -> list[str]:\n        return _dedup(match.upper() for match in re.findall(r"\\bP\\d+\\b", grounding_text or "", re.I))\n\n    def _contrastive_prompt(\n        self,\n        *,\n        expr: LinguisticExpression,\n        state: PipelineState,\n        grounding_text: str,\n    ) -> list[dict[str, str]]:\n        task = (state.profile_config or {}).get("_input_task_guidance", {}) or {}\n        parsed_instance = _parse_relation_instance(expr.text)\n        if parsed_instance is None:\n            raise ValueError(f"Not a structured relation instance: {expr.text}")\n        source, predicate, target = parsed_instance\n        layer_guidance = (task.get("layer_guidance") or {}).get("layer02", {})\n        contrastive_rules = task.get("contrastive_rules") or []\n        examples = task.get("relation_examples") or []\n        allowed = task.get("allowed_relation_ids") or list(self.catalog)\n        retrieved_ids = self._candidate_ids_from_grounding(grounding_text)\n        priority_ids = task.get("priority_relation_ids") or []\n        candidate_ids = _dedup([*retrieved_ids, *priority_ids])\n        candidate_ids = [x for x in candidate_ids if x in set(allowed) and x in self.catalog][:12]\n        if not candidate_ids:\n            candidate_ids = [x for x in priority_ids if x in self.catalog][:12]\n\n        system = """\nYou are NeoOLAF Layer 2 for ontology-constrained candidate enrichment.\n\nYou receive exactly one relation instance already extracted by Layer 1. Your\nonly semantic task is to select the single best DocRED/Wikidata property from\nthe supplied ontology candidates, or return found=false when none is supported.\n\nUse source type, target type, direction, predicate specificity, ontology\ndefinition, domain and range. Prefer the most specific supported property.\nContrast close alternatives explicitly. Never create a new source, target or\nrelation instance. Never use gold annotations.\n\nReturn JSON only:\n{\n  "found": true,\n  "selected_relation_id": "P159",\n  "selected_relation_label": "headquarters location",\n  "aliases": ["is based in"],\n  "synonyms": [],\n  "lexical_variants": [],\n  "definition": "...",\n  "decision": "P159 is preferred over P131 because ..."\n}\n\nor {"found": false, "decision": "..."}.\n""".strip()\n        user = f"""\nRelation instance:\n- source: {source}\n- lexical predicate: {predicate}\n- target: {target}\n- Layer 1 justification: {expr.justification}\n\nLayer-specific rules:\n{_json_block(layer_guidance, 5000)}\n\nContrastive rules:\n{_json_block(contrastive_rules, 7000)}\n\nSynthetic examples:\n{_json_block(examples, 7000)}\n\nRetrieved ontology evidence:\n{grounding_text}\n\nAllowed candidate properties for this decision:\n{_json_block(self._compact_catalog(candidate_ids), 12000)}\n\nChoose exactly one relation only when supported. Return JSON only.\n""".strip()\n        return [{"role": "system", "content": system}, {"role": "user", "content": user}]\n\n    def _record_decision(self, row: dict[str, Any]) -> None:\n        with self._decision_lock:\n            self._decisions.append(row)\n\n    def _process_relation_contrastive(self, expr: LinguisticExpression, state: PipelineState) -> EnrichedExpression:\n        parsed_instance = _parse_relation_instance(expr.text)\n        if parsed_instance is None:\n            raise ValueError(f"Malformed relation instance: {expr.text}")\n        source, predicate, target = parsed_instance\n        grounding_text = ""\n        if self.rag_adapter is not None:\n            grounding = self.rag_adapter.ground(GroundingRequest(\n                layer_name=self.name,\n                query=f"{predicate} {source} {target}",\n                payload={\n                    "relation_instance": expr.text,\n                    "source": source,\n                    "predicate": predicate,\n                    "target": target,\n                    "justification": expr.justification,\n                },\n                preferred_sources=["ontology"],\n                top_k=10,\n            ))\n            grounding_text = build_grounding_context(grounding)\n\n        messages = self._contrastive_prompt(expr=expr, state=state, grounding_text=grounding_text)\n        raw = self.ollama_backend.chat(\n            model=state.llm_model,\n            messages=messages,\n            temperature=0.0,\n        )\n        parsed = self.ollama_backend.extract_json(raw)\n        if not isinstance(parsed, dict):\n            raise ValueError("Layer 2 contrastive response is not a JSON object")\n        found = bool(parsed.get("found", False))\n        selected_id = _relation_id(parsed.get("selected_relation_id"))\n        if found and selected_id not in self.catalog:\n            raise ValueError(f"Layer 2 selected invalid relation ID: {selected_id}")\n\n        if found:\n            item = self.catalog[selected_id]\n            canonical = f"{selected_id} : {item[\'label\']}"\n            hints = _dedup([\n                f"controlled_relation:{canonical}",\n                "promote_to_ontology:true",\n                item.get("uri"),\n                item.get("label"),\n                f"source_label:{source}",\n                f"target_label:{target}",\n                f"lexical_predicate:{predicate}",\n                f"source_type:{self._type_from_justification(expr.justification, \'source_type\')}",\n                f"target_type:{self._type_from_justification(expr.justification, \'target_type\')}",\n                f"domain:{\', \'.join(item.get(\'domain_uris\') or [])}" if item.get("domain_uris") else None,\n                f"range:{\', \'.join(item.get(\'range_uris\') or [])}" if item.get("range_uris") else None,\n                f"contrastive_decision:{parsed.get(\'decision\', \'\')}",\n            ])\n            definition = str(parsed.get("definition") or item.get("comment") or "").strip()\n        else:\n            canonical = None\n            hints = _dedup([\n                "promote_to_ontology:false",\n                f"source_label:{source}",\n                f"target_label:{target}",\n                f"lexical_predicate:{predicate}",\n                f"contrastive_decision:{parsed.get(\'decision\', \'\')}",\n            ])\n            definition = str(parsed.get("decision") or "No supported ontology relation selected.")\n\n        self._record_decision({\n            "expr_id": expr.expr_id,\n            "relation_instance": expr.text,\n            "source": source,\n            "predicate": predicate,\n            "target": target,\n            "found": found,\n            "selected_relation_id": selected_id,\n            "canonical_relation": canonical,\n            "decision": parsed.get("decision"),\n            "ontology_hints": hints,\n        })\n        return EnrichedExpression(\n            base_expression=expr,\n            aliases=_dedup([expr.text, predicate, *(parsed.get("aliases") or [])]),\n            synonyms=_dedup(parsed.get("synonyms") or []),\n            lexical_variants=_dedup(parsed.get("lexical_variants") or []),\n            alias_sources={value: ["source" if value in {expr.text, predicate} else "llm"] for value in _dedup([expr.text, predicate, *(parsed.get("aliases") or [])])},\n            synonym_sources={value: ["llm"] for value in _dedup(parsed.get("synonyms") or [])},\n            lexical_variant_sources={value: ["llm"] for value in _dedup(parsed.get("lexical_variants") or [])},\n            definition=definition,\n            ontology_hints=hints,\n            enrichment_evidence=[EnrichmentEvidence(\n                source="llm",\n                content=json.dumps(parsed, ensure_ascii=False),\n                reference=state.llm_model,\n            )],\n        )\n\n    @staticmethod\n    def _type_from_justification(justification: str, key: str) -> str:\n        match = re.search(rf"\\b{re.escape(key)}\\s*=\\s*([A-Za-z]+)", justification or "", re.I)\n        return match.group(1).upper() if match else "UNKNOWN"\n\n    def _process_relation_with_retries(self, index: int, expr: LinguisticExpression, state: PipelineState) -> EnrichedExpression | None:\n        last_exc: Exception | None = None\n        for attempt in range(self.retry_failed_calls + 1):\n            try:\n                return self._process_relation_contrastive(expr, state)\n            except Exception as exc:\n                last_exc = exc\n                if attempt < self.retry_failed_calls and self.retry_sleep_seconds > 0:\n                    time.sleep(self.retry_sleep_seconds)\n        if last_exc is not None:\n            self._record_failure(expr, index, last_exc, attempt=self.retry_failed_calls)\n        return None\n\n    def _run(self, state: PipelineState) -> PipelineState:\n        expressions = list(state.linguistic_expressions)\n        if self.max_expressions is not None:\n            expressions = expressions[: self.max_expressions]\n        self._failed_details = []\n        self._decisions = []\n\n        relation_jobs: list[tuple[int, LinguisticExpression]] = []\n        enriched_by_index: dict[int, EnrichedExpression] = {}\n        for index, expr in enumerate(expressions):\n            if self._is_relation(expr):\n                relation_jobs.append((index, expr))\n            else:\n                enriched_by_index[index] = self._process_expression_conservative(expr, state)\n\n        with ThreadPoolExecutor(max_workers=self.max_concurrency) as executor:\n            futures = {\n                executor.submit(self._process_relation_with_retries, index, expr, state): index\n                for index, expr in relation_jobs\n            }\n            for future in as_completed(futures):\n                index = futures[future]\n                try:\n                    result = future.result()\n                except Exception as exc:\n                    self._record_failure(expressions[index], index, exc, attempt="unhandled")\n                    continue\n                if result is not None:\n                    enriched_by_index[index] = result\n\n        state.enriched_expressions = [enriched_by_index[i] for i in sorted(enriched_by_index)]\n        self._save_failed_expressions(state)\n        self.decision_log_path.parent.mkdir(parents=True, exist_ok=True)\n        write_json(self.decision_log_path, sorted(self._decisions, key=lambda x: x.get("expr_id", "")))\n        state.log(\n            f"[{self.name}] selective enrichment; total={len(expressions)}; "\n            f"deterministic_nodes={len(expressions)-len(relation_jobs)}; "\n            f"relation_llm_calls={len(relation_jobs)}; enriched={len(state.enriched_expressions)}; "\n            f"failed={len(self._failed_details)}"\n        )\n        return state\n\n\nclass StructuredEndpointValidatedRelationLayer(v3.ParallelCandidateRelationExtractionLayer):\n    """Layer 4 with exact endpoint resolution and coarse type/range validation."""\n\n    def __init__(\n        self,\n        *args: Any,\n        endpoint_log_path: str | Path,\n        rejection_log_path: str | Path,\n        type_constraints: dict[str, Any] | None = None,\n        **kwargs: Any,\n    ) -> None:\n        super().__init__(*args, **kwargs)\n        self.endpoint_log_path = Path(endpoint_log_path)\n        self.rejection_log_path = Path(rejection_log_path)\n        self.type_constraints = {str(k).upper(): v for k, v in (type_constraints or {}).items()}\n        self._decision_lock = threading.Lock()\n        self._endpoint_decisions: list[dict[str, Any]] = []\n        self._rejections: list[dict[str, Any]] = []\n\n    @staticmethod\n    def _hint_value(hints: Iterable[str], prefix: str) -> str | None:\n        prefix_l = prefix.lower()\n        for hint in hints or []:\n            text = str(hint)\n            if text.lower().startswith(prefix_l):\n                return text.split(":", 1)[1].strip()\n        return None\n\n    @staticmethod\n    def _candidate_roles(candidate: Any) -> set[str]:\n        roles: set[str] = set()\n        for hint in candidate.ontology_hints or []:\n            text = str(hint)\n            if text.lower().startswith("semantic_role:"):\n                role = text.split(":", 1)[1].strip().lower()\n                if "org" in role or role in {"organization", "organisation"}:\n                    roles.add("ORG")\n                elif "per" in role or role in {"person", "human"}:\n                    roles.add("PER")\n                elif "loc" in role or role in {"location", "city", "country"}:\n                    roles.add("LOC")\n                elif "time" in role or role == "date":\n                    roles.add("TIME")\n                elif "num" in role or role == "number":\n                    roles.add("NUM")\n                else:\n                    roles.add("MISC")\n            upper = text.upper()\n            for coarse in ["ORG", "PER", "LOC", "TIME", "NUM", "MISC"]:\n                if re.search(rf"\\b{coarse}\\b", upper):\n                    roles.add(coarse)\n        return roles or {"UNKNOWN"}\n\n    def _candidate_matches(self, state: PipelineState, expected: str) -> list[Any]:\n        expected_n = _norm(expected)\n        result: list[Any] = []\n        for candidate in [\n            *(state.entity_candidates or []),\n            *(state.event_candidates or []),\n            *(state.attribute_candidates or []),\n        ]:\n            labels = [candidate.canonical_label, *(candidate.aliases or []), *(m.text for m in candidate.mentions or [])]\n            if any(_norm(label) == expected_n for label in labels):\n                result.append(candidate)\n        return result\n\n    def _valid_types(self, relation_id: str | None, source: Any, target: Any) -> tuple[bool, str]:\n        if not relation_id or relation_id not in self.type_constraints:\n            return True, "no_coarse_constraint"\n        cfg = self.type_constraints[relation_id]\n        allowed_source = set(cfg.get("source") or [])\n        allowed_target = set(cfg.get("target") or [])\n        source_roles = self._candidate_roles(source)\n        target_roles = self._candidate_roles(target)\n        source_ok = not allowed_source or bool(source_roles & allowed_source)\n        target_ok = not allowed_target or bool(target_roles & allowed_target)\n        reason = (\n            f"relation={relation_id}; source_roles={sorted(source_roles)} allowed={sorted(allowed_source)}; "\n            f"target_roles={sorted(target_roles)} allowed={sorted(allowed_target)}"\n        )\n        return source_ok and target_ok, reason\n\n    def _record_endpoint(self, row: dict[str, Any]) -> None:\n        with self._decision_lock:\n            self._endpoint_decisions.append(row)\n\n    def _record_rejection(self, row: dict[str, Any]) -> None:\n        with self._decision_lock:\n            self._rejections.append(row)\n\n    def _process_relation_mention(\n        self,\n        *,\n        state: PipelineState,\n        relation_mention: dict[str, Any],\n        chunk_to_local_candidates: dict[str, list[dict[str, Any]]],\n    ) -> CandidateRelationAssertion | None:\n        relation_candidate = relation_mention["relation_candidate"]\n        hints = list(relation_candidate.ontology_hints or [])\n        source_label = self._hint_value(hints, "source_label:")\n        target_label = self._hint_value(hints, "target_label:")\n        relation_id = _relation_id(relation_candidate.canonical_label) or _relation_id(" ".join(hints))\n\n        if source_label and target_label:\n            source_matches = self._candidate_matches(state, source_label)\n            target_matches = self._candidate_matches(state, target_label)\n            if len(source_matches) == 1 and len(target_matches) == 1:\n                source = source_matches[0]\n                target = target_matches[0]\n                valid, reason = self._valid_types(relation_id, source, target)\n                if not valid:\n                    self._record_rejection({\n                        "mode": "structured_exact",\n                        "relation_candidate_id": relation_candidate.candidate_id,\n                        "relation_id": relation_id,\n                        "source": source.canonical_label,\n                        "target": target.canonical_label,\n                        "reason": reason,\n                    })\n                    return None\n                self._record_endpoint({\n                    "mode": "structured_exact",\n                    "relation_candidate_id": relation_candidate.candidate_id,\n                    "relation_id": relation_id,\n                    "source": source.canonical_label,\n                    "target": target.canonical_label,\n                    "llm_call": False,\n                    "type_validation": reason,\n                })\n                return CandidateRelationAssertion(\n                    assertion_id="pending",\n                    relation_candidate_id=relation_candidate.candidate_id,\n                    relation_label=relation_candidate.canonical_label,\n                    source_candidate_id=source.candidate_id,\n                    source_candidate_label=source.canonical_label,\n                    source_candidate_type=source.candidate_type,\n                    target_candidate_id=target.candidate_id,\n                    target_candidate_label=target.canonical_label,\n                    target_candidate_type=target.candidate_type,\n                    chunk_id=relation_mention["chunk_id"],\n                    justification=(\n                        "Exact Layer 1 source/target labels resolved to Layer 3 candidates; "\n                        "predicate was canonically linked by Layer 2."\n                    ),\n                    confidence=1.0,\n                    evidence=relation_mention["evidence"],\n                )\n\n        assertion = super()._process_relation_mention(\n            state=state,\n            relation_mention=relation_mention,\n            chunk_to_local_candidates=chunk_to_local_candidates,\n        )\n        if assertion is None:\n            self._record_endpoint({\n                "mode": "native_llm_fallback",\n                "relation_candidate_id": relation_candidate.candidate_id,\n                "relation_id": relation_id,\n                "source_hint": source_label,\n                "target_hint": target_label,\n                "llm_call": True,\n                "found": False,\n            })\n            return None\n        source = self._find_candidate_by_id(state, assertion.source_candidate_id)\n        target = self._find_candidate_by_id(state, assertion.target_candidate_id)\n        valid, reason = self._valid_types(relation_id, source, target)\n        if not valid:\n            self._record_rejection({\n                "mode": "native_llm_fallback",\n                "relation_candidate_id": relation_candidate.candidate_id,\n                "relation_id": relation_id,\n                "source": assertion.source_candidate_label,\n                "target": assertion.target_candidate_label,\n                "reason": reason,\n            })\n            return None\n        self._record_endpoint({\n            "mode": "native_llm_fallback",\n            "relation_candidate_id": relation_candidate.candidate_id,\n            "relation_id": relation_id,\n            "source": assertion.source_candidate_label,\n            "target": assertion.target_candidate_label,\n            "llm_call": True,\n            "found": True,\n            "type_validation": reason,\n        })\n        return assertion\n\n    def _run(self, state: PipelineState) -> PipelineState:\n        self._endpoint_decisions = []\n        self._rejections = []\n        state = super()._run(state)\n        self.endpoint_log_path.parent.mkdir(parents=True, exist_ok=True)\n        write_json(self.endpoint_log_path, self._endpoint_decisions)\n        write_json(self.rejection_log_path, self._rejections)\n        state.log(\n            f"[{self.name}] endpoint modes: exact={sum(x.get(\'mode\') == \'structured_exact\' for x in self._endpoint_decisions)}; "\n            f"llm={sum(x.get(\'mode\') == \'native_llm_fallback\' for x in self._endpoint_decisions)}; "\n            f"rejected_by_type={len(self._rejections)}"\n        )\n        return state\n\n\ndef _layer_cfg(profile: dict[str, Any], layer_name: str) -> dict[str, Any]:\n    return dict((profile.get("layers") or {}).get(layer_name) or {})\n\n\ndef _make_backend(\n    *,\n    logger: SharedCallLogger,\n    layer_tag: str,\n    model_host: str,\n    api_key: str,\n    cfg: dict[str, Any],\n    fallback_max_tokens: int,\n    fallback_timeout: int,\n    reasoning_effort: str,\n) -> TaggedLoggedBackend:\n    core = OpenAICompatibleBackend(\n        backend_name="openrouter",\n        host=model_host,\n        api_key=api_key,\n        timeout=int(cfg.get("request_timeout_seconds", fallback_timeout)),\n        max_tokens=int(cfg.get("max_output_tokens", fallback_max_tokens)),\n        reasoning_effort=reasoning_effort,\n        exclude_reasoning=True,\n    )\n    return TaggedLoggedBackend(\n        core,\n        logger,\n        layer_tag=layer_tag,\n        response_hard_cap_chars=cfg.get("response_hard_cap_chars"),\n    )\n\n\ndef choose_chunk_size(text: str, max_safe_chars: int = 24000) -> int:\n    return v2.choose_chunk_size(text, max_safe_chars)\n\n\ndef build_document(record: dict[str, Any], source_path: str | Path) -> Document:\n    return v2.build_document(record, source_path)\n\n\ndef build_pipeline(\n    *,\n    backends: dict[str, TaggedLoggedBackend],\n    rag_adapter: PriorityOntologyRAGAdapter,\n    profile_config: dict[str, Any],\n    relation_catalog_path: str | Path,\n    chunk_size: int,\n    run_dir: str | Path,\n    workers: int = 12,\n    verbose: bool = True,\n) -> Pipeline:\n    workers = max(1, int(workers))\n    l2_cfg = _layer_cfg(profile_config, "layer02_candidate_enrichment")\n    l4_cfg = _layer_cfg(profile_config, "layer04_candidate_relation_extraction")\n    retry_default = int((profile_config.get("orchestration") or {}).get("retry_failed_calls", 1))\n    sleep_default = float((profile_config.get("orchestration") or {}).get("retry_sleep_seconds", 1.0))\n    l2_workers = int(l2_cfg.get("max_concurrency", workers))\n    l4_workers = int(l4_cfg.get("max_concurrency", min(workers, 8)))\n    run_dir = Path(run_dir)\n\n    layers = [\n        PreprocessingLayer(\n            chunk_size=chunk_size,\n            overlap=0,\n            enable_chunking=True,\n            translate=False,\n            save_intermediate=True,\n            verbose=verbose,\n            profile_config=profile_config,\n        ),\n        DocREDRelationInstanceExtractionLayer(\n            backends["layer01"],\n            decision_log_path=run_dir / "run_logs/layer01_relation_instances.json",\n            max_chunks=1,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_backend=rag_adapter,\n            max_concurrency=1,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n            rag_enabled=False,\n        ),\n        SelectiveContrastiveCandidateEnrichmentLayer(\n            backends["layer02"],\n            wikipedia_source=OfflineWikipediaSource(),\n            wikidata_source=OfflineWikidataSource(),\n            web_search_source=OfflineWebSearchSource(),\n            relation_catalog_path=relation_catalog_path,\n            decision_log_path=run_dir / "run_logs/layer02_contrastive_decisions.json",\n            max_expressions=None,\n            use_web_search=False,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=l2_workers,\n            retry_failed_calls=int(l2_cfg.get("retry_failed_calls", retry_default)),\n            retry_sleep_seconds=sleep_default,\n        ),\n        CanonicalizingCandidateTypingResolutionLayer(\n            backends["other"],\n            relation_catalog_path=relation_catalog_path,\n            max_expressions=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=1,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        StructuredEndpointValidatedRelationLayer(\n            backends["layer04"],\n            endpoint_log_path=run_dir / "run_logs/layer04_endpoint_assignment.json",\n            rejection_log_path=run_dir / "run_logs/layer04_constraint_rejections.json",\n            type_constraints=(profile_config.get("docred_type_constraints") or {}),\n            max_relation_mentions=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=l4_workers,\n            retry_failed_calls=int(l4_cfg.get("retry_failed_calls", retry_default)),\n            retry_sleep_seconds=sleep_default,\n            max_attempts_per_relation=int(l4_cfg.get("max_attempts_per_relation", 1)),\n            retry_wait_seconds=float(l4_cfg.get("retry_wait_seconds", 0.5)),\n            failure_log_path=run_dir / "run_logs/layer04_relation_errors.jsonl",\n        ),\n        CandidateTripleGenerationLayer(\n            max_assertions=None,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n            save_intermediate=True,\n            verbose=verbose,\n        ),\n        ConceptRelationInductionLayer(\n            backends["other"],\n            max_concept_inputs=None,\n            max_relation_inputs=None,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n        ),\n        HierarchisationLayer(\n            backends["other"],\n            max_concept_pairs=None,\n            max_relation_pairs=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            rag_adapter=rag_adapter,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        AxiomSchemataExtractionLayer(\n            backends["other"],\n            max_relation_schema_inputs=None,\n            max_subclass_inputs=None,\n            temperature=0.0,\n            rag_adapter=rag_adapter,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        GeneralAxiomExtractionLayer(\n            backends["other"],\n            max_schema_inputs=None,\n            max_description_inputs=None,\n            temperature=0.0,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        ValidationReasoningLayer(\n            max_triples=None,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        InferenceCompletionLayer(\n            max_inferred_triples=None,\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n            retry_failed_calls=retry_default,\n            retry_sleep_seconds=sleep_default,\n        ),\n        SerializationLayer(\n            output_subdir="exports",\n            save_intermediate=True,\n            verbose=verbose,\n            max_concurrency=workers,\n        ),\n    ]\n    return Pipeline(layers=layers, verbose=verbose, continue_from_last=False)\n\n\ndef run_native_pipeline(\n    *,\n    project_root: str | Path,\n    input_jsonl: str | Path,\n    ontology_path: str | Path,\n    profile_path: str | Path,\n    guidance_path: str | Path,\n    relation_catalog_path: str | Path,\n    relation_aliases_path: str | Path,\n    run_dir: str | Path,\n    model_name: str,\n    api_key: str,\n    host: str = "https://openrouter.ai/api/v1",\n    workers: int = 12,\n    max_tokens: int = 4096,\n    request_timeout: int = 180,\n    reasoning_effort: str = "minimal",\n    verbose: bool = True,\n    clean_run_dir: bool = True,\n) -> PipelineState:\n    project_root = Path(project_root).resolve()\n    input_jsonl = Path(input_jsonl).resolve()\n    ontology_path = Path(ontology_path).resolve()\n    profile_path = Path(profile_path).resolve()\n    guidance_path = Path(guidance_path).resolve()\n    relation_catalog_path = Path(relation_catalog_path).resolve()\n    relation_aliases_path = Path(relation_aliases_path).resolve()\n    run_dir = Path(run_dir).resolve()\n    if clean_run_dir and run_dir.exists():\n        shutil.rmtree(run_dir)\n    run_dir.mkdir(parents=True, exist_ok=True)\n    logs_dir = run_dir / "run_logs"\n    logs_dir.mkdir(parents=True, exist_ok=True)\n\n    records = read_jsonl(input_jsonl)\n    if len(records) != 1:\n        raise ValueError(f"This notebook expects exactly one input document, found {len(records)}")\n    record = records[0]\n    if "entities" in record or "relations" in record:\n        raise ValueError("Pipeline input must not contain gold entities or relations.")\n    if not api_key:\n        raise ValueError("OPENROUTER_API_KEY is not set.")\n\n    profile = load_document_profile(profile_path=profile_path)\n    profile_dict = profile.to_state_dict()\n    profile_dict["_input_task_guidance"] = record.get("task_guidance") or {}\n    guidance = load_user_guidance(str(guidance_path)) or UserGuidance()\n    guidance = v3.merge_input_task_guidance(guidance, record)\n    write_json(run_dir / "input_task_guidance.json", record.get("task_guidance") or {})\n    write_json(run_dir / "effective_user_guidance.json", asdict(guidance))\n\n    seed_ontology = SeedOntologyLoader().load(str(ontology_path))\n    if len(seed_ontology.properties_by_uri) < 90:\n        raise RuntimeError(f"Only {len(seed_ontology.properties_by_uri)} ontology properties were loaded.")\n\n    chunk_size = choose_chunk_size(record["text"], int(profile.get("chunking.max_safe_chunk_chars", 24000)))\n    logger = SharedCallLogger(logs_dir)\n    backends = {\n        "layer01": _make_backend(\n            logger=logger, layer_tag="layer01", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer01_linguistic_expression_extraction"),\n            fallback_max_tokens=max_tokens, fallback_timeout=request_timeout,\n            reasoning_effort=reasoning_effort,\n        ),\n        "layer02": _make_backend(\n            logger=logger, layer_tag="layer02_relation_only", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer02_candidate_enrichment"),\n            fallback_max_tokens=512, fallback_timeout=60,\n            reasoning_effort=reasoning_effort,\n        ),\n        "layer04": _make_backend(\n            logger=logger, layer_tag="layer04_fallback_only", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer04_candidate_relation_extraction"),\n            fallback_max_tokens=384, fallback_timeout=60,\n            reasoning_effort=reasoning_effort,\n        ),\n        "other": _make_backend(\n            logger=logger, layer_tag="other", model_host=host, api_key=api_key,\n            cfg={}, fallback_max_tokens=768, fallback_timeout=90,\n            reasoning_effort=reasoning_effort,\n        ),\n    }\n    aliases = read_json(relation_aliases_path)\n    rag_adapter = PriorityOntologyRAGAdapter(\n        seed_ontology,\n        log_path=logs_dir / "ontology_retrieval.jsonl",\n        top_k=int(profile.get("rag.top_k", 10)),\n        query_expansions=profile.get("rag.query_expansions", {}) or {},\n        relation_aliases=aliases,\n        priority_property_ids=profile.get("rag.priority_property_ids", []) or [],\n    )\n    pipeline = build_pipeline(\n        backends=backends,\n        rag_adapter=rag_adapter,\n        profile_config=profile_dict,\n        relation_catalog_path=relation_catalog_path,\n        chunk_size=chunk_size,\n        run_dir=run_dir,\n        workers=workers,\n        verbose=verbose,\n    )\n    state = PipelineState(\n        document=build_document(record, input_jsonl),\n        llm_model=model_name,\n        user_guidance=guidance,\n        seed_ontology=seed_ontology,\n        artifact_dir=str(run_dir),\n        profile_name=profile.name,\n        profile_config=profile_dict,\n    )\n    runner = Runner(\n        pipeline=pipeline,\n        runs_root=str(run_dir.parent),\n        verbose=verbose,\n        max_workers=workers,\n        enable_checkpoints=True,\n        save_chunk_checkpoints=False,\n    )\n    manifest = {\n        "document_id": record["document_id"],\n        "title": record.get("title"),\n        "model_name": model_name,\n        "profile_name": profile.name,\n        "profile_path": str(profile_path),\n        "guidance_path": str(guidance_path),\n        "ontology_path": str(ontology_path),\n        "ontology_classes": len(seed_ontology.classes_by_uri),\n        "ontology_properties": len(seed_ontology.properties_by_uri),\n        "input_has_gold": False,\n        "chunk_size": chunk_size,\n        "whole_document_single_chunk_expected": len(record["text"]) <= chunk_size,\n        "workers": workers,\n        "selective_layer02": True,\n        "structured_layer04_exact_endpoint_resolution": True,\n        "empty_schema_candidates_injected": False,\n        "anti_cheating": profile.get("anti_cheating", {}),\n        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(run_dir / "run_manifest.json", manifest)\n\n    console_log = logs_dir / "console.log"\n    errors_path = logs_dir / "pipeline_errors.jsonl"\n    started = time.time()\n    with console_log.open("w", encoding="utf-8") as handle:\n        tee_out = Tee(sys.stdout, handle)\n        tee_err = Tee(sys.stderr, handle)\n        try:\n            with redirect_stdout(tee_out), redirect_stderr(tee_err):\n                final_state = runner.run(state, from_layer=0, to_layer=12, run_dir=run_dir)\n        except Exception as exc:\n            append_jsonl(errors_path, {\n                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n            })\n            raise\n\n    manifest.update({\n        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "elapsed_seconds": round(time.time() - started, 3),\n        "final_counts": state_counts(final_state),\n    })\n    write_json(run_dir / "run_manifest.json", manifest)\n    return final_state\n\n\ndef write_relation_trace_v4(*, run_dir: str | Path, gold_jsonl: str | Path, catalog_path: str | Path) -> list[dict[str, Any]]:\n    """Corrected layer-first relation trace for one DocRED document."""\n    run_dir = Path(run_dir)\n    gold = read_jsonl(gold_jsonl)[0]\n    states = {index: state for index, _, state in load_layer_states(run_dir)}\n    l1 = states.get(1)\n    l2 = states.get(2)\n    l3 = states.get(3)\n    l4 = states.get(4)\n    l5 = states.get(5)\n\n    relation_decisions = read_json(run_dir / "run_logs/layer02_contrastive_decisions.json") if (run_dir / "run_logs/layer02_contrastive_decisions.json").is_file() else []\n    decision_by_id = {row.get("selected_relation_id"): [] for row in relation_decisions if row.get("selected_relation_id")}\n    for row in relation_decisions:\n        if row.get("selected_relation_id"):\n            decision_by_id.setdefault(row["selected_relation_id"], []).append(row)\n\n    def entity_label(entity_id: str) -> str:\n        entity = gold["entities"][entity_id]\n        mentions = entity.get("mentions") or []\n        return mentions[0]["trigger_word"] if mentions else entity_id\n\n    rows: list[dict[str, Any]] = []\n    for relation_label, pairs in gold.get("relations", {}).items():\n        relation_id = _relation_id(relation_label)\n        for source_id, target_id in pairs:\n            source_label = entity_label(source_id)\n            target_label = entity_label(target_id)\n            l1_texts = [_norm(x.text) for x in (l1.linguistic_expressions if l1 else [])]\n            source_l1 = _norm(source_label) in l1_texts\n            target_l1 = _norm(target_label) in l1_texts\n            l1_instances = [\n                x for x in (l1.linguistic_expressions if l1 else [])\n                if x.label == "relation_instance" and _parse_relation_instance(x.text)\n            ]\n            exact_l1_instance = any(\n                _norm(_parse_relation_instance(x.text)[0]) == _norm(source_label)\n                and _norm(_parse_relation_instance(x.text)[2]) == _norm(target_label)\n                for x in l1_instances\n            )\n            linked_l2 = any(\n                row.get("selected_relation_id") == relation_id\n                and _norm(row.get("source")) == _norm(source_label)\n                and _norm(row.get("target")) == _norm(target_label)\n                for row in relation_decisions\n            )\n            linked_l3 = any(\n                _relation_id(c.canonical_label) == relation_id\n                and any(\n                    (lambda parsed: parsed and _norm(parsed[0]) == _norm(source_label) and _norm(parsed[2]) == _norm(target_label))(\n                        _parse_relation_instance(alias)\n                    )\n                    for alias in [*list(c.aliases or []), *(m.text for m in c.mentions or [])]\n                )\n                for c in (l3.relation_candidates if l3 else [])\n            )\n            assertion_l4 = any(\n                _relation_id(a.relation_label) == relation_id\n                and _norm(a.source_candidate_label) == _norm(source_label)\n                and _norm(a.target_candidate_label) == _norm(target_label)\n                for a in (l4.candidate_relation_assertions if l4 else [])\n            )\n            triple_l5 = any(\n                _relation_id(t.predicate_label) == relation_id\n                and _norm(t.subject_label) == _norm(source_label)\n                and _norm(t.object_label) == _norm(target_label)\n                for t in (l5.candidate_triples if l5 else [])\n            )\n            if not source_l1 or not target_l1:\n                failure = "layer01_endpoint_missing"\n            elif not exact_l1_instance:\n                failure = "layer01_relation_instance_missing"\n            elif not linked_l2:\n                failure = "layer02_wrong_or_missing_controlled_relation"\n            elif not linked_l3:\n                failure = "layer03_candidate_typing_or_resolution"\n            elif not assertion_l4:\n                failure = "layer04_endpoint_direction_or_type_validation"\n            elif not triple_l5:\n                failure = "layer05_triple_materialization"\n            else:\n                failure = "survived_to_layer05"\n            rows.append({\n                "gold_relation_id": relation_id,\n                "gold_relation_label": relation_label,\n                "source": source_label,\n                "target": target_label,\n                "source_available_layer01": source_l1,\n                "target_available_layer01": target_l1,\n                "relation_instance_layer01": exact_l1_instance,\n                "canonical_relation_layer02": linked_l2,\n                "relation_candidate_layer03": linked_l3,\n                "assertion_layer04": assertion_l4,\n                "triple_layer05": triple_l5,\n                "first_failure": failure,\n            })\n    analysis_dir = run_dir / "analysis"\n    analysis_dir.mkdir(parents=True, exist_ok=True)\n    import csv\n    path = analysis_dir / "gold_relation_trace_v4.csv"\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]) if rows else [])\n        if rows:\n            writer.writeheader()\n            writer.writerows(rows)\n    return rows\n\n\ndef analyze_run(\n    *,\n    run_dir: str | Path,\n    gold_jsonl: str | Path,\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> dict[str, Any]:\n    summary = v3.analyze_run(\n        run_dir=run_dir,\n        gold_jsonl=gold_jsonl,\n        catalog_path=catalog_path,\n        aliases_path=aliases_path,\n    )\n    trace = write_relation_trace_v4(\n        run_dir=run_dir,\n        gold_jsonl=gold_jsonl,\n        catalog_path=catalog_path,\n    )\n    counts: dict[str, int] = {}\n    for row in trace:\n        counts[row["first_failure"]] = counts.get(row["first_failure"], 0) + 1\n    summary["gold_relation_trace_v4"] = trace\n    summary["failure_counts_v4"] = counts\n    write_json(Path(run_dir) / "analysis/analysis_summary_v4.json", summary)\n    return summary\n', 'docred_native_ablation_v5.py': 'from __future__ import annotations\n\n"""NeoOLAF native DocRED one-document ablation support, v5.\n\nThis module is experiment-only: it does not modify ``src/neoolaf``.\n\nChanges relative to v4:\n\n* the single whole-document Layer 1 call performs an explicit country-coverage\n  self-check and emits separate country relation instances when supported by\n  the document, demonyms, coreference, or very high-confidence geography;\n* Layer 2 keeps the ontology RAG call but sends only a compact top-k property\n  shortlist, relation-specific rules, and at most two relevant examples;\n* transparent profile guardrails canonicalize already-extracted relation\n  instances for difficult DocRED distinctions (P127/P749/P361, P159/P276/P131,\n  P571/P577, and P17/P27). They never create a new relation instance;\n* deterministic entity projection is used only after pipeline execution for\n  benchmark evaluation and can map mention variants such as\n  ``Athens metropolitan area`` to the DocRED ``Athens`` cluster;\n* projection methods, Layer 1 country coverage, raw LLM choices, guardrail\n  overrides, compact candidates, and prompt sizes are saved for auditing.\n\nThe full native Layer 0--12 pipeline still runs. No direct DocRED extractor,\nsource-entity anchoring, gold relation hint, closure rule, or post-hoc relation\ninvention is introduced.\n"""\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom contextlib import redirect_stderr, redirect_stdout\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any, Iterable\nimport csv\nimport json\nimport re\nimport shutil\nimport sys\nimport threading\nimport time\nimport traceback\nimport unicodedata\n\nimport docred_native_ablation as v2\nimport docred_native_ablation_v3 as v3\nimport docred_native_ablation_v4 as v4\n\nfrom neoolaf.core.pipeline import Pipeline\nfrom neoolaf.core.pipeline_state import PipelineState\nfrom neoolaf.core.runner import Runner\nfrom neoolaf.domain.documents import Document\nfrom neoolaf.domain.enriched_expression import EnrichedExpression, EnrichmentEvidence\nfrom neoolaf.domain.linguistic_expression import LinguisticExpression\nfrom neoolaf.domain.user_guidance import UserGuidance\nfrom neoolaf.grounding.rag.formatting import build_grounding_context\nfrom neoolaf.grounding.rag.types import GroundingRequest\nfrom neoolaf.ontology.loader import SeedOntologyLoader\nfrom neoolaf.profiles.profile_loader import load_document_profile\n\nfrom experiments.methods.run_neoolaf import (\n    OfflineWebSearchSource,\n    OfflineWikipediaSource,\n    OfflineWikidataSource,\n    OpenAICompatibleBackend,\n    load_user_guidance,\n)\n\n# Re-export notebook helpers.\nread_json = v4.read_json\nread_jsonl = v4.read_jsonl\nwrite_json = v4.write_json\nappend_jsonl = v4.append_jsonl\nload_layer_states = v4.load_layer_states\nstate_counts = v4.state_counts\nsafe_name = v4.safe_name\nTee = v4.Tee\nLAYER_NAMES = v4.LAYER_NAMES\nSharedCallLogger = v4.SharedCallLogger\nTaggedLoggedBackend = v4.TaggedLoggedBackend\nPriorityOntologyRAGAdapter = v4.PriorityOntologyRAGAdapter\nCanonicalizingCandidateTypingResolutionLayer = v4.CanonicalizingCandidateTypingResolutionLayer\n\n\ndef _csv_cell(value: Any) -> Any:\n    """Serialize structured values predictably for CSV output."""\n    if isinstance(value, (dict, list, tuple, set)):\n        return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)\n    return value\n\n\ndef write_csv_rows(path: str | Path, rows: Iterable[dict[str, Any]]) -> None:\n    """Write heterogeneous dictionaries without dropping optional audit fields.\n\n    Projection rows are intentionally heterogeneous: ambiguous projections add\n    ``candidate_entity_ids`` while exact projections do not. Deriving the CSV\n    schema from only the first row therefore crashes on later ambiguous rows.\n    This helper builds a stable union of all keys and serializes structured cells.\n    """\n    rows = list(rows)\n    if not rows:\n        return\n    fieldnames: list[str] = []\n    seen: set[str] = set()\n    for row in rows:\n        for key in row:\n            if key not in seen:\n                seen.add(key)\n                fieldnames.append(key)\n    normalized = [\n        {key: _csv_cell(row.get(key)) for key in fieldnames}\n        for row in rows\n    ]\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(normalized)\n\n\ndef _dedup(values: Iterable[Any]) -> list[str]:\n    result: list[str] = []\n    seen: set[str] = set()\n    for value in values:\n        if value is None:\n            continue\n        text = str(value).strip()\n        if not text or text in seen:\n            continue\n        seen.add(text)\n        result.append(text)\n    return result\n\n\ndef _norm(text: Any) -> str:\n    value = unicodedata.normalize("NFKD", str(text or ""))\n    value = "".join(ch for ch in value if not unicodedata.combining(ch))\n    value = value.lower().replace("–", "-").replace("—", "-")\n    value = re.sub(r"[^a-z0-9]+", " ", value)\n    return re.sub(r"\\s+", " ", value).strip()\n\n\ndef _relation_id(text: Any) -> str | None:\n    match = re.search(r"\\bP\\d+\\b", str(text or ""), re.I)\n    return match.group(0).upper() if match else None\n\n\ndef _parse_relation_instance(text: Any) -> tuple[str, str, str] | None:\n    parts = [part.strip() for part in str(text or "").split("||")]\n    return tuple(parts) if len(parts) == 3 and all(parts) else None\n\n\ndef _json_block(value: Any, max_chars: int = 8000) -> str:\n    text = json.dumps(value, ensure_ascii=False, separators=(",", ":"))\n    return text if len(text) <= max_chars else text[:max_chars] + "..."\n\n\ndef _short(text: Any, max_chars: int = 220) -> str:\n    value = re.sub(r"\\s+", " ", str(text or "")).strip()\n    return value if len(value) <= max_chars else value[: max_chars - 3] + "..."\n\n\nclass CountryAwareRelationInstanceExtractionLayer(v4.DocREDRelationInstanceExtractionLayer):\n    """One native whole-document Layer 1 call with a country coverage checklist."""\n\n    def __init__(self, *args: Any, country_audit_path: str | Path, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self.country_audit_path = Path(country_audit_path)\n\n    def _prompt(self, state: PipelineState, chunk_text: str) -> list[dict[str, str]]:\n        task = (state.profile_config or {}).get("_input_task_guidance", {}) or {}\n        layer_guidance = (task.get("layer_guidance") or {}).get("layer01", {})\n        examples = task.get("relation_instance_examples_v5") or task.get("relation_instance_examples") or []\n        country_examples = task.get("country_propagation_examples") or []\n        priority_specs = task.get("priority_relation_specs") or []\n\n        compact_specs = [\n            {\n                "id": item.get("relation_id"),\n                "label": item.get("label"),\n                "direction": item.get("direction"),\n            }\n            for item in priority_specs\n            if item.get("relation_id") in {\n                "P17", "P27", "P127", "P131", "P159", "P361", "P463",\n                "P571", "P577", "P749",\n            }\n        ]\n\n        system = """\nYou are NeoOLAF Layer 1 for document-level DocRED linguistic-expression extraction.\n\nReturn endpoint expressions and relation-instance expressions only.\n\nEndpoint labels:\nentity_org, entity_per, entity_loc, entity_time, entity_num, entity_misc\n\nEvery relation instance MUST be:\nSOURCE || LEXICAL RELATION PHRASE || TARGET\nwith label relation_instance.\n\nMandatory relation coverage procedure:\n1. Extract every exact named/literal endpoint needed by a relation.\n2. Split every independently supported source-target pair. A date and a place\n   in the same clause must be two relation instances.\n3. Identify country anchors from explicit country names, unambiguous demonyms,\n   and phrases such as "the country".\n4. For each non-human named organization or location, emit a separate country\n   relation instance when the text, a demonym/coreference chain, or very\n   high-confidence city-country knowledge supports it. This includes the main\n   organization, its explicitly associated corporate group, and named cities.\n5. For country relation instances, use one of these lexical predicates exactly:\n   "is in country", "demonym implies country", or "the country refers to".\n   Normalize a demonym to the country noun only when unambiguous and explain it.\n   Do not use country of citizenship for people here.\n6. Before returning, self-check that each supported ORG/LOC -> country pair has\n   its own relation_instance.\n\nUse exact surface forms whenever possible. A controlled inferred endpoint may\nuse a canonical country noun. Do not select P-identifiers in Layer 1. Do not\ninvent obscure facts and do not use gold annotations.\n\nReturn JSON only:\n{"expressions":[{"text":"...","label":"entity_org","justification":"..."},\n{"text":"SOURCE || predicate || TARGET","label":"relation_instance",\n"justification":"explicit/inferred; source_type=ORG; target_type=LOC; evidence=..."}],\n"coverage_check":{"country_anchor":"...","supported_country_subjects":["..."]}}\n""".strip()\n\n        user = f"""\nLayer 1 profile:\n{_json_block(layer_guidance, 4500)}\n\nRelations to notice at this layer (do not choose their P-ID yet):\n{_json_block(compact_specs, 4500)}\n\nSynthetic relation-instance examples:\n{_json_block(examples, 6500)}\n\nSynthetic country-propagation examples:\n{_json_block(country_examples, 5000)}\n\nDocument:\n\\"\\"\\"\n{chunk_text}\n\\"\\"\\"\n\nExtract endpoint expressions and one structured relation instance per supported\npair. Perform the country-coverage self-check. Return JSON only.\n""".strip()\n        return [{"role": "system", "content": system}, {"role": "user", "content": user}]\n\n    def _run(self, state: PipelineState) -> PipelineState:\n        state = super()._run(state)\n        endpoints = []\n        relations = []\n        country_relations = []\n        country_predicate_terms = {\n            "country", "demonym", "nationality", "national", "sovereign state",\n            "is in", "belongs to",\n        }\n        for expr in state.linguistic_expressions or []:\n            parsed = _parse_relation_instance(expr.text)\n            if parsed:\n                source, predicate, target = parsed\n                row = {\n                    "expr_id": expr.expr_id,\n                    "source": source,\n                    "predicate": predicate,\n                    "target": target,\n                    "justification": expr.justification,\n                }\n                relations.append(row)\n                predicate_n = _norm(predicate)\n                if any(term in predicate_n for term in country_predicate_terms):\n                    country_relations.append(row)\n            else:\n                endpoints.append({\n                    "expr_id": expr.expr_id,\n                    "text": expr.text,\n                    "label": expr.label,\n                })\n        audit = {\n            "endpoint_count": len(endpoints),\n            "relation_instance_count": len(relations),\n            "country_relation_instance_count": len(country_relations),\n            "country_relation_instances": country_relations,\n            "note": (\n                "This is a non-gold runtime audit. It reports what the single native "\n                "Layer 1 call emitted and does not create missing relations."\n            ),\n        }\n        self.country_audit_path.parent.mkdir(parents=True, exist_ok=True)\n        write_json(self.country_audit_path, audit)\n        state.log(\n            f"[{self.name}] country coverage audit; country_relation_instances={len(country_relations)}"\n        )\n        return state\n\n\nclass CompactGuardrailedCandidateEnrichmentLayer(v4.SelectiveContrastiveCandidateEnrichmentLayer):\n    """Relation-only Layer 2 with compact ontology prompts and transparent guardrails."""\n\n    def __init__(self, *args: Any, compact_prompt_log_path: str | Path, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self.compact_prompt_log_path = Path(compact_prompt_log_path)\n        self._compact_prompt_rows: list[dict[str, Any]] = []\n        self._compact_lock = threading.Lock()\n\n    @staticmethod\n    def _endpoint_type_from_state(state: PipelineState, label: str) -> str:\n        target = _norm(label)\n        mapping = {\n            "entity_org": "ORG", "entity_per": "PER", "entity_loc": "LOC",\n            "entity_time": "TIME", "entity_num": "NUM", "entity_misc": "MISC",\n        }\n        for expr in state.linguistic_expressions or []:\n            if _parse_relation_instance(expr.text):\n                continue\n            if _norm(expr.text) == target:\n                return mapping.get(str(expr.label).lower(), "UNKNOWN")\n        return "UNKNOWN"\n\n    @staticmethod\n    def _lexical_candidate_ids(\n        predicate: str,\n        source: str,\n        target: str,\n        source_type: str,\n        target_type: str,\n    ) -> list[str]:\n        p = _norm(predicate)\n        context = _norm(f"{source} {predicate} {target}")\n        ids: list[str] = []\n\n        if any(term in p for term in ["headquarter", "based in", "base in", "based at"]):\n            ids += ["P159", "P131", "P276"]\n        if any(term in p for term in ["relaunch", "launch", "open", "establish", "found"]):\n            if target_type == "TIME":\n                ids += ["P571", "P580", "P577"]\n            elif source_type == "ORG" and target_type == "LOC":\n                ids += ["P159", "P276", "P131"]\n        if "part of" in p or "belong" in p or "component of" in p:\n            if source_type == "ORG" and target_type == "ORG":\n                ids += ["P127", "P749", "P361"]\n            else:\n                ids += ["P361", "P527"]\n        if any(term in context for term in ["subsidiary", "branch", "division", "laboratory", "lab ", "unit of"]):\n            ids += ["P749", "P127", "P361"]\n        if "member of" in p or "membership" in p:\n            ids += ["P463", "P361"]\n        if any(term in p for term in ["country", "demonym", "national", "is in"]):\n            ids += ["P17", "P27", "P131"]\n        if "available on" in p or "broadcast on" in p or "platform" in p:\n            ids += ["P400", "P449"]\n        if any(term in p for term in ["published", "publication", "released"]):\n            ids += ["P577", "P571"]\n        if "located in" in p:\n            ids += ["P131", "P276", "P159"]\n        return _dedup(ids)\n\n    @staticmethod\n    def _relevant_rows(\n        rows: Iterable[dict[str, Any]],\n        *,\n        candidate_ids: set[str],\n        predicate: str,\n        limit: int,\n    ) -> list[dict[str, Any]]:\n        predicate_tokens = set(_norm(predicate).split())\n        scored: list[tuple[int, dict[str, Any]]] = []\n        for row in rows or []:\n            row_ids = {\n                value.upper()\n                for value in re.findall(r"\\bP\\d+\\b", json.dumps(row, ensure_ascii=False), re.I)\n            }\n            text_tokens = set(_norm(json.dumps(row, ensure_ascii=False)).split())\n            score = 4 * len(row_ids & candidate_ids) + len(predicate_tokens & text_tokens)\n            if score > 0:\n                scored.append((score, row))\n        scored.sort(key=lambda item: item[0], reverse=True)\n        return [row for _, row in scored[:limit]]\n\n    def _compact_catalog_rows(\n        self,\n        candidate_ids: list[str],\n        type_constraints: dict[str, Any],\n    ) -> list[dict[str, Any]]:\n        rows: list[dict[str, Any]] = []\n        for relation_id in candidate_ids:\n            item = self.catalog.get(relation_id)\n            if not item:\n                continue\n            constraint = type_constraints.get(relation_id, {})\n            rows.append({\n                "id": relation_id,\n                "label": item.get("label"),\n                "definition": _short(item.get("comment"), 210),\n                "source_types": constraint.get("source") or [],\n                "target_types": constraint.get("target") or [],\n            })\n        return rows\n\n    def _candidate_ids(\n        self,\n        *,\n        state: PipelineState,\n        source: str,\n        predicate: str,\n        target: str,\n        source_type: str,\n        target_type: str,\n        grounding_text: str,\n    ) -> list[str]:\n        task = (state.profile_config or {}).get("_input_task_guidance", {}) or {}\n        allowed = set(task.get("allowed_relation_ids") or self.catalog)\n        profile_sets = task.get("predicate_candidate_sets") or []\n        profile_ids: list[str] = []\n        predicate_n = _norm(predicate)\n        for row in profile_sets:\n            triggers = [_norm(x) for x in row.get("triggers", [])]\n            if any(trigger and trigger in predicate_n for trigger in triggers):\n                profile_ids.extend(row.get("candidate_relation_ids") or [])\n        lexical = self._lexical_candidate_ids(predicate, source, target, source_type, target_type)\n        retrieved = self._candidate_ids_from_grounding(grounding_text)\n        cap = int((state.profile_config or {}).get("layer02_compact_candidate_cap", 5))\n        ordered = _dedup([*profile_ids, *lexical, *retrieved])\n        result = [rid for rid in ordered if rid in allowed and rid in self.catalog][:cap]\n        if not result:\n            result = [rid for rid in task.get("priority_relation_ids", []) if rid in self.catalog][:cap]\n        return result\n\n    @staticmethod\n    def _guardrail_relation_id(\n        *,\n        selected_id: str | None,\n        predicate: str,\n        source: str,\n        target: str,\n        source_type: str,\n        target_type: str,\n        candidate_ids: Iterable[str],\n    ) -> tuple[str | None, str | None]:\n        p = _norm(predicate)\n        context = _norm(f"{source} {predicate} {target}")\n        candidates = set(candidate_ids)\n\n        def choose(relation_id: str, reason: str) -> tuple[str | None, str | None]:\n            if relation_id in candidates:\n                return relation_id, reason\n            return selected_id, None\n\n        if source_type == "ORG" and target_type == "ORG":\n            if any(term in context for term in ["subsidiary", "branch", "division", "laboratory", "lab ", "unit of"]):\n                return choose("P749", "Explicit branch/subsidiary/unit wording requires child-to-parent P749.")\n            if "part of" in p and any(term in context for term in ["group", "company", "corporate", "media"]):\n                return choose("P127", "Corporate/media-group part-of wording is canonicalized to P127 owned by; P749 is reserved for explicit branch/subsidiary units.")\n\n        if source_type == "ORG" and target_type == "LOC":\n            if any(term in p for term in ["based in", "headquarter", "base in", "based at"]):\n                return choose("P159", "Organization base/headquarters city requires P159.")\n            if any(term in p for term in ["relaunch", "launch", "opened", "established in"]):\n                return choose("P159", "DocRED organization launch/relaunch location convention uses P159 rather than generic P276/P131.")\n\n        if source_type in {"ORG", "LOC", "MISC"} and target_type == "LOC":\n            if any(term in p for term in ["country", "demonym", "national"]):\n                return choose("P17", "Non-human entity/place to sovereign country requires P17, not P27.")\n\n        if target_type == "TIME" and source_type in {"ORG", "LOC", "MISC"}:\n            if any(term in p for term in ["relaunch", "launch", "establish", "found", "inception"]):\n                return choose("P571", "Organization/entity establishment or relaunch date requires P571.")\n\n        return selected_id, None\n\n    def _contrastive_prompt_compact(\n        self,\n        *,\n        expr: LinguisticExpression,\n        state: PipelineState,\n        source: str,\n        predicate: str,\n        target: str,\n        source_type: str,\n        target_type: str,\n        candidate_ids: list[str],\n    ) -> list[dict[str, str]]:\n        task = (state.profile_config or {}).get("_input_task_guidance", {}) or {}\n        constraints = (state.profile_config or {}).get("docred_type_constraints", {}) or {}\n        candidate_set = set(candidate_ids)\n        rules = self._relevant_rows(\n            task.get("contrastive_rules") or [],\n            candidate_ids=candidate_set,\n            predicate=predicate,\n            limit=3,\n        )\n        examples = self._relevant_rows(\n            task.get("relation_examples") or [],\n            candidate_ids=candidate_set,\n            predicate=predicate,\n            limit=2,\n        )\n        candidates = self._compact_catalog_rows(candidate_ids, constraints)\n\n        system = """\nYou are NeoOLAF Layer 2. Classify one already-extracted relation instance into\nexactly one supplied DocRED ontology property, or found=false.\n\nUse endpoint types, direction, lexical meaning, specificity, and the compact\nontology definitions. Prefer the specific benchmark property over a generic\nproperty. Do not create endpoints or relation instances. Do not use gold data.\n\nReturn JSON only:\n{"found":true,"selected_relation_id":"P159","decision":"brief contrastive reason"}\nor {"found":false,"decision":"brief reason"}.\n""".strip()\n        user = f"""\nINSTANCE: {source} || {predicate} || {target}\nTYPES: {source_type} -> {target_type}\nEVIDENCE: {_short(expr.justification, 500)}\nCANDIDATES: {_json_block(candidates, 3500)}\nRELEVANT_RULES: {_json_block(rules, 1800)}\nRELEVANT_EXAMPLES: {_json_block(examples, 2200)}\nChoose one candidate ID or found=false. JSON only.\n""".strip()\n        return [{"role": "system", "content": system}, {"role": "user", "content": user}]\n\n    def _process_relation_contrastive(self, expr: LinguisticExpression, state: PipelineState) -> EnrichedExpression:\n        parsed_instance = _parse_relation_instance(expr.text)\n        if parsed_instance is None:\n            raise ValueError(f"Malformed relation instance: {expr.text}")\n        source, predicate, target = parsed_instance\n        source_type = self._type_from_justification(expr.justification, "source_type")\n        target_type = self._type_from_justification(expr.justification, "target_type")\n        if source_type == "UNKNOWN":\n            source_type = self._endpoint_type_from_state(state, source)\n        if target_type == "UNKNOWN":\n            target_type = self._endpoint_type_from_state(state, target)\n\n        grounding_text = ""\n        if self.rag_adapter is not None:\n            grounding = self.rag_adapter.ground(GroundingRequest(\n                layer_name=self.name,\n                query=f"{predicate} {source_type} {target_type}",\n                payload={\n                    "relation_instance": expr.text,\n                    "source": source,\n                    "predicate": predicate,\n                    "target": target,\n                    "source_type": source_type,\n                    "target_type": target_type,\n                    "justification": expr.justification,\n                },\n                preferred_sources=["ontology"],\n                top_k=6,\n            ))\n            grounding_text = build_grounding_context(grounding)\n\n        candidate_ids = self._candidate_ids(\n            state=state,\n            source=source,\n            predicate=predicate,\n            target=target,\n            source_type=source_type,\n            target_type=target_type,\n            grounding_text=grounding_text,\n        )\n        messages = self._contrastive_prompt_compact(\n            expr=expr,\n            state=state,\n            source=source,\n            predicate=predicate,\n            target=target,\n            source_type=source_type,\n            target_type=target_type,\n            candidate_ids=candidate_ids,\n        )\n        raw = self.ollama_backend.chat(\n            model=state.llm_model,\n            messages=messages,\n            temperature=0.0,\n        )\n        parsed = self.ollama_backend.extract_json(raw)\n        if not isinstance(parsed, dict):\n            raise ValueError("Layer 2 compact response is not a JSON object")\n\n        found = bool(parsed.get("found", False))\n        raw_selected_id = _relation_id(parsed.get("selected_relation_id"))\n        if found and raw_selected_id not in candidate_ids:\n            raise ValueError(\n                f"Layer 2 selected {raw_selected_id}, outside compact candidates {candidate_ids}"\n            )\n        selected_id, override_reason = self._guardrail_relation_id(\n            selected_id=raw_selected_id if found else None,\n            predicate=predicate,\n            source=source,\n            target=target,\n            source_type=source_type,\n            target_type=target_type,\n            candidate_ids=candidate_ids,\n        )\n        if selected_id is not None:\n            found = True\n        if found and selected_id not in self.catalog:\n            raise ValueError(f"Layer 2 selected invalid relation ID: {selected_id}")\n\n        if found:\n            item = self.catalog[selected_id]\n            canonical = f"{selected_id} : {item[\'label\']}"\n            decision_text = str(parsed.get("decision") or "").strip()\n            if override_reason:\n                decision_text = f"{decision_text} PROFILE_GUARDRAIL: {override_reason}".strip()\n            hints = _dedup([\n                f"controlled_relation:{canonical}",\n                "promote_to_ontology:true",\n                item.get("uri"),\n                item.get("label"),\n                f"source_label:{source}",\n                f"target_label:{target}",\n                f"lexical_predicate:{predicate}",\n                f"source_type:{source_type}",\n                f"target_type:{target_type}",\n                f"contrastive_decision:{decision_text}",\n            ])\n            definition = str(item.get("comment") or item.get("label") or "").strip()\n        else:\n            canonical = None\n            decision_text = str(parsed.get("decision") or "No supported ontology relation selected.")\n            hints = _dedup([\n                "promote_to_ontology:false",\n                f"source_label:{source}",\n                f"target_label:{target}",\n                f"lexical_predicate:{predicate}",\n                f"source_type:{source_type}",\n                f"target_type:{target_type}",\n                f"contrastive_decision:{decision_text}",\n            ])\n            definition = decision_text\n\n        decision_row = {\n            "expr_id": expr.expr_id,\n            "relation_instance": expr.text,\n            "source": source,\n            "predicate": predicate,\n            "target": target,\n            "source_type": source_type,\n            "target_type": target_type,\n            "candidate_relation_ids": candidate_ids,\n            "found": found,\n            "raw_selected_relation_id": raw_selected_id,\n            "selected_relation_id": selected_id,\n            "canonical_relation": canonical,\n            "guardrail_override": bool(override_reason and selected_id != raw_selected_id),\n            "guardrail_reason": override_reason,\n            "decision": decision_text,\n            "ontology_hints": hints,\n            "system_chars": len(messages[0]["content"]),\n            "user_chars": len(messages[1]["content"]),\n        }\n        self._record_decision(decision_row)\n        with self._compact_lock:\n            self._compact_prompt_rows.append({\n                key: decision_row[key]\n                for key in [\n                    "expr_id", "relation_instance", "candidate_relation_ids",\n                    "raw_selected_relation_id", "selected_relation_id",\n                    "guardrail_override", "system_chars", "user_chars",\n                ]\n            })\n\n        aliases = _dedup([expr.text, predicate])\n        return EnrichedExpression(\n            base_expression=expr,\n            aliases=aliases,\n            synonyms=[],\n            lexical_variants=[],\n            alias_sources={value: ["source"] for value in aliases},\n            synonym_sources={},\n            lexical_variant_sources={},\n            definition=definition,\n            ontology_hints=hints,\n            enrichment_evidence=[EnrichmentEvidence(\n                source="llm",\n                content=json.dumps(parsed, ensure_ascii=False),\n                reference=state.llm_model,\n            )],\n        )\n\n    def _run(self, state: PipelineState) -> PipelineState:\n        self._compact_prompt_rows = []\n        state = super()._run(state)\n        self.compact_prompt_log_path.parent.mkdir(parents=True, exist_ok=True)\n        write_json(\n            self.compact_prompt_log_path,\n            sorted(self._compact_prompt_rows, key=lambda row: row.get("expr_id", "")),\n        )\n        return state\n\n\n# ---------------------------------------------------------------------------\n# Post-run mention-aware benchmark projection (evaluation only)\n# ---------------------------------------------------------------------------\n\n_ADMIN_SUFFIXES = [\n    ("metropolitan", "area"), ("metro", "area"), ("urban", "area"),\n    ("city",), ("county",), ("province",), ("region",), ("district",),\n    ("municipality",),\n]\n_STOPWORDS = {"the", "of"}\n\n\ndef _tokens(text: Any) -> tuple[str, ...]:\n    return tuple(token for token in _norm(text).split() if token not in _STOPWORDS)\n\n\ndef _strip_admin_suffix(tokens: tuple[str, ...]) -> tuple[str, ...]:\n    result = tokens\n    changed = True\n    while changed and result:\n        changed = False\n        for suffix in _ADMIN_SUFFIXES:\n            if len(result) > len(suffix) and result[-len(suffix):] == suffix:\n                result = result[:-len(suffix)]\n                changed = True\n                break\n    return result\n\n\ndef _candidate_values(candidate: Any) -> list[str]:\n    values = [getattr(candidate, "canonical_label", "")]\n    values.extend(getattr(candidate, "aliases", []) or [])\n    values.extend(getattr(candidate, "synonyms", []) or [])\n    values.extend(getattr(candidate, "lexical_variants", []) or [])\n    values.extend(getattr(mention, "text", "") for mention in getattr(candidate, "mentions", []) or [])\n    return _dedup(values)\n\n\ndef _gold_alias_rows(gold: dict[str, Any]) -> dict[str, list[str]]:\n    return {\n        entity_id: _dedup(\n            mention.get("trigger_word")\n            for mention in payload.get("mentions", [])\n            if mention.get("trigger_word")\n        )\n        for entity_id, payload in gold.get("entities", {}).items()\n    }\n\n\ndef project_values_to_gold(values: Iterable[str], gold: dict[str, Any]) -> dict[str, Any]:\n    candidate_values = _dedup(values)\n    gold_aliases = _gold_alias_rows(gold)\n    matches: list[dict[str, Any]] = []\n\n    for candidate_value in candidate_values:\n        c_norm = _norm(candidate_value)\n        c_tokens = _tokens(candidate_value)\n        c_stripped = _strip_admin_suffix(c_tokens)\n        if not c_norm:\n            continue\n        for entity_id, aliases in gold_aliases.items():\n            for alias in aliases:\n                g_norm = _norm(alias)\n                g_tokens = _tokens(alias)\n                g_stripped = _strip_admin_suffix(g_tokens)\n                method = None\n                score = 0.0\n                if c_norm == g_norm:\n                    method, score = "exact", 1.0\n                elif c_stripped and c_stripped == g_stripped:\n                    method, score = "administrative_suffix_normalization", 0.96\n                elif len(g_tokens) >= 1 and len(c_tokens) > len(g_tokens):\n                    # Require a token-boundary prefix/suffix match, not arbitrary substring.\n                    if c_tokens[: len(g_tokens)] == g_tokens or c_tokens[-len(g_tokens):] == g_tokens:\n                        method, score = "token_boundary_containment", 0.88\n                elif len(c_tokens) >= 2 and len(g_tokens) > len(c_tokens):\n                    if g_tokens[: len(c_tokens)] == c_tokens or g_tokens[-len(c_tokens):] == c_tokens:\n                        method, score = "reverse_token_boundary_containment", 0.84\n                if method:\n                    matches.append({\n                        "entity_id": entity_id,\n                        "candidate_value": candidate_value,\n                        "gold_alias": alias,\n                        "method": method,\n                        "score": score,\n                    })\n\n    if not matches:\n        return {\n            "entity_id": None,\n            "method": "unmapped",\n            "score": 0.0,\n            "candidate_value": candidate_values[0] if candidate_values else None,\n            "gold_alias": None,\n            "ambiguous": False,\n        }\n    best_score = max(row["score"] for row in matches)\n    best = [row for row in matches if row["score"] == best_score]\n    best_ids = {row["entity_id"] for row in best}\n    if len(best_ids) != 1:\n        return {\n            "entity_id": None,\n            "method": "ambiguous",\n            "score": best_score,\n            "candidate_value": best[0]["candidate_value"],\n            "gold_alias": None,\n            "ambiguous": True,\n            "candidate_entity_ids": sorted(best_ids),\n        }\n    selected = next(row for row in best if row["entity_id"] in best_ids)\n    return {**selected, "ambiguous": False}\n\n\ndef project_candidate_to_gold(candidate: Any, gold: dict[str, Any]) -> dict[str, Any]:\n    if candidate is None:\n        return {"entity_id": None, "method": "missing_candidate", "score": 0.0, "ambiguous": False}\n    return project_values_to_gold(_candidate_values(candidate), gold)\n\n\ndef native_predictions_v5(\n    state: PipelineState,\n    gold: dict[str, Any],\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> list[dict[str, Any]]:\n    _, label_to_id = v2.relation_lookup(catalog_path, aliases_path)\n    candidate_by_id = {\n        candidate.candidate_id: candidate\n        for candidate in [\n            *(state.entity_candidates or []), *(state.event_candidates or []),\n            *(state.attribute_candidates or []), *(state.relation_candidates or []),\n        ]\n    }\n    relation_by_id = {candidate.candidate_id: candidate for candidate in state.relation_candidates or []}\n    rows: list[dict[str, Any]] = []\n    for triple in state.candidate_triples or []:\n        subject = candidate_by_id.get(triple.subject_id)\n        obj = candidate_by_id.get(triple.object_id)\n        relation_candidate = relation_by_id.get(triple.predicate_id)\n        relation_id = v2.map_predicate(\n            triple.predicate_label,\n            getattr(relation_candidate, "ontology_hints", []) if relation_candidate else [],\n            label_to_id,\n        )\n        head_projection = project_candidate_to_gold(subject, gold)\n        tail_projection = project_candidate_to_gold(obj, gold)\n        rows.append({\n            "triple_id": triple.triple_id,\n            "subject_label": triple.subject_label,\n            "predicate_label": triple.predicate_label,\n            "object_label": triple.object_label,\n            "head_id": head_projection.get("entity_id"),\n            "relation_id": relation_id,\n            "tail_id": tail_projection.get("entity_id"),\n            "fully_mapped": bool(head_projection.get("entity_id") and relation_id and tail_projection.get("entity_id")),\n            "head_projection_method": head_projection.get("method"),\n            "head_projection_score": head_projection.get("score"),\n            "head_matched_value": head_projection.get("candidate_value"),\n            "tail_projection_method": tail_projection.get("method"),\n            "tail_projection_score": tail_projection.get("score"),\n            "tail_matched_value": tail_projection.get("candidate_value"),\n            "confidence": triple.confidence,\n            "justification": triple.justification,\n        })\n    return rows\n\n\ndef assertion_predictions_v5(\n    state: PipelineState,\n    gold: dict[str, Any],\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> list[dict[str, Any]]:\n    _, label_to_id = v2.relation_lookup(catalog_path, aliases_path)\n    candidates = {\n        c.candidate_id: c\n        for c in [\n            *(state.entity_candidates or []), *(state.event_candidates or []),\n            *(state.attribute_candidates or []),\n        ]\n    }\n    relations = {c.candidate_id: c for c in state.relation_candidates or []}\n    rows: list[dict[str, Any]] = []\n    for assertion in state.candidate_relation_assertions or []:\n        src = candidates.get(assertion.source_candidate_id)\n        dst = candidates.get(assertion.target_candidate_id)\n        rel = relations.get(assertion.relation_candidate_id)\n        relation_id = v2.map_predicate(\n            assertion.relation_label,\n            getattr(rel, "ontology_hints", []) if rel else [],\n            label_to_id,\n        )\n        head = project_candidate_to_gold(src, gold)\n        tail = project_candidate_to_gold(dst, gold)\n        rows.append({\n            "assertion_id": assertion.assertion_id,\n            "head_id": head.get("entity_id"),\n            "relation_id": relation_id,\n            "tail_id": tail.get("entity_id"),\n            "fully_mapped": bool(head.get("entity_id") and relation_id and tail.get("entity_id")),\n            "source_label": assertion.source_candidate_label,\n            "predicate_label": assertion.relation_label,\n            "target_label": assertion.target_candidate_label,\n            "head_projection_method": head.get("method"),\n            "tail_projection_method": tail.get("method"),\n        })\n    return rows\n\n\ndef write_entity_projection_audit(\n    run_dir: str | Path,\n    state: PipelineState,\n    gold: dict[str, Any],\n) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    for candidate in [\n        *(state.entity_candidates or []), *(state.event_candidates or []),\n        *(state.attribute_candidates or []),\n    ]:\n        projection = project_candidate_to_gold(candidate, gold)\n        rows.append({\n            "candidate_id": candidate.candidate_id,\n            "canonical_label": candidate.canonical_label,\n            "candidate_values": " | ".join(_candidate_values(candidate)),\n            **projection,\n        })\n    analysis_dir = Path(run_dir) / "analysis"\n    analysis_dir.mkdir(parents=True, exist_ok=True)\n    write_json(analysis_dir / "entity_projection_audit_v5.json", rows)\n    write_csv_rows(analysis_dir / "entity_projection_audit_v5.csv", rows)\n    return rows\n\n\ndef write_cumulative_evaluation_v5(\n    run_dir: str | Path,\n    gold: dict[str, Any],\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    for index, name, state in load_layer_states(run_dir):\n        if index < 4:\n            predictions = []\n        elif index == 4:\n            predictions = assertion_predictions_v5(state, gold, catalog_path, aliases_path)\n        else:\n            predictions = native_predictions_v5(state, gold, catalog_path, aliases_path)\n        evaluation = v2.strict_evaluate(predictions, gold)\n        rows.append({\n            "layer_index": index,\n            "layer_name": name,\n            "mapped_predictions": sum(1 for row in predictions if row.get("fully_mapped")),\n            **{key: evaluation[key] for key in [\n                "predicted", "gold", "true_positive", "false_positive",\n                "false_negative", "precision", "recall", "f1",\n            ]},\n        })\n    analysis_dir = Path(run_dir) / "analysis"\n    write_json(analysis_dir / "cumulative_strict_evaluation_v5.json", rows)\n    write_csv_rows(analysis_dir / "cumulative_strict_evaluation_v5.csv", rows)\n    return rows\n\n\ndef write_relation_trace_v5(\n    *,\n    run_dir: str | Path,\n    gold_jsonl: str | Path,\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> list[dict[str, Any]]:\n    run_dir = Path(run_dir)\n    gold = read_jsonl(gold_jsonl)[0]\n    states = {index: state for index, _, state in load_layer_states(run_dir)}\n    l1, l2, l3, l4, l5 = (states.get(i) for i in [1, 2, 3, 4, 5])\n    decisions_path = run_dir / "run_logs/layer02_contrastive_decisions.json"\n    decisions = read_json(decisions_path) if decisions_path.is_file() else []\n\n    l1_endpoint_ids: set[str] = set()\n    l1_relation_keys: set[tuple[str, str]] = set()\n    for expr in (l1.linguistic_expressions if l1 else []):\n        parsed = _parse_relation_instance(expr.text)\n        if parsed:\n            source, _, target = parsed\n            src = project_values_to_gold([source], gold).get("entity_id")\n            dst = project_values_to_gold([target], gold).get("entity_id")\n            if src and dst:\n                l1_relation_keys.add((src, dst))\n        else:\n            projected = project_values_to_gold([expr.text], gold).get("entity_id")\n            if projected:\n                l1_endpoint_ids.add(projected)\n\n    l2_keys: set[tuple[str, str, str]] = set()\n    for row in decisions:\n        rid = row.get("selected_relation_id")\n        src = project_values_to_gold([row.get("source")], gold).get("entity_id")\n        dst = project_values_to_gold([row.get("target")], gold).get("entity_id")\n        if src and rid and dst:\n            l2_keys.add((src, rid, dst))\n\n    l3_keys: set[tuple[str, str, str]] = set()\n    for candidate in (l3.relation_candidates if l3 else []):\n        rid = _relation_id(candidate.canonical_label) or _relation_id(" ".join(candidate.ontology_hints or []))\n        for value in [*(candidate.aliases or []), *(m.text for m in candidate.mentions or [])]:\n            parsed = _parse_relation_instance(value)\n            if not parsed:\n                continue\n            src = project_values_to_gold([parsed[0]], gold).get("entity_id")\n            dst = project_values_to_gold([parsed[2]], gold).get("entity_id")\n            if src and rid and dst:\n                l3_keys.add((src, rid, dst))\n\n    l4_rows = assertion_predictions_v5(l4, gold, catalog_path, aliases_path) if l4 else []\n    l4_keys = {(r["head_id"], r["relation_id"], r["tail_id"]) for r in l4_rows if r.get("fully_mapped")}\n    l5_rows = native_predictions_v5(l5, gold, catalog_path, aliases_path) if l5 else []\n    l5_keys = {(r["head_id"], r["relation_id"], r["tail_id"]) for r in l5_rows if r.get("fully_mapped")}\n\n    def entity_label(entity_id: str) -> str:\n        mentions = gold["entities"][entity_id].get("mentions") or []\n        return mentions[0].get("trigger_word") if mentions else entity_id\n\n    rows: list[dict[str, Any]] = []\n    for head, relation_id, tail in sorted(v2.gold_triples(gold)):\n        source_available = head in l1_endpoint_ids\n        target_available = tail in l1_endpoint_ids\n        relation_instance = (head, tail) in l1_relation_keys\n        linked_l2 = (head, relation_id, tail) in l2_keys\n        linked_l3 = (head, relation_id, tail) in l3_keys\n        assertion_l4 = (head, relation_id, tail) in l4_keys\n        triple_l5 = (head, relation_id, tail) in l5_keys\n        if not source_available or not target_available:\n            failure = "layer01_endpoint_missing_after_projection"\n        elif not relation_instance:\n            failure = "layer01_relation_instance_missing"\n        elif not linked_l2:\n            failure = "layer02_wrong_or_missing_controlled_relation"\n        elif not linked_l3:\n            failure = "layer03_candidate_typing_or_resolution"\n        elif not assertion_l4:\n            failure = "layer04_endpoint_direction_or_type_validation"\n        elif not triple_l5:\n            failure = "layer05_triple_materialization_or_projection"\n        else:\n            failure = "survived_to_layer05"\n        rows.append({\n            "head_id": head,\n            "source": entity_label(head),\n            "relation_id": relation_id,\n            "tail_id": tail,\n            "target": entity_label(tail),\n            "source_available_layer01": source_available,\n            "target_available_layer01": target_available,\n            "relation_instance_layer01": relation_instance,\n            "canonical_relation_layer02": linked_l2,\n            "relation_candidate_layer03": linked_l3,\n            "assertion_layer04": assertion_l4,\n            "triple_layer05": triple_l5,\n            "first_failure": failure,\n        })\n    analysis_dir = run_dir / "analysis"\n    write_json(analysis_dir / "gold_relation_trace_v5.json", rows)\n    write_csv_rows(analysis_dir / "gold_relation_trace_v5.csv", rows)\n    return rows\n\n\ndef write_native_views_v5(\n    run_dir: str | Path,\n    state: PipelineState,\n    gold: dict[str, Any],\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> dict[str, list[dict[str, Any]]]:\n    analysis_dir = Path(run_dir) / "analysis"\n    mapped = native_predictions_v5(state, gold, catalog_path, aliases_path)\n    relations = {c.candidate_id: c for c in state.relation_candidates or []}\n    lexical: list[dict[str, Any]] = []\n    canonical: list[dict[str, Any]] = []\n    for triple, mapped_row in zip(state.candidate_triples or [], mapped):\n        rel = relations.get(triple.predicate_id)\n        row = {\n            "triple_id": triple.triple_id,\n            "subject_label": triple.subject_label,\n            "predicate_label": triple.predicate_label,\n            "object_label": triple.object_label,\n            "relation_candidate_id": triple.predicate_id,\n            "relation_aliases": list(getattr(rel, "aliases", []) or []) if rel else [],\n            "ontology_hints": list(getattr(rel, "ontology_hints", []) or []) if rel else [],\n            "confidence": triple.confidence,\n            "justification": triple.justification,\n        }\n        lexical.append(row)\n        if mapped_row.get("relation_id"):\n            canonical.append({**row, **mapped_row})\n    gold_set = v2.gold_triples(gold)\n    not_in_gold = [\n        {**row, "manual_review_required": True}\n        for row in mapped\n        if row.get("fully_mapped") and (row["head_id"], row["relation_id"], row["tail_id"]) not in gold_set\n    ]\n    files = {\n        "native_lexical_triples_v5": lexical,\n        "ontology_canonical_triples_v5": canonical,\n        "strict_docred_predictions_v5": mapped,\n        "predictions_not_in_gold_manual_review_v5": not_in_gold,\n    }\n    for name, rows in files.items():\n        write_json(analysis_dir / f"{name}.json", rows)\n        write_csv_rows(analysis_dir / f"{name}.csv", rows)\n    return files\n\n\n# ---------------------------------------------------------------------------\n# Pipeline construction and execution\n# ---------------------------------------------------------------------------\n\n\ndef _layer_cfg(profile: dict[str, Any], layer_name: str) -> dict[str, Any]:\n    return dict((profile.get("layers") or {}).get(layer_name) or {})\n\n\ndef _make_backend(\n    *,\n    logger: SharedCallLogger,\n    layer_tag: str,\n    model_host: str,\n    api_key: str,\n    cfg: dict[str, Any],\n    fallback_max_tokens: int,\n    fallback_timeout: int,\n    reasoning_effort: str,\n) -> TaggedLoggedBackend:\n    core = OpenAICompatibleBackend(\n        backend_name="openrouter",\n        host=model_host,\n        api_key=api_key,\n        timeout=int(cfg.get("request_timeout_seconds", fallback_timeout)),\n        max_tokens=int(cfg.get("max_output_tokens", fallback_max_tokens)),\n        reasoning_effort=reasoning_effort,\n        exclude_reasoning=True,\n    )\n    return TaggedLoggedBackend(\n        core,\n        logger,\n        layer_tag=layer_tag,\n        response_hard_cap_chars=cfg.get("response_hard_cap_chars"),\n    )\n\n\ndef choose_chunk_size(text: str, max_safe_chars: int = 24000) -> int:\n    return v4.choose_chunk_size(text, max_safe_chars)\n\n\ndef build_document(record: dict[str, Any], source_path: str | Path) -> Document:\n    return v4.build_document(record, source_path)\n\n\ndef build_pipeline(\n    *,\n    backends: dict[str, TaggedLoggedBackend],\n    rag_adapter: PriorityOntologyRAGAdapter,\n    profile_config: dict[str, Any],\n    relation_catalog_path: str | Path,\n    chunk_size: int,\n    run_dir: str | Path,\n    workers: int = 16,\n    verbose: bool = True,\n) -> Pipeline:\n    pipeline = v4.build_pipeline(\n        backends=backends,\n        rag_adapter=rag_adapter,\n        profile_config=profile_config,\n        relation_catalog_path=relation_catalog_path,\n        chunk_size=chunk_size,\n        run_dir=run_dir,\n        workers=workers,\n        verbose=verbose,\n    )\n    retry_default = int((profile_config.get("orchestration") or {}).get("retry_failed_calls", 1))\n    sleep_default = float((profile_config.get("orchestration") or {}).get("retry_sleep_seconds", 1.0))\n    l2_cfg = _layer_cfg(profile_config, "layer02_candidate_enrichment")\n    run_dir = Path(run_dir)\n\n    pipeline.layers[1] = CountryAwareRelationInstanceExtractionLayer(\n        backends["layer01"],\n        decision_log_path=run_dir / "run_logs/layer01_relation_instances.json",\n        country_audit_path=run_dir / "run_logs/layer01_country_coverage_audit.json",\n        max_chunks=1,\n        temperature=0.0,\n        save_intermediate=True,\n        verbose=verbose,\n        rag_backend=rag_adapter,\n        max_concurrency=1,\n        retry_failed_calls=0,\n        retry_sleep_seconds=sleep_default,\n        rag_enabled=False,\n    )\n    pipeline.layers[2] = CompactGuardrailedCandidateEnrichmentLayer(\n        backends["layer02"],\n        wikipedia_source=OfflineWikipediaSource(),\n        wikidata_source=OfflineWikidataSource(),\n        web_search_source=OfflineWebSearchSource(),\n        relation_catalog_path=relation_catalog_path,\n        decision_log_path=run_dir / "run_logs/layer02_contrastive_decisions.json",\n        compact_prompt_log_path=run_dir / "run_logs/layer02_compact_prompt_audit.json",\n        max_expressions=None,\n        use_web_search=False,\n        save_intermediate=True,\n        verbose=verbose,\n        rag_adapter=rag_adapter,\n        max_concurrency=int(l2_cfg.get("max_concurrency", workers)),\n        retry_failed_calls=int(l2_cfg.get("retry_failed_calls", retry_default)),\n        retry_sleep_seconds=sleep_default,\n    )\n    return pipeline\n\n\ndef run_native_pipeline(\n    *,\n    project_root: str | Path,\n    input_jsonl: str | Path,\n    ontology_path: str | Path,\n    profile_path: str | Path,\n    guidance_path: str | Path,\n    relation_catalog_path: str | Path,\n    relation_aliases_path: str | Path,\n    run_dir: str | Path,\n    model_name: str,\n    api_key: str,\n    host: str = "https://openrouter.ai/api/v1",\n    workers: int = 16,\n    max_tokens: int = 4096,\n    request_timeout: int = 120,\n    reasoning_effort: str = "minimal",\n    verbose: bool = True,\n    clean_run_dir: bool = True,\n) -> PipelineState:\n    project_root = Path(project_root).resolve()\n    input_jsonl = Path(input_jsonl).resolve()\n    ontology_path = Path(ontology_path).resolve()\n    profile_path = Path(profile_path).resolve()\n    guidance_path = Path(guidance_path).resolve()\n    relation_catalog_path = Path(relation_catalog_path).resolve()\n    relation_aliases_path = Path(relation_aliases_path).resolve()\n    run_dir = Path(run_dir).resolve()\n    if clean_run_dir and run_dir.exists():\n        shutil.rmtree(run_dir)\n    run_dir.mkdir(parents=True, exist_ok=True)\n    logs_dir = run_dir / "run_logs"\n    logs_dir.mkdir(parents=True, exist_ok=True)\n\n    records = read_jsonl(input_jsonl)\n    if len(records) != 1:\n        raise ValueError(f"This notebook expects exactly one input document, found {len(records)}")\n    record = records[0]\n    if "entities" in record or "relations" in record:\n        raise ValueError("Pipeline input must not contain gold entities or relations.")\n    if not api_key:\n        raise ValueError("OPENROUTER_API_KEY is not set.")\n\n    profile = load_document_profile(profile_path=profile_path)\n    profile_dict = profile.to_state_dict()\n    profile_dict["_input_task_guidance"] = record.get("task_guidance") or {}\n    guidance = load_user_guidance(str(guidance_path)) or UserGuidance()\n    guidance = v3.merge_input_task_guidance(guidance, record)\n    write_json(run_dir / "input_task_guidance.json", record.get("task_guidance") or {})\n    write_json(run_dir / "effective_user_guidance.json", asdict(guidance))\n\n    seed_ontology = SeedOntologyLoader().load(str(ontology_path))\n    if len(seed_ontology.properties_by_uri) < 90:\n        raise RuntimeError(f"Only {len(seed_ontology.properties_by_uri)} ontology properties were loaded.")\n\n    chunk_size = choose_chunk_size(record["text"], int(profile.get("chunking.max_safe_chunk_chars", 24000)))\n    logger = SharedCallLogger(logs_dir)\n    backends = {\n        "layer01": _make_backend(\n            logger=logger, layer_tag="layer01_country_aware", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer01_linguistic_expression_extraction"),\n            fallback_max_tokens=max_tokens, fallback_timeout=request_timeout,\n            reasoning_effort=reasoning_effort,\n        ),\n        "layer02": _make_backend(\n            logger=logger, layer_tag="layer02_compact_topk", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer02_candidate_enrichment"),\n            fallback_max_tokens=384, fallback_timeout=60,\n            reasoning_effort=reasoning_effort,\n        ),\n        "layer04": _make_backend(\n            logger=logger, layer_tag="layer04_fallback_only", model_host=host, api_key=api_key,\n            cfg=_layer_cfg(profile_dict, "layer04_candidate_relation_extraction"),\n            fallback_max_tokens=384, fallback_timeout=60,\n            reasoning_effort=reasoning_effort,\n        ),\n        "other": _make_backend(\n            logger=logger, layer_tag="other", model_host=host, api_key=api_key,\n            cfg={}, fallback_max_tokens=768, fallback_timeout=90,\n            reasoning_effort=reasoning_effort,\n        ),\n    }\n    aliases = read_json(relation_aliases_path)\n    rag_adapter = PriorityOntologyRAGAdapter(\n        seed_ontology,\n        log_path=logs_dir / "ontology_retrieval.jsonl",\n        top_k=int(profile.get("rag.top_k", 6)),\n        query_expansions=profile.get("rag.query_expansions", {}) or {},\n        relation_aliases=aliases,\n        priority_property_ids=profile.get("rag.priority_property_ids", []) or [],\n    )\n    pipeline = build_pipeline(\n        backends=backends,\n        rag_adapter=rag_adapter,\n        profile_config=profile_dict,\n        relation_catalog_path=relation_catalog_path,\n        chunk_size=chunk_size,\n        run_dir=run_dir,\n        workers=workers,\n        verbose=verbose,\n    )\n    state = PipelineState(\n        document=build_document(record, input_jsonl),\n        llm_model=model_name,\n        user_guidance=guidance,\n        seed_ontology=seed_ontology,\n        artifact_dir=str(run_dir),\n        profile_name=profile.name,\n        profile_config=profile_dict,\n    )\n    runner = Runner(\n        pipeline=pipeline,\n        runs_root=str(run_dir.parent),\n        verbose=verbose,\n        max_workers=workers,\n        enable_checkpoints=True,\n        save_chunk_checkpoints=False,\n    )\n    manifest = {\n        "document_id": record["document_id"],\n        "title": record.get("title"),\n        "model_name": model_name,\n        "profile_name": profile.name,\n        "profile_path": str(profile_path),\n        "guidance_path": str(guidance_path),\n        "ontology_path": str(ontology_path),\n        "ontology_classes": len(seed_ontology.classes_by_uri),\n        "ontology_properties": len(seed_ontology.properties_by_uri),\n        "input_has_gold": False,\n        "chunk_size": chunk_size,\n        "whole_document_single_chunk_expected": len(record["text"]) <= chunk_size,\n        "workers": workers,\n        "layer02_compact_candidate_cap": profile_dict.get("layer02_compact_candidate_cap", 5),\n        "layer01_country_coverage_check": True,\n        "layer02_profile_guardrails": True,\n        "mention_aware_projection_after_execution_only": True,\n        "anti_cheating": profile.get("anti_cheating", {}),\n        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(run_dir / "run_manifest.json", manifest)\n\n    console_log = logs_dir / "console.log"\n    errors_path = logs_dir / "pipeline_errors.jsonl"\n    started = time.time()\n    with console_log.open("w", encoding="utf-8") as handle:\n        tee_out = Tee(sys.stdout, handle)\n        tee_err = Tee(sys.stderr, handle)\n        try:\n            with redirect_stdout(tee_out), redirect_stderr(tee_err):\n                final_state = runner.run(state, from_layer=0, to_layer=12, run_dir=run_dir)\n        except Exception as exc:\n            append_jsonl(errors_path, {\n                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n            })\n            raise\n\n    manifest.update({\n        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "elapsed_seconds": round(time.time() - started, 3),\n        "final_counts": state_counts(final_state),\n    })\n    write_json(run_dir / "run_manifest.json", manifest)\n    return final_state\n\n\ndef analyze_run(\n    *,\n    run_dir: str | Path,\n    gold_jsonl: str | Path,\n    catalog_path: str | Path,\n    aliases_path: str | Path,\n) -> dict[str, Any]:\n    # Retain all earlier analyses for comparison, then replace the principal\n    # benchmark-facing metrics with the audited v5 projection.\n    summary = v4.analyze_run(\n        run_dir=run_dir,\n        gold_jsonl=gold_jsonl,\n        catalog_path=catalog_path,\n        aliases_path=aliases_path,\n    )\n    run_dir = Path(run_dir)\n    gold = read_jsonl(gold_jsonl)[0]\n    states = {index: state for index, _, state in load_layer_states(run_dir)}\n    final_state = states[max(states)]\n\n    predictions = native_predictions_v5(final_state, gold, catalog_path, aliases_path)\n    evaluation = v2.strict_evaluate(predictions, gold)\n    cumulative = write_cumulative_evaluation_v5(run_dir, gold, catalog_path, aliases_path)\n    trace = write_relation_trace_v5(\n        run_dir=run_dir,\n        gold_jsonl=gold_jsonl,\n        catalog_path=catalog_path,\n        aliases_path=aliases_path,\n    )\n    projection_audit = write_entity_projection_audit(run_dir, final_state, gold)\n    native_views = write_native_views_v5(run_dir, final_state, gold, catalog_path, aliases_path)\n    failure_counts: dict[str, int] = {}\n    for row in trace:\n        failure_counts[row["first_failure"]] = failure_counts.get(row["first_failure"], 0) + 1\n\n    summary["strict_evaluation_before_v5_projection"] = summary.get("strict_evaluation")\n    summary["strict_evaluation"] = evaluation\n    summary["strict_evaluation_v5"] = evaluation\n    summary["strict_docred_predictions_v5"] = predictions\n    summary["cumulative_strict_evaluation_v5"] = cumulative\n    summary["gold_relation_trace_v5"] = trace\n    summary["failure_counts_v5"] = failure_counts\n    summary["entity_projection_audit_v5"] = projection_audit\n    summary.update(native_views)\n    write_json(run_dir / "analysis/analysis_summary_v5.json", summary)\n    return summary\n', 'docred_native_batch_v6.py': 'from __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport multiprocessing as mp\nimport os\nimport re\nimport shutil\nimport statistics\nimport time\nimport traceback\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any, Iterable, Iterator, Sequence\n\nimport docred_native_ablation_v5 as v5\n\n\nBATCH_VERSION = "v6-batch-native-v5.1"\nGOLD_FIELDS = {"entities", "relations", "labels", "vertexSet"}\n\n\ndef read_json(path: str | Path) -> Any:\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path: str | Path, value: Any) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef append_jsonl(path: str | Path, value: Any) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps(value, ensure_ascii=False, default=_json_default) + "\\n")\n\n\ndef read_jsonl(path: str | Path) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    with Path(path).open("r", encoding="utf-8") as handle:\n        for line_number, raw in enumerate(handle, 1):\n            line = raw.strip()\n            if not line:\n                continue\n            value = json.loads(line)\n            if not isinstance(value, dict):\n                raise TypeError(f"Expected a JSON object at {path}:{line_number}")\n            rows.append(value)\n    return rows\n\n\ndef iter_jsonl(path: str | Path) -> Iterator[dict[str, Any]]:\n    with Path(path).open("r", encoding="utf-8") as handle:\n        for line_number, raw in enumerate(handle, 1):\n            line = raw.strip()\n            if not line:\n                continue\n            value = json.loads(line)\n            if not isinstance(value, dict):\n                raise TypeError(f"Expected a JSON object at {path}:{line_number}")\n            yield value\n\n\ndef _json_default(value: Any) -> Any:\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, set):\n        return sorted(value)\n    if isinstance(value, tuple):\n        return list(value)\n    if hasattr(value, "__dict__"):\n        return vars(value)\n    return str(value)\n\n\ndef _stable_json(value: Any) -> str:\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=_json_default)\n\n\ndef sha256_bytes(payload: bytes) -> str:\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef sha256_file(path: str | Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef document_id(record: dict[str, Any], fallback_index: int | None = None) -> str:\n    value = (\n        record.get("document_id")\n        or record.get("doc_id")\n        or record.get("id")\n        or record.get("title")\n    )\n    if value is None:\n        if fallback_index is None:\n            raise KeyError("Document record has no document_id, doc_id, id, or title")\n        value = f"document_{fallback_index:06d}"\n    return str(value)\n\n\ndef safe_slug(value: str, *, max_length: int = 80) -> str:\n    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._-")\n    if not cleaned:\n        cleaned = "document"\n    suffix = hashlib.sha1(value.encode("utf-8")).hexdigest()[:10]\n    prefix_limit = max(8, max_length - len(suffix) - 1)\n    return f"{cleaned[:prefix_limit]}_{suffix}"\n\n\ndef strip_gold(record: dict[str, Any], task_guidance: dict[str, Any]) -> dict[str, Any]:\n    # Keep the original raw-text/document structure, but never expose benchmark\n    # entities or relations to the native NeoOLAF pipeline.\n    result = {key: value for key, value in record.items() if key not in GOLD_FIELDS}\n    result["document_id"] = document_id(record)\n    result["task_guidance"] = task_guidance\n    if "text" not in result or not str(result.get("text") or "").strip():\n        sentences = result.get("sentences") or []\n        if isinstance(sentences, list):\n            result["text"] = " ".join(str(sentence) for sentence in sentences)\n    if not str(result.get("text") or "").strip():\n        raise ValueError(f"Document {result[\'document_id\']} has no usable text")\n    return result\n\n\ndef record_matches_type(record: dict[str, Any], type_filter: str | None) -> bool:\n    if type_filter is None or str(type_filter).strip().lower() in {"", "all", "*"}:\n        return True\n    expected = str(type_filter).strip().lower()\n    actual = str(record.get("type") or record.get("split") or "").strip().lower()\n    return actual == expected\n\n\n@dataclass(frozen=True)\nclass PreparedDocument:\n    selection_index: int\n    source_index: int\n    document_id: str\n    title: str | None\n    slug: str\n    input_jsonl: str\n    gold_jsonl: str\n    run_dir: str\n    input_sha256: str\n    gold_sha256: str\n\n\n@dataclass(frozen=True)\nclass BatchRunConfig:\n    project_root: str\n    ontology_path: str\n    profile_path: str\n    guidance_path: str\n    relation_catalog_path: str\n    relation_aliases_path: str\n    model_name: str\n    host: str = "https://openrouter.ai/api/v1"\n    document_workers: int = 4\n    layer_workers: int = 16\n    reasoning_effort: str = "minimal"\n    max_tokens: int = 4096\n    request_timeout: int = 120\n    resume_completed: bool = True\n    retry_failed_documents: bool = True\n    document_attempts: int = 2\n    retry_backoff_seconds: float = 8.0\n    launch_stagger_seconds: float = 0.75\n    verbose_documents: bool = False\n    progress_every: int = 1\n\n\ndef prepare_documents(\n    *,\n    dataset_jsonl: str | Path,\n    task_guidance_path: str | Path,\n    batch_root: str | Path,\n    run_all_documents: bool,\n    smoke_document_limit: int = 5,\n    type_filter: str | None = None,\n    start_index: int = 0,\n) -> list[PreparedDocument]:\n    dataset_jsonl = Path(dataset_jsonl).resolve()\n    task_guidance_path = Path(task_guidance_path).resolve()\n    batch_root = Path(batch_root).resolve()\n    selected_root = batch_root / "selected_documents"\n    selected_root.mkdir(parents=True, exist_ok=True)\n    task_guidance = read_json(task_guidance_path)\n\n    selected: list[PreparedDocument] = []\n    aggregate_input = batch_root / "selected_input_no_gold.jsonl"\n    aggregate_gold = batch_root / "selected_gold.jsonl"\n    aggregate_input.parent.mkdir(parents=True, exist_ok=True)\n\n    input_handle = aggregate_input.open("w", encoding="utf-8")\n    gold_handle = aggregate_gold.open("w", encoding="utf-8")\n    try:\n        matched_index = 0\n        for source_index, record in enumerate(iter_jsonl(dataset_jsonl)):\n            if not record_matches_type(record, type_filter):\n                continue\n            if matched_index < start_index:\n                matched_index += 1\n                continue\n            if not run_all_documents and len(selected) >= smoke_document_limit:\n                break\n\n            doc_id = document_id(record, source_index)\n            slug = safe_slug(doc_id)\n            doc_root = selected_root / f"{len(selected):06d}_{slug}"\n            doc_root.mkdir(parents=True, exist_ok=True)\n            input_record = strip_gold(record, task_guidance)\n            input_path = doc_root / "input.jsonl"\n            gold_path = doc_root / "gold.jsonl"\n            input_line = json.dumps(input_record, ensure_ascii=False, separators=(",", ":"))\n            gold_line = json.dumps(record, ensure_ascii=False, separators=(",", ":"))\n            input_path.write_text(input_line + "\\n", encoding="utf-8")\n            gold_path.write_text(gold_line + "\\n", encoding="utf-8")\n            input_handle.write(input_line + "\\n")\n            gold_handle.write(gold_line + "\\n")\n\n            selected.append(PreparedDocument(\n                selection_index=len(selected),\n                source_index=source_index,\n                document_id=doc_id,\n                title=str(record.get("title")) if record.get("title") is not None else None,\n                slug=slug,\n                input_jsonl=str(input_path),\n                gold_jsonl=str(gold_path),\n                run_dir=str(batch_root / "document_runs" / f"{len(selected):06d}_{slug}"),\n                input_sha256=sha256_bytes((input_line + "\\n").encode("utf-8")),\n                gold_sha256=sha256_bytes((gold_line + "\\n").encode("utf-8")),\n            ))\n            matched_index += 1\n    finally:\n        input_handle.close()\n        gold_handle.close()\n\n    if not selected:\n        raise ValueError(\n            f"No documents selected from {dataset_jsonl}; type_filter={type_filter!r}, "\n            f"start_index={start_index}"\n        )\n\n    selection_manifest = {\n        "batch_version": BATCH_VERSION,\n        "dataset_jsonl": str(dataset_jsonl),\n        "dataset_sha256": sha256_file(dataset_jsonl),\n        "task_guidance_path": str(task_guidance_path),\n        "task_guidance_sha256": sha256_file(task_guidance_path),\n        "run_all_documents": run_all_documents,\n        "smoke_document_limit": smoke_document_limit,\n        "type_filter": type_filter,\n        "start_index": start_index,\n        "selected_count": len(selected),\n        "documents": [asdict(item) for item in selected],\n        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(batch_root / "selection_manifest.json", selection_manifest)\n    return selected\n\n\ndef _scientific_fingerprint(job: dict[str, Any], config: BatchRunConfig) -> str:\n    payload = {\n        "batch_version": BATCH_VERSION,\n        "input_sha256": job["input_sha256"],\n        "gold_sha256": job["gold_sha256"],\n        "model_name": config.model_name,\n        "host": config.host,\n        "reasoning_effort": config.reasoning_effort,\n        "layer_workers": config.layer_workers,\n        "max_tokens": config.max_tokens,\n        "request_timeout": config.request_timeout,\n        "ontology_sha256": sha256_file(config.ontology_path),\n        "profile_sha256": sha256_file(config.profile_path),\n        "guidance_sha256": sha256_file(config.guidance_path),\n        "relation_catalog_sha256": sha256_file(config.relation_catalog_path),\n        "relation_aliases_sha256": sha256_file(config.relation_aliases_path),\n    }\n    return sha256_bytes(_stable_json(payload).encode("utf-8"))\n\n\ndef _set_metrics(predicted: set[str], expected: set[str]) -> dict[str, Any]:\n    tp = predicted & expected\n    fp = predicted - expected\n    fn = expected - predicted\n    precision = len(tp) / len(predicted) if predicted else 0.0\n    recall = len(tp) / len(expected) if expected else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {\n        "predicted": len(predicted),\n        "gold": len(expected),\n        "true_positive": len(tp),\n        "false_positive": len(fp),\n        "false_negative": len(fn),\n        "precision": precision,\n        "recall": recall,\n        "f1": f1,\n        "tp": sorted(tp),\n        "fp": sorted(fp),\n        "fn": sorted(fn),\n    }\n\n\ndef _entity_metrics(run_dir: Path, gold: dict[str, Any], predictions: list[dict[str, Any]]) -> dict[str, Any]:\n    states = {index: state for index, _, state in v5.load_layer_states(run_dir)}\n    final_state = states[max(states)]\n    candidate_values = [\n        *(final_state.entity_candidates or []),\n        *(final_state.event_candidates or []),\n        *(final_state.attribute_candidates or []),\n    ]\n    predicted_entities: set[str] = set()\n    for candidate in candidate_values:\n        projection = v5.project_candidate_to_gold(candidate, gold)\n        entity_id = projection.get("entity_id")\n        if entity_id:\n            predicted_entities.add(str(entity_id))\n    gold_entities = set(str(key) for key in (gold.get("entities") or {}).keys())\n\n    predicted_endpoints: set[str] = set()\n    for row in predictions:\n        if not row.get("fully_mapped"):\n            continue\n        predicted_endpoints.add(str(row["head_id"]))\n        predicted_endpoints.add(str(row["tail_id"]))\n    gold_endpoints: set[str] = set()\n    for _, pairs in (gold.get("relations") or {}).items():\n        for head, tail in pairs:\n            gold_endpoints.add(str(head))\n            gold_endpoints.add(str(tail))\n    return {\n        "entity_inventory": _set_metrics(predicted_entities, gold_entities),\n        "relation_endpoint_inventory": _set_metrics(predicted_endpoints, gold_endpoints),\n    }\n\n\ndef _is_transient_exception(exc: BaseException) -> bool:\n    text = f"{type(exc).__name__}: {exc}".lower()\n    markers = [\n        "429", "rate limit", "too many requests", "timeout", "timed out",\n        "connection reset", "connection aborted", "502", "503", "504",\n        "server error", "temporarily unavailable",\n    ]\n    return any(marker in text for marker in markers)\n\n\ndef _worker_run_document(job: dict[str, Any], config_dict: dict[str, Any], api_key: str) -> dict[str, Any]:\n    config = BatchRunConfig(**config_dict)\n    run_dir = Path(job["run_dir"]).resolve()\n    result_path = run_dir / "document_result.json"\n    failure_path = run_dir / "document_failure.json"\n    fingerprint = _scientific_fingerprint(job, config)\n\n    if config.launch_stagger_seconds > 0:\n        time.sleep((int(job["selection_index"]) % max(1, config.document_workers)) * config.launch_stagger_seconds)\n\n    if config.resume_completed and result_path.is_file():\n        previous = read_json(result_path)\n        if previous.get("status") == "completed" and previous.get("scientific_fingerprint") == fingerprint:\n            previous["status"] = "skipped_completed"\n            previous["resumed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")\n            return previous\n\n    attempts = max(1, int(config.document_attempts if config.retry_failed_documents else 1))\n    started = time.time()\n    last_exc: BaseException | None = None\n\n    for attempt in range(1, attempts + 1):\n        try:\n            if run_dir.exists():\n                shutil.rmtree(run_dir)\n            run_dir.mkdir(parents=True, exist_ok=True)\n            v5.run_native_pipeline(\n                project_root=config.project_root,\n                input_jsonl=job["input_jsonl"],\n                ontology_path=config.ontology_path,\n                profile_path=config.profile_path,\n                guidance_path=config.guidance_path,\n                relation_catalog_path=config.relation_catalog_path,\n                relation_aliases_path=config.relation_aliases_path,\n                run_dir=run_dir,\n                model_name=config.model_name,\n                api_key=api_key,\n                host=config.host,\n                workers=config.layer_workers,\n                max_tokens=config.max_tokens,\n                request_timeout=config.request_timeout,\n                reasoning_effort=config.reasoning_effort,\n                verbose=config.verbose_documents,\n                clean_run_dir=False,\n            )\n            summary = v5.analyze_run(\n                run_dir=run_dir,\n                gold_jsonl=job["gold_jsonl"],\n                catalog_path=config.relation_catalog_path,\n                aliases_path=config.relation_aliases_path,\n            )\n            strict = summary["strict_evaluation_v5"]\n            predictions = summary["strict_docred_predictions_v5"]\n            gold = read_jsonl(job["gold_jsonl"])[0]\n            entities = _entity_metrics(run_dir, gold, predictions)\n            manifest = read_json(run_dir / "run_manifest.json")\n            result = {\n                "status": "completed",\n                "batch_version": BATCH_VERSION,\n                "scientific_fingerprint": fingerprint,\n                "selection_index": job["selection_index"],\n                "source_index": job["source_index"],\n                "document_id": job["document_id"],\n                "title": job.get("title"),\n                "slug": job["slug"],\n                "run_dir": str(run_dir),\n                "attempt": attempt,\n                "wall_seconds": round(time.time() - started, 3),\n                "pipeline_seconds": manifest.get("elapsed_seconds"),\n                "relation_metrics": strict,\n                "entity_metrics": entities,\n                "predictions": predictions,\n                "cumulative_evaluation": summary.get("cumulative_strict_evaluation_v5", []),\n                "failure_counts": summary.get("failure_counts_v5", {}),\n                "final_counts": manifest.get("final_counts", {}),\n                "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            }\n            write_json(result_path, result)\n            if failure_path.exists():\n                failure_path.unlink()\n            return result\n        except BaseException as exc:  # persist every document-level failure\n            last_exc = exc\n            transient = _is_transient_exception(exc)\n            failure = {\n                "status": "failed",\n                "batch_version": BATCH_VERSION,\n                "scientific_fingerprint": fingerprint,\n                "selection_index": job["selection_index"],\n                "source_index": job["source_index"],\n                "document_id": job["document_id"],\n                "title": job.get("title"),\n                "slug": job["slug"],\n                "run_dir": str(run_dir),\n                "attempt": attempt,\n                "attempts_allowed": attempts,\n                "transient": transient,\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n                "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            }\n            run_dir.mkdir(parents=True, exist_ok=True)\n            write_json(failure_path, failure)\n            if attempt < attempts and transient:\n                time.sleep(config.retry_backoff_seconds * attempt)\n                continue\n            return failure\n\n    assert last_exc is not None\n    raise last_exc\n\n\ndef run_documents_parallel(\n    *,\n    documents: Sequence[PreparedDocument],\n    config: BatchRunConfig,\n    api_key: str,\n    batch_root: str | Path,\n) -> list[dict[str, Any]]:\n    if not api_key:\n        raise ValueError("OPENROUTER_API_KEY is not set")\n    if not 1 <= int(config.document_workers) <= 5:\n        raise ValueError("document_workers must be between 1 and 5 for this notebook")\n    if int(config.layer_workers) < 1:\n        raise ValueError("layer_workers must be positive")\n\n    batch_root = Path(batch_root).resolve()\n    batch_root.mkdir(parents=True, exist_ok=True)\n    events_path = batch_root / "batch_events.jsonl"\n    # Begin a new event stream for this invocation. Per-document results remain resumable.\n    events_path.write_text("", encoding="utf-8")\n    config_dict = asdict(config)\n    jobs = [asdict(item) for item in documents]\n    results: list[dict[str, Any]] = []\n    started = time.time()\n\n    # spawn is required for safe Windows/Jupyter document isolation. Each child\n    # has independent stdout redirection, NeoOLAF state, and layer thread pools.\n    context = mp.get_context("spawn")\n    with ProcessPoolExecutor(max_workers=config.document_workers, mp_context=context) as pool:\n        future_to_job = {\n            pool.submit(_worker_run_document, job, config_dict, api_key): job\n            for job in jobs\n        }\n        completed = 0\n        for future in as_completed(future_to_job):\n            job = future_to_job[future]\n            completed += 1\n            try:\n                result = future.result()\n            except BaseException as exc:\n                result = {\n                    "status": "failed_parent_process",\n                    "selection_index": job["selection_index"],\n                    "source_index": job["source_index"],\n                    "document_id": job["document_id"],\n                    "title": job.get("title"),\n                    "slug": job["slug"],\n                    "run_dir": job["run_dir"],\n                    "error_type": type(exc).__name__,\n                    "error": str(exc),\n                    "traceback": traceback.format_exc(),\n                }\n            results.append(result)\n            event = {\n                "completed": completed,\n                "total": len(jobs),\n                "status": result.get("status"),\n                "document_id": result.get("document_id"),\n                "pipeline_seconds": result.get("pipeline_seconds"),\n                "elapsed_batch_seconds": round(time.time() - started, 3),\n                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n            }\n            append_jsonl(events_path, event)\n            progress_every = max(1, int(config.progress_every))\n            if completed == 1 or completed == len(jobs) or completed % progress_every == 0 or str(event["status"]).startswith("failed"):\n                print(\n                    f"[{completed}/{len(jobs)}] {event[\'status\']}: "\n                    f"{event[\'document_id\']} | pipeline={event[\'pipeline_seconds\']}s"\n                )\n\n    results.sort(key=lambda item: int(item.get("selection_index", 10**12)))\n    write_json(batch_root / "document_results.json", results)\n    return results\n\n\ndef _aggregate_count_metrics(rows: Iterable[dict[str, Any]]) -> dict[str, Any]:\n    rows = list(rows)\n    predicted = sum(int(row.get("predicted", 0)) for row in rows)\n    gold = sum(int(row.get("gold", 0)) for row in rows)\n    tp = sum(int(row.get("true_positive", 0)) for row in rows)\n    fp = sum(int(row.get("false_positive", 0)) for row in rows)\n    fn = sum(int(row.get("false_negative", 0)) for row in rows)\n    precision = tp / predicted if predicted else 0.0\n    recall = tp / gold if gold else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {\n        "predicted": predicted,\n        "gold": gold,\n        "true_positive": tp,\n        "false_positive": fp,\n        "false_negative": fn,\n        "precision": precision,\n        "recall": recall,\n        "f1": f1,\n    }\n\n\ndef _mean(values: Iterable[float]) -> float:\n    values = list(values)\n    return statistics.fmean(values) if values else 0.0\n\n\ndef _write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str] | None = None) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        path.write_text("", encoding="utf-8")\n        return\n    if fieldnames is None:\n        fieldnames = []\n        seen: set[str] = set()\n        for row in rows:\n            for key in row:\n                if key not in seen:\n                    seen.add(key)\n                    fieldnames.append(key)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef aggregate_batch_results(\n    *,\n    results: Sequence[dict[str, Any]],\n    batch_root: str | Path,\n    relation_catalog_path: str | Path | None = None,\n) -> dict[str, Any]:\n    batch_root = Path(batch_root).resolve()\n    analysis_root = batch_root / "aggregate_analysis"\n    analysis_root.mkdir(parents=True, exist_ok=True)\n    completed = [row for row in results if row.get("status") in {"completed", "skipped_completed"}]\n    failed = [row for row in results if row.get("status") not in {"completed", "skipped_completed"}]\n\n    relation_rows = [row["relation_metrics"] for row in completed]\n    entity_rows = [row["entity_metrics"]["entity_inventory"] for row in completed]\n    endpoint_rows = [row["entity_metrics"]["relation_endpoint_inventory"] for row in completed]\n\n    micro_rel = _aggregate_count_metrics(relation_rows)\n    micro_ent = _aggregate_count_metrics(entity_rows)\n    micro_end = _aggregate_count_metrics(endpoint_rows)\n    macro_rel = {\n        "precision": _mean(row.get("precision", 0.0) for row in relation_rows),\n        "recall": _mean(row.get("recall", 0.0) for row in relation_rows),\n        "f1": _mean(row.get("f1", 0.0) for row in relation_rows),\n    }\n    macro_ent = {\n        "precision": _mean(row.get("precision", 0.0) for row in entity_rows),\n        "recall": _mean(row.get("recall", 0.0) for row in entity_rows),\n        "f1": _mean(row.get("f1", 0.0) for row in entity_rows),\n    }\n\n    per_document: list[dict[str, Any]] = []\n    prediction_rows: list[dict[str, Any]] = []\n    relation_counts: dict[str, dict[str, int]] = {}\n    failure_counts: dict[str, int] = {}\n    for result in completed:\n        rel = result["relation_metrics"]\n        ent = result["entity_metrics"]["entity_inventory"]\n        endpoint = result["entity_metrics"]["relation_endpoint_inventory"]\n        per_document.append({\n            "selection_index": result.get("selection_index"),\n            "source_index": result.get("source_index"),\n            "document_id": result.get("document_id"),\n            "title": result.get("title"),\n            "status": result.get("status"),\n            "pipeline_seconds": result.get("pipeline_seconds"),\n            "wall_seconds": result.get("wall_seconds"),\n            "relation_predicted": rel.get("predicted"),\n            "relation_gold": rel.get("gold"),\n            "relation_tp": rel.get("true_positive"),\n            "relation_fp": rel.get("false_positive"),\n            "relation_fn": rel.get("false_negative"),\n            "relation_precision": rel.get("precision"),\n            "relation_recall": rel.get("recall"),\n            "relation_f1": rel.get("f1"),\n            "entity_precision": ent.get("precision"),\n            "entity_recall": ent.get("recall"),\n            "entity_f1": ent.get("f1"),\n            "endpoint_precision": endpoint.get("precision"),\n            "endpoint_recall": endpoint.get("recall"),\n            "endpoint_f1": endpoint.get("f1"),\n            "run_dir": result.get("run_dir"),\n        })\n        for prediction in result.get("predictions", []):\n            prediction_rows.append({\n                "document_id": result.get("document_id"),\n                "title": result.get("title"),\n                **prediction,\n            })\n        for bucket in ("tp", "fp", "fn"):\n            for triple in rel.get(bucket, []):\n                if len(triple) != 3:\n                    continue\n                relation_id = str(triple[1])\n                counts = relation_counts.setdefault(relation_id, {"tp": 0, "fp": 0, "fn": 0})\n                counts[bucket] += 1\n        for reason, count in (result.get("failure_counts") or {}).items():\n            failure_counts[str(reason)] = failure_counts.get(str(reason), 0) + int(count)\n\n    labels: dict[str, str] = {}\n    if relation_catalog_path and Path(relation_catalog_path).is_file():\n        catalog = read_json(relation_catalog_path)\n        for row in catalog.get("properties", []):\n            relation_id = str(row.get("property_id") or row.get("id") or "")\n            if relation_id:\n                labels[relation_id] = str(row.get("label") or "")\n    per_relation: list[dict[str, Any]] = []\n    for relation_id, counts in sorted(relation_counts.items()):\n        predicted = counts["tp"] + counts["fp"]\n        gold = counts["tp"] + counts["fn"]\n        precision = counts["tp"] / predicted if predicted else 0.0\n        recall = counts["tp"] / gold if gold else 0.0\n        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n        per_relation.append({\n            "relation_id": relation_id,\n            "label": labels.get(relation_id, ""),\n            "predicted": predicted,\n            "gold": gold,\n            "true_positive": counts["tp"],\n            "false_positive": counts["fp"],\n            "false_negative": counts["fn"],\n            "precision": precision,\n            "recall": recall,\n            "f1": f1,\n        })\n\n    # Aggregate cumulative layer evaluation using micro counts at each layer.\n    layer_buckets: dict[int, dict[str, Any]] = {}\n    for result in completed:\n        for row in result.get("cumulative_evaluation", []):\n            index = int(row.get("layer_index", -1))\n            bucket = layer_buckets.setdefault(index, {\n                "layer_index": index,\n                "layer_name": row.get("layer_name"),\n                "predicted": 0,\n                "gold": 0,\n                "true_positive": 0,\n                "false_positive": 0,\n                "false_negative": 0,\n            })\n            for key in ["predicted", "gold", "true_positive", "false_positive", "false_negative"]:\n                bucket[key] += int(row.get(key, 0))\n    cumulative: list[dict[str, Any]] = []\n    for index in sorted(layer_buckets):\n        row = layer_buckets[index]\n        p = row["true_positive"] / row["predicted"] if row["predicted"] else 0.0\n        r = row["true_positive"] / row["gold"] if row["gold"] else 0.0\n        f1 = 2 * p * r / (p + r) if p + r else 0.0\n        cumulative.append({**row, "precision": p, "recall": r, "f1": f1})\n\n    total_pipeline_seconds = sum(float(row.get("pipeline_seconds") or 0.0) for row in completed)\n    summary = {\n        "batch_version": BATCH_VERSION,\n        "documents_requested": len(results),\n        "documents_completed_or_resumed": len(completed),\n        "documents_failed": len(failed),\n        "micro_relation": micro_rel,\n        "macro_relation": macro_rel,\n        "micro_entity_inventory": micro_ent,\n        "macro_entity_inventory": macro_ent,\n        "micro_relation_endpoint_inventory": micro_end,\n        "total_document_pipeline_seconds": total_pipeline_seconds,\n        "mean_document_pipeline_seconds": total_pipeline_seconds / len(completed) if completed else 0.0,\n        "median_document_pipeline_seconds": statistics.median(\n            [float(row.get("pipeline_seconds") or 0.0) for row in completed]\n        ) if completed else 0.0,\n        "first_failure_counts": failure_counts,\n        "failed_documents": failed,\n        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n\n    write_json(analysis_root / "batch_summary.json", summary)\n    write_json(analysis_root / "document_results_compact.json", per_document)\n    write_json(analysis_root / "per_relation_metrics.json", per_relation)\n    write_json(analysis_root / "cumulative_layer_micro_evaluation.json", cumulative)\n    _write_csv(analysis_root / "per_document_metrics.csv", per_document)\n    _write_csv(analysis_root / "per_relation_metrics.csv", per_relation)\n    _write_csv(analysis_root / "cumulative_layer_micro_evaluation.csv", cumulative)\n    with (analysis_root / "predictions.jsonl").open("w", encoding="utf-8") as handle:\n        for row in prediction_rows:\n            handle.write(json.dumps(row, ensure_ascii=False, default=_json_default) + "\\n")\n    with (analysis_root / "failed_documents.jsonl").open("w", encoding="utf-8") as handle:\n        for row in failed:\n            handle.write(json.dumps(row, ensure_ascii=False, default=_json_default) + "\\n")\n    return {\n        "summary": summary,\n        "per_document": per_document,\n        "per_relation": per_relation,\n        "cumulative": cumulative,\n        "predictions": prediction_rows,\n        "failed": failed,\n    }\n\n\ndef run_batch(\n    *,\n    dataset_jsonl: str | Path,\n    task_guidance_path: str | Path,\n    batch_root: str | Path,\n    run_all_documents: bool,\n    smoke_document_limit: int,\n    type_filter: str | None,\n    start_index: int,\n    config: BatchRunConfig,\n    api_key: str,\n) -> dict[str, Any]:\n    batch_root = Path(batch_root).resolve()\n    batch_root.mkdir(parents=True, exist_ok=True)\n    documents = prepare_documents(\n        dataset_jsonl=dataset_jsonl,\n        task_guidance_path=task_guidance_path,\n        batch_root=batch_root,\n        run_all_documents=run_all_documents,\n        smoke_document_limit=smoke_document_limit,\n        type_filter=type_filter,\n        start_index=start_index,\n    )\n    invocation = {\n        "batch_version": BATCH_VERSION,\n        "run_all_documents": run_all_documents,\n        "smoke_document_limit": smoke_document_limit,\n        "type_filter": type_filter,\n        "start_index": start_index,\n        "selected_documents": len(documents),\n        "config": {**asdict(config), "api_key": "NOT_STORED"},\n        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(batch_root / "batch_invocation.json", invocation)\n    results = run_documents_parallel(\n        documents=documents,\n        config=config,\n        api_key=api_key,\n        batch_root=batch_root,\n    )\n    aggregate = aggregate_batch_results(\n        results=results,\n        batch_root=batch_root,\n        relation_catalog_path=config.relation_catalog_path,\n    )\n    invocation["completed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")\n    invocation["documents_completed_or_resumed"] = aggregate["summary"]["documents_completed_or_resumed"]\n    invocation["documents_failed"] = aggregate["summary"]["documents_failed"]\n    write_json(batch_root / "batch_invocation.json", invocation)\n    return {\n        "documents": [asdict(item) for item in documents],\n        "results": results,\n        **aggregate,\n    }\n', 'docred_native_batch_v6_1.py': 'from __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport multiprocessing as mp\nimport os\nimport re\nimport shutil\nimport statistics\nimport time\nimport traceback\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any, Iterable, Iterator, Sequence\n\nimport docred_native_ablation_v5 as v5\n\n\nBATCH_VERSION = "v6.1-batch-native-v5.1-analysis-recovery"\nGOLD_FIELDS = {"entities", "relations", "labels", "vertexSet"}\n\n\ndef read_json(path: str | Path) -> Any:\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path: str | Path, value: Any) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef append_jsonl(path: str | Path, value: Any) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps(value, ensure_ascii=False, default=_json_default) + "\\n")\n\n\ndef read_jsonl(path: str | Path) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    with Path(path).open("r", encoding="utf-8") as handle:\n        for line_number, raw in enumerate(handle, 1):\n            line = raw.strip()\n            if not line:\n                continue\n            value = json.loads(line)\n            if not isinstance(value, dict):\n                raise TypeError(f"Expected a JSON object at {path}:{line_number}")\n            rows.append(value)\n    return rows\n\n\ndef iter_jsonl(path: str | Path) -> Iterator[dict[str, Any]]:\n    with Path(path).open("r", encoding="utf-8") as handle:\n        for line_number, raw in enumerate(handle, 1):\n            line = raw.strip()\n            if not line:\n                continue\n            value = json.loads(line)\n            if not isinstance(value, dict):\n                raise TypeError(f"Expected a JSON object at {path}:{line_number}")\n            yield value\n\n\ndef _json_default(value: Any) -> Any:\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, set):\n        return sorted(value)\n    if isinstance(value, tuple):\n        return list(value)\n    if hasattr(value, "__dict__"):\n        return vars(value)\n    return str(value)\n\n\ndef _stable_json(value: Any) -> str:\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=_json_default)\n\n\ndef sha256_bytes(payload: bytes) -> str:\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef sha256_file(path: str | Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef document_id(record: dict[str, Any], fallback_index: int | None = None) -> str:\n    value = (\n        record.get("document_id")\n        or record.get("doc_id")\n        or record.get("id")\n        or record.get("title")\n    )\n    if value is None:\n        if fallback_index is None:\n            raise KeyError("Document record has no document_id, doc_id, id, or title")\n        value = f"document_{fallback_index:06d}"\n    return str(value)\n\n\ndef safe_slug(value: str, *, max_length: int = 80) -> str:\n    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._-")\n    if not cleaned:\n        cleaned = "document"\n    suffix = hashlib.sha1(value.encode("utf-8")).hexdigest()[:10]\n    prefix_limit = max(8, max_length - len(suffix) - 1)\n    return f"{cleaned[:prefix_limit]}_{suffix}"\n\n\ndef strip_gold(record: dict[str, Any], task_guidance: dict[str, Any]) -> dict[str, Any]:\n    # Keep the original raw-text/document structure, but never expose benchmark\n    # entities or relations to the native NeoOLAF pipeline.\n    result = {key: value for key, value in record.items() if key not in GOLD_FIELDS}\n    result["document_id"] = document_id(record)\n    result["task_guidance"] = task_guidance\n    if "text" not in result or not str(result.get("text") or "").strip():\n        sentences = result.get("sentences") or []\n        if isinstance(sentences, list):\n            result["text"] = " ".join(str(sentence) for sentence in sentences)\n    if not str(result.get("text") or "").strip():\n        raise ValueError(f"Document {result[\'document_id\']} has no usable text")\n    return result\n\n\ndef record_matches_type(record: dict[str, Any], type_filter: str | None) -> bool:\n    if type_filter is None or str(type_filter).strip().lower() in {"", "all", "*"}:\n        return True\n    expected = str(type_filter).strip().lower()\n    actual = str(record.get("type") or record.get("split") or "").strip().lower()\n    return actual == expected\n\n\n@dataclass(frozen=True)\nclass PreparedDocument:\n    selection_index: int\n    source_index: int\n    document_id: str\n    title: str | None\n    slug: str\n    input_jsonl: str\n    gold_jsonl: str\n    run_dir: str\n    input_sha256: str\n    gold_sha256: str\n\n\n@dataclass(frozen=True)\nclass BatchRunConfig:\n    project_root: str\n    ontology_path: str\n    profile_path: str\n    guidance_path: str\n    relation_catalog_path: str\n    relation_aliases_path: str\n    model_name: str\n    host: str = "https://openrouter.ai/api/v1"\n    document_workers: int = 4\n    layer_workers: int = 16\n    reasoning_effort: str = "minimal"\n    max_tokens: int = 4096\n    request_timeout: int = 120\n    resume_completed: bool = True\n    retry_failed_documents: bool = True\n    document_attempts: int = 2\n    retry_backoff_seconds: float = 8.0\n    launch_stagger_seconds: float = 0.75\n    verbose_documents: bool = False\n    progress_every: int = 1\n\n\ndef prepare_documents(\n    *,\n    dataset_jsonl: str | Path,\n    task_guidance_path: str | Path,\n    batch_root: str | Path,\n    run_all_documents: bool,\n    smoke_document_limit: int = 5,\n    type_filter: str | None = None,\n    start_index: int = 0,\n) -> list[PreparedDocument]:\n    dataset_jsonl = Path(dataset_jsonl).resolve()\n    task_guidance_path = Path(task_guidance_path).resolve()\n    batch_root = Path(batch_root).resolve()\n    selected_root = batch_root / "selected_documents"\n    selected_root.mkdir(parents=True, exist_ok=True)\n    task_guidance = read_json(task_guidance_path)\n\n    selected: list[PreparedDocument] = []\n    aggregate_input = batch_root / "selected_input_no_gold.jsonl"\n    aggregate_gold = batch_root / "selected_gold.jsonl"\n    aggregate_input.parent.mkdir(parents=True, exist_ok=True)\n\n    input_handle = aggregate_input.open("w", encoding="utf-8")\n    gold_handle = aggregate_gold.open("w", encoding="utf-8")\n    try:\n        matched_index = 0\n        for source_index, record in enumerate(iter_jsonl(dataset_jsonl)):\n            if not record_matches_type(record, type_filter):\n                continue\n            if matched_index < start_index:\n                matched_index += 1\n                continue\n            if not run_all_documents and len(selected) >= smoke_document_limit:\n                break\n\n            doc_id = document_id(record, source_index)\n            slug = safe_slug(doc_id)\n            doc_root = selected_root / f"{len(selected):06d}_{slug}"\n            doc_root.mkdir(parents=True, exist_ok=True)\n            input_record = strip_gold(record, task_guidance)\n            input_path = doc_root / "input.jsonl"\n            gold_path = doc_root / "gold.jsonl"\n            input_line = json.dumps(input_record, ensure_ascii=False, separators=(",", ":"))\n            gold_line = json.dumps(record, ensure_ascii=False, separators=(",", ":"))\n            input_path.write_text(input_line + "\\n", encoding="utf-8")\n            gold_path.write_text(gold_line + "\\n", encoding="utf-8")\n            input_handle.write(input_line + "\\n")\n            gold_handle.write(gold_line + "\\n")\n\n            selected.append(PreparedDocument(\n                selection_index=len(selected),\n                source_index=source_index,\n                document_id=doc_id,\n                title=str(record.get("title")) if record.get("title") is not None else None,\n                slug=slug,\n                input_jsonl=str(input_path),\n                gold_jsonl=str(gold_path),\n                run_dir=str(batch_root / "document_runs" / f"{len(selected):06d}_{slug}"),\n                input_sha256=sha256_bytes((input_line + "\\n").encode("utf-8")),\n                gold_sha256=sha256_bytes((gold_line + "\\n").encode("utf-8")),\n            ))\n            matched_index += 1\n    finally:\n        input_handle.close()\n        gold_handle.close()\n\n    if not selected:\n        raise ValueError(\n            f"No documents selected from {dataset_jsonl}; type_filter={type_filter!r}, "\n            f"start_index={start_index}"\n        )\n\n    selection_manifest = {\n        "batch_version": BATCH_VERSION,\n        "dataset_jsonl": str(dataset_jsonl),\n        "dataset_sha256": sha256_file(dataset_jsonl),\n        "task_guidance_path": str(task_guidance_path),\n        "task_guidance_sha256": sha256_file(task_guidance_path),\n        "run_all_documents": run_all_documents,\n        "smoke_document_limit": smoke_document_limit,\n        "type_filter": type_filter,\n        "start_index": start_index,\n        "selected_count": len(selected),\n        "documents": [asdict(item) for item in selected],\n        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(batch_root / "selection_manifest.json", selection_manifest)\n    return selected\n\n\ndef _scientific_fingerprint(job: dict[str, Any], config: BatchRunConfig) -> str:\n    payload = {\n        "batch_version": BATCH_VERSION,\n        "input_sha256": job["input_sha256"],\n        "gold_sha256": job["gold_sha256"],\n        "model_name": config.model_name,\n        "host": config.host,\n        "reasoning_effort": config.reasoning_effort,\n        "layer_workers": config.layer_workers,\n        "max_tokens": config.max_tokens,\n        "request_timeout": config.request_timeout,\n        "ontology_sha256": sha256_file(config.ontology_path),\n        "profile_sha256": sha256_file(config.profile_path),\n        "guidance_sha256": sha256_file(config.guidance_path),\n        "relation_catalog_sha256": sha256_file(config.relation_catalog_path),\n        "relation_aliases_sha256": sha256_file(config.relation_aliases_path),\n    }\n    return sha256_bytes(_stable_json(payload).encode("utf-8"))\n\n\ndef _set_metrics(predicted: set[str], expected: set[str]) -> dict[str, Any]:\n    tp = predicted & expected\n    fp = predicted - expected\n    fn = expected - predicted\n    precision = len(tp) / len(predicted) if predicted else 0.0\n    recall = len(tp) / len(expected) if expected else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {\n        "predicted": len(predicted),\n        "gold": len(expected),\n        "true_positive": len(tp),\n        "false_positive": len(fp),\n        "false_negative": len(fn),\n        "precision": precision,\n        "recall": recall,\n        "f1": f1,\n        "tp": sorted(tp),\n        "fp": sorted(fp),\n        "fn": sorted(fn),\n    }\n\n\ndef _entity_metrics(run_dir: Path, gold: dict[str, Any], predictions: list[dict[str, Any]]) -> dict[str, Any]:\n    states = {index: state for index, _, state in v5.load_layer_states(run_dir)}\n    final_state = states[max(states)]\n    candidate_values = [\n        *(final_state.entity_candidates or []),\n        *(final_state.event_candidates or []),\n        *(final_state.attribute_candidates or []),\n    ]\n    predicted_entities: set[str] = set()\n    for candidate in candidate_values:\n        projection = v5.project_candidate_to_gold(candidate, gold)\n        entity_id = projection.get("entity_id")\n        if entity_id:\n            predicted_entities.add(str(entity_id))\n    gold_entities = set(str(key) for key in (gold.get("entities") or {}).keys())\n\n    predicted_endpoints: set[str] = set()\n    for row in predictions:\n        if not row.get("fully_mapped"):\n            continue\n        predicted_endpoints.add(str(row["head_id"]))\n        predicted_endpoints.add(str(row["tail_id"]))\n    gold_endpoints: set[str] = set()\n    for _, pairs in (gold.get("relations") or {}).items():\n        for head, tail in pairs:\n            gold_endpoints.add(str(head))\n            gold_endpoints.add(str(tail))\n    return {\n        "entity_inventory": _set_metrics(predicted_entities, gold_entities),\n        "relation_endpoint_inventory": _set_metrics(predicted_endpoints, gold_endpoints),\n    }\n\n\ndef _is_transient_exception(exc: BaseException) -> bool:\n    text = f"{type(exc).__name__}: {exc}".lower()\n    markers = [\n        "429", "rate limit", "too many requests", "timeout", "timed out",\n        "connection reset", "connection aborted", "502", "503", "504",\n        "server error", "temporarily unavailable",\n    ]\n    return any(marker in text for marker in markers)\n\n\ndef _completed_pipeline_manifest(\n    run_dir: Path,\n    job: dict[str, Any],\n    config: BatchRunConfig,\n) -> dict[str, Any] | None:\n    """Return a compatible completed pipeline manifest, if one exists.\n\n    This permits evaluation-only recovery after a post-pipeline analysis failure\n    without deleting artifacts or paying for the LLM calls a second time.\n    """\n    path = run_dir / "run_manifest.json"\n    if not path.is_file():\n        return None\n    try:\n        manifest = read_json(path)\n    except Exception:\n        return None\n    if not manifest.get("completed_at"):\n        return None\n    if str(manifest.get("document_id")) != str(job.get("document_id")):\n        return None\n    if str(manifest.get("model_name")) != str(config.model_name):\n        return None\n    # The v5.1 run manifest records the exact scientific resource paths. Resolve\n    # them when possible, but tolerate copied repositories with the same basename.\n    for manifest_key, configured in [\n        ("profile_path", config.profile_path),\n        ("guidance_path", config.guidance_path),\n        ("ontology_path", config.ontology_path),\n    ]:\n        recorded = str(manifest.get(manifest_key) or "")\n        if recorded and Path(recorded).name != Path(configured).name:\n            return None\n    return manifest\n\n\ndef _completed_result_from_saved_pipeline(\n    *,\n    job: dict[str, Any],\n    config: BatchRunConfig,\n    run_dir: Path,\n    fingerprint: str,\n    manifest: dict[str, Any],\n    started: float,\n) -> dict[str, Any]:\n    """Rebuild evaluation from saved Layer 0--12 artifacts only."""\n    summary = v5.analyze_run(\n        run_dir=run_dir,\n        gold_jsonl=job["gold_jsonl"],\n        catalog_path=config.relation_catalog_path,\n        aliases_path=config.relation_aliases_path,\n    )\n    strict = summary["strict_evaluation_v5"]\n    predictions = summary["strict_docred_predictions_v5"]\n    gold = read_jsonl(job["gold_jsonl"])[0]\n    entities = _entity_metrics(run_dir, gold, predictions)\n    return {\n        "status": "completed",\n        "batch_version": BATCH_VERSION,\n        "scientific_fingerprint": fingerprint,\n        "selection_index": job["selection_index"],\n        "source_index": job["source_index"],\n        "document_id": job["document_id"],\n        "title": job.get("title"),\n        "slug": job["slug"],\n        "run_dir": str(run_dir),\n        "attempt": 0,\n        "wall_seconds": round(time.time() - started, 3),\n        "pipeline_seconds": manifest.get("elapsed_seconds"),\n        "pipeline_reused": True,\n        "analysis_recovered_from_saved_artifacts": True,\n        "relation_metrics": strict,\n        "entity_metrics": entities,\n        "predictions": predictions,\n        "cumulative_evaluation": summary.get("cumulative_strict_evaluation_v5", []),\n        "failure_counts": summary.get("failure_counts_v5", {}),\n        "final_counts": manifest.get("final_counts", {}),\n        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n\n\ndef _worker_run_document(job: dict[str, Any], config_dict: dict[str, Any], api_key: str) -> dict[str, Any]:\n    config = BatchRunConfig(**config_dict)\n    run_dir = Path(job["run_dir"]).resolve()\n    result_path = run_dir / "document_result.json"\n    failure_path = run_dir / "document_failure.json"\n    fingerprint = _scientific_fingerprint(job, config)\n\n    if config.launch_stagger_seconds > 0:\n        time.sleep((int(job["selection_index"]) % max(1, config.document_workers)) * config.launch_stagger_seconds)\n\n    if config.resume_completed and result_path.is_file():\n        previous = read_json(result_path)\n        if previous.get("status") == "completed" and previous.get("scientific_fingerprint") == fingerprint:\n            previous["status"] = "skipped_completed"\n            previous["resumed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")\n            return previous\n\n    attempts = max(1, int(config.document_attempts if config.retry_failed_documents else 1))\n    started = time.time()\n    last_exc: BaseException | None = None\n\n    # A previous v6 invocation may have completed all NeoOLAF layers and failed\n    # only while writing evaluation CSV files. Recover analysis first and never\n    # delete or rerun a compatible completed pipeline merely because evaluation\n    # failed. This protects both artifacts and OpenRouter credits.\n    manifest = _completed_pipeline_manifest(run_dir, job, config)\n    if config.resume_completed and manifest is not None:\n        try:\n            result = _completed_result_from_saved_pipeline(\n                job=job, config=config, run_dir=run_dir, fingerprint=fingerprint,\n                manifest=manifest, started=started,\n            )\n            write_json(result_path, result)\n            if failure_path.exists():\n                failure_path.unlink()\n            return result\n        except BaseException as exc:\n            failure = {\n                "status": "failed_analysis_recovery",\n                "failure_stage": "analysis_only",\n                "pipeline_completed": True,\n                "pipeline_seconds": manifest.get("elapsed_seconds"),\n                "batch_version": BATCH_VERSION,\n                "scientific_fingerprint": fingerprint,\n                "selection_index": job["selection_index"],\n                "source_index": job["source_index"],\n                "document_id": job["document_id"],\n                "title": job.get("title"),\n                "slug": job["slug"],\n                "run_dir": str(run_dir),\n                "attempt": 0,\n                "attempts_allowed": attempts,\n                "transient": False,\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n                "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            }\n            write_json(failure_path, failure)\n            return failure\n\n    for attempt in range(1, attempts + 1):\n        try:\n            if run_dir.exists():\n                shutil.rmtree(run_dir)\n            run_dir.mkdir(parents=True, exist_ok=True)\n            v5.run_native_pipeline(\n                project_root=config.project_root,\n                input_jsonl=job["input_jsonl"],\n                ontology_path=config.ontology_path,\n                profile_path=config.profile_path,\n                guidance_path=config.guidance_path,\n                relation_catalog_path=config.relation_catalog_path,\n                relation_aliases_path=config.relation_aliases_path,\n                run_dir=run_dir,\n                model_name=config.model_name,\n                api_key=api_key,\n                host=config.host,\n                workers=config.layer_workers,\n                max_tokens=config.max_tokens,\n                request_timeout=config.request_timeout,\n                reasoning_effort=config.reasoning_effort,\n                verbose=config.verbose_documents,\n                clean_run_dir=False,\n            )\n            summary = v5.analyze_run(\n                run_dir=run_dir,\n                gold_jsonl=job["gold_jsonl"],\n                catalog_path=config.relation_catalog_path,\n                aliases_path=config.relation_aliases_path,\n            )\n            strict = summary["strict_evaluation_v5"]\n            predictions = summary["strict_docred_predictions_v5"]\n            gold = read_jsonl(job["gold_jsonl"])[0]\n            entities = _entity_metrics(run_dir, gold, predictions)\n            manifest = read_json(run_dir / "run_manifest.json")\n            result = {\n                "status": "completed",\n                "batch_version": BATCH_VERSION,\n                "scientific_fingerprint": fingerprint,\n                "selection_index": job["selection_index"],\n                "source_index": job["source_index"],\n                "document_id": job["document_id"],\n                "title": job.get("title"),\n                "slug": job["slug"],\n                "run_dir": str(run_dir),\n                "attempt": attempt,\n                "wall_seconds": round(time.time() - started, 3),\n                "pipeline_seconds": manifest.get("elapsed_seconds"),\n                "relation_metrics": strict,\n                "entity_metrics": entities,\n                "predictions": predictions,\n                "cumulative_evaluation": summary.get("cumulative_strict_evaluation_v5", []),\n                "failure_counts": summary.get("failure_counts_v5", {}),\n                "final_counts": manifest.get("final_counts", {}),\n                "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            }\n            write_json(result_path, result)\n            if failure_path.exists():\n                failure_path.unlink()\n            return result\n        except BaseException as exc:  # persist every document-level failure\n            last_exc = exc\n            transient = _is_transient_exception(exc)\n            failure = {\n                "status": "failed",\n                "batch_version": BATCH_VERSION,\n                "scientific_fingerprint": fingerprint,\n                "selection_index": job["selection_index"],\n                "source_index": job["source_index"],\n                "document_id": job["document_id"],\n                "title": job.get("title"),\n                "slug": job["slug"],\n                "run_dir": str(run_dir),\n                "attempt": attempt,\n                "attempts_allowed": attempts,\n                "transient": transient,\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n                "traceback": traceback.format_exc(),\n                "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            }\n            run_dir.mkdir(parents=True, exist_ok=True)\n            write_json(failure_path, failure)\n            if attempt < attempts and transient:\n                time.sleep(config.retry_backoff_seconds * attempt)\n                continue\n            return failure\n\n    assert last_exc is not None\n    raise last_exc\n\n\ndef run_documents_parallel(\n    *,\n    documents: Sequence[PreparedDocument],\n    config: BatchRunConfig,\n    api_key: str,\n    batch_root: str | Path,\n) -> list[dict[str, Any]]:\n    if not api_key:\n        raise ValueError("OPENROUTER_API_KEY is not set")\n    if not 1 <= int(config.document_workers) <= 5:\n        raise ValueError("document_workers must be between 1 and 5 for this notebook")\n    if int(config.layer_workers) < 1:\n        raise ValueError("layer_workers must be positive")\n\n    batch_root = Path(batch_root).resolve()\n    batch_root.mkdir(parents=True, exist_ok=True)\n    events_path = batch_root / "batch_events.jsonl"\n    # Begin a new event stream for this invocation. Per-document results remain resumable.\n    events_path.write_text("", encoding="utf-8")\n    config_dict = asdict(config)\n    jobs = [asdict(item) for item in documents]\n    results: list[dict[str, Any]] = []\n    started = time.time()\n\n    # spawn is required for safe Windows/Jupyter document isolation. Each child\n    # has independent stdout redirection, NeoOLAF state, and layer thread pools.\n    context = mp.get_context("spawn")\n    with ProcessPoolExecutor(max_workers=config.document_workers, mp_context=context) as pool:\n        future_to_job = {\n            pool.submit(_worker_run_document, job, config_dict, api_key): job\n            for job in jobs\n        }\n        completed = 0\n        for future in as_completed(future_to_job):\n            job = future_to_job[future]\n            completed += 1\n            try:\n                result = future.result()\n            except BaseException as exc:\n                result = {\n                    "status": "failed_parent_process",\n                    "selection_index": job["selection_index"],\n                    "source_index": job["source_index"],\n                    "document_id": job["document_id"],\n                    "title": job.get("title"),\n                    "slug": job["slug"],\n                    "run_dir": job["run_dir"],\n                    "error_type": type(exc).__name__,\n                    "error": str(exc),\n                    "traceback": traceback.format_exc(),\n                }\n            results.append(result)\n            event = {\n                "completed": completed,\n                "total": len(jobs),\n                "status": result.get("status"),\n                "document_id": result.get("document_id"),\n                "pipeline_seconds": result.get("pipeline_seconds"),\n                "pipeline_reused": result.get("pipeline_reused", False),\n                "attempt": result.get("attempt"),\n                "transient": result.get("transient"),\n                "error_type": result.get("error_type"),\n                "error": result.get("error"),\n                "elapsed_batch_seconds": round(time.time() - started, 3),\n                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n            }\n            append_jsonl(events_path, event)\n            progress_every = max(1, int(config.progress_every))\n            if completed == 1 or completed == len(jobs) or completed % progress_every == 0 or str(event["status"]).startswith("failed"):\n                pipeline_text = (\n                    f"{event[\'pipeline_seconds\']}s"\n                    if event.get("pipeline_seconds") is not None else "n/a"\n                )\n                suffix = " | reused saved pipeline" if event.get("pipeline_reused") else ""\n                if str(event.get("status", "")).startswith("failed"):\n                    suffix += (\n                        f" | {event.get(\'error_type\')}: {event.get(\'error\')}"\n                        f" | transient={event.get(\'transient\')}"\n                    )\n                print(\n                    f"[{completed}/{len(jobs)}] {event[\'status\']}: "\n                    f"{event[\'document_id\']} | pipeline={pipeline_text}{suffix}"\n                )\n\n    results.sort(key=lambda item: int(item.get("selection_index", 10**12)))\n    write_json(batch_root / "document_results.json", results)\n    return results\n\n\ndef _aggregate_count_metrics(rows: Iterable[dict[str, Any]]) -> dict[str, Any]:\n    rows = list(rows)\n    predicted = sum(int(row.get("predicted", 0)) for row in rows)\n    gold = sum(int(row.get("gold", 0)) for row in rows)\n    tp = sum(int(row.get("true_positive", 0)) for row in rows)\n    fp = sum(int(row.get("false_positive", 0)) for row in rows)\n    fn = sum(int(row.get("false_negative", 0)) for row in rows)\n    precision = tp / predicted if predicted else 0.0\n    recall = tp / gold if gold else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {\n        "predicted": predicted,\n        "gold": gold,\n        "true_positive": tp,\n        "false_positive": fp,\n        "false_negative": fn,\n        "precision": precision,\n        "recall": recall,\n        "f1": f1,\n    }\n\n\ndef _mean(values: Iterable[float]) -> float:\n    values = list(values)\n    return statistics.fmean(values) if values else 0.0\n\n\ndef _write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str] | None = None) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        path.write_text("", encoding="utf-8")\n        return\n    if fieldnames is None:\n        fieldnames = []\n        seen: set[str] = set()\n        for row in rows:\n            for key in row:\n                if key not in seen:\n                    seen.add(key)\n                    fieldnames.append(key)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef aggregate_batch_results(\n    *,\n    results: Sequence[dict[str, Any]],\n    batch_root: str | Path,\n    relation_catalog_path: str | Path | None = None,\n) -> dict[str, Any]:\n    batch_root = Path(batch_root).resolve()\n    analysis_root = batch_root / "aggregate_analysis"\n    analysis_root.mkdir(parents=True, exist_ok=True)\n    completed = [row for row in results if row.get("status") in {"completed", "skipped_completed"}]\n    failed = [row for row in results if row.get("status") not in {"completed", "skipped_completed"}]\n\n    relation_rows = [row["relation_metrics"] for row in completed]\n    entity_rows = [row["entity_metrics"]["entity_inventory"] for row in completed]\n    endpoint_rows = [row["entity_metrics"]["relation_endpoint_inventory"] for row in completed]\n\n    micro_rel = _aggregate_count_metrics(relation_rows)\n    micro_ent = _aggregate_count_metrics(entity_rows)\n    micro_end = _aggregate_count_metrics(endpoint_rows)\n    macro_rel = {\n        "precision": _mean(row.get("precision", 0.0) for row in relation_rows),\n        "recall": _mean(row.get("recall", 0.0) for row in relation_rows),\n        "f1": _mean(row.get("f1", 0.0) for row in relation_rows),\n    }\n    macro_ent = {\n        "precision": _mean(row.get("precision", 0.0) for row in entity_rows),\n        "recall": _mean(row.get("recall", 0.0) for row in entity_rows),\n        "f1": _mean(row.get("f1", 0.0) for row in entity_rows),\n    }\n\n    per_document: list[dict[str, Any]] = []\n    prediction_rows: list[dict[str, Any]] = []\n    relation_counts: dict[str, dict[str, int]] = {}\n    failure_counts: dict[str, int] = {}\n    for result in completed:\n        rel = result["relation_metrics"]\n        ent = result["entity_metrics"]["entity_inventory"]\n        endpoint = result["entity_metrics"]["relation_endpoint_inventory"]\n        per_document.append({\n            "selection_index": result.get("selection_index"),\n            "source_index": result.get("source_index"),\n            "document_id": result.get("document_id"),\n            "title": result.get("title"),\n            "status": result.get("status"),\n            "pipeline_seconds": result.get("pipeline_seconds"),\n            "wall_seconds": result.get("wall_seconds"),\n            "relation_predicted": rel.get("predicted"),\n            "relation_gold": rel.get("gold"),\n            "relation_tp": rel.get("true_positive"),\n            "relation_fp": rel.get("false_positive"),\n            "relation_fn": rel.get("false_negative"),\n            "relation_precision": rel.get("precision"),\n            "relation_recall": rel.get("recall"),\n            "relation_f1": rel.get("f1"),\n            "entity_precision": ent.get("precision"),\n            "entity_recall": ent.get("recall"),\n            "entity_f1": ent.get("f1"),\n            "endpoint_precision": endpoint.get("precision"),\n            "endpoint_recall": endpoint.get("recall"),\n            "endpoint_f1": endpoint.get("f1"),\n            "run_dir": result.get("run_dir"),\n        })\n        for prediction in result.get("predictions", []):\n            prediction_rows.append({\n                "document_id": result.get("document_id"),\n                "title": result.get("title"),\n                **prediction,\n            })\n        for bucket in ("tp", "fp", "fn"):\n            for triple in rel.get(bucket, []):\n                if len(triple) != 3:\n                    continue\n                relation_id = str(triple[1])\n                counts = relation_counts.setdefault(relation_id, {"tp": 0, "fp": 0, "fn": 0})\n                counts[bucket] += 1\n        for reason, count in (result.get("failure_counts") or {}).items():\n            failure_counts[str(reason)] = failure_counts.get(str(reason), 0) + int(count)\n\n    labels: dict[str, str] = {}\n    if relation_catalog_path and Path(relation_catalog_path).is_file():\n        catalog = read_json(relation_catalog_path)\n        for row in catalog.get("properties", []):\n            relation_id = str(row.get("property_id") or row.get("id") or "")\n            if relation_id:\n                labels[relation_id] = str(row.get("label") or "")\n    per_relation: list[dict[str, Any]] = []\n    for relation_id, counts in sorted(relation_counts.items()):\n        predicted = counts["tp"] + counts["fp"]\n        gold = counts["tp"] + counts["fn"]\n        precision = counts["tp"] / predicted if predicted else 0.0\n        recall = counts["tp"] / gold if gold else 0.0\n        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n        per_relation.append({\n            "relation_id": relation_id,\n            "label": labels.get(relation_id, ""),\n            "predicted": predicted,\n            "gold": gold,\n            "true_positive": counts["tp"],\n            "false_positive": counts["fp"],\n            "false_negative": counts["fn"],\n            "precision": precision,\n            "recall": recall,\n            "f1": f1,\n        })\n\n    # Aggregate cumulative layer evaluation using micro counts at each layer.\n    layer_buckets: dict[int, dict[str, Any]] = {}\n    for result in completed:\n        for row in result.get("cumulative_evaluation", []):\n            index = int(row.get("layer_index", -1))\n            bucket = layer_buckets.setdefault(index, {\n                "layer_index": index,\n                "layer_name": row.get("layer_name"),\n                "predicted": 0,\n                "gold": 0,\n                "true_positive": 0,\n                "false_positive": 0,\n                "false_negative": 0,\n            })\n            for key in ["predicted", "gold", "true_positive", "false_positive", "false_negative"]:\n                bucket[key] += int(row.get(key, 0))\n    cumulative: list[dict[str, Any]] = []\n    for index in sorted(layer_buckets):\n        row = layer_buckets[index]\n        p = row["true_positive"] / row["predicted"] if row["predicted"] else 0.0\n        r = row["true_positive"] / row["gold"] if row["gold"] else 0.0\n        f1 = 2 * p * r / (p + r) if p + r else 0.0\n        cumulative.append({**row, "precision": p, "recall": r, "f1": f1})\n\n    total_pipeline_seconds = sum(float(row.get("pipeline_seconds") or 0.0) for row in completed)\n    summary = {\n        "batch_version": BATCH_VERSION,\n        "documents_requested": len(results),\n        "documents_completed_or_resumed": len(completed),\n        "documents_failed": len(failed),\n        "micro_relation": micro_rel,\n        "macro_relation": macro_rel,\n        "micro_entity_inventory": micro_ent,\n        "macro_entity_inventory": macro_ent,\n        "micro_relation_endpoint_inventory": micro_end,\n        "total_document_pipeline_seconds": total_pipeline_seconds,\n        "mean_document_pipeline_seconds": total_pipeline_seconds / len(completed) if completed else 0.0,\n        "median_document_pipeline_seconds": statistics.median(\n            [float(row.get("pipeline_seconds") or 0.0) for row in completed]\n        ) if completed else 0.0,\n        "first_failure_counts": failure_counts,\n        "failed_documents": failed,\n        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n\n    write_json(analysis_root / "batch_summary.json", summary)\n    write_json(analysis_root / "document_results_compact.json", per_document)\n    write_json(analysis_root / "per_relation_metrics.json", per_relation)\n    write_json(analysis_root / "cumulative_layer_micro_evaluation.json", cumulative)\n    _write_csv(analysis_root / "per_document_metrics.csv", per_document)\n    _write_csv(analysis_root / "per_relation_metrics.csv", per_relation)\n    _write_csv(analysis_root / "cumulative_layer_micro_evaluation.csv", cumulative)\n    with (analysis_root / "predictions.jsonl").open("w", encoding="utf-8") as handle:\n        for row in prediction_rows:\n            handle.write(json.dumps(row, ensure_ascii=False, default=_json_default) + "\\n")\n    with (analysis_root / "failed_documents.jsonl").open("w", encoding="utf-8") as handle:\n        for row in failed:\n            handle.write(json.dumps(row, ensure_ascii=False, default=_json_default) + "\\n")\n    return {\n        "summary": summary,\n        "per_document": per_document,\n        "per_relation": per_relation,\n        "cumulative": cumulative,\n        "predictions": prediction_rows,\n        "failed": failed,\n    }\n\n\ndef run_batch(\n    *,\n    dataset_jsonl: str | Path,\n    task_guidance_path: str | Path,\n    batch_root: str | Path,\n    run_all_documents: bool,\n    smoke_document_limit: int,\n    type_filter: str | None,\n    start_index: int,\n    config: BatchRunConfig,\n    api_key: str,\n) -> dict[str, Any]:\n    batch_root = Path(batch_root).resolve()\n    batch_root.mkdir(parents=True, exist_ok=True)\n    documents = prepare_documents(\n        dataset_jsonl=dataset_jsonl,\n        task_guidance_path=task_guidance_path,\n        batch_root=batch_root,\n        run_all_documents=run_all_documents,\n        smoke_document_limit=smoke_document_limit,\n        type_filter=type_filter,\n        start_index=start_index,\n    )\n    invocation = {\n        "batch_version": BATCH_VERSION,\n        "run_all_documents": run_all_documents,\n        "smoke_document_limit": smoke_document_limit,\n        "type_filter": type_filter,\n        "start_index": start_index,\n        "selected_documents": len(documents),\n        "config": {**asdict(config), "api_key": "NOT_STORED"},\n        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(batch_root / "batch_invocation.json", invocation)\n    results = run_documents_parallel(\n        documents=documents,\n        config=config,\n        api_key=api_key,\n        batch_root=batch_root,\n    )\n    aggregate = aggregate_batch_results(\n        results=results,\n        batch_root=batch_root,\n        relation_catalog_path=config.relation_catalog_path,\n    )\n    invocation["completed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")\n    invocation["documents_completed_or_resumed"] = aggregate["summary"]["documents_completed_or_resumed"]\n    invocation["documents_failed"] = aggregate["summary"]["documents_failed"]\n    write_json(batch_root / "batch_invocation.json", invocation)\n    return {\n        "documents": [asdict(item) for item in documents],\n        "results": results,\n        **aggregate,\n    }\n', 'docred_native_batch_v6_2_dev_streaming.py': 'from __future__ import annotations\n\nimport csv\nimport json\nimport multiprocessing as mp\nimport statistics\nimport time\nimport traceback\nfrom concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any, Iterator\n\nimport docred_native_batch_v6_1 as base\n\n\nORCHESTRATOR_VERSION = "v6.2-dev-filtered-bounded-streaming"\nEXACT_REQUIRED_TYPE = "dev"\n\nBatchRunConfig = base.BatchRunConfig\nPreparedDocument = base.PreparedDocument\nread_json = base.read_json\nwrite_json = base.write_json\nappend_jsonl = base.append_jsonl\niter_jsonl = base.iter_jsonl\nsha256_bytes = base.sha256_bytes\nsha256_file = base.sha256_file\ndocument_id = base.document_id\nsafe_slug = base.safe_slug\nstrip_gold = base.strip_gold\n\n\ndef _json_default(value: Any) -> Any:\n    return base._json_default(value)\n\n\ndef is_exact_type(record: dict[str, Any], required_type: str = EXACT_REQUIRED_TYPE) -> bool:\n    """Match only the literal JSON key ``type``; never fall back to ``split``."""\n    return str(record.get("type") or "").strip().lower() == str(required_type).strip().lower()\n\n\ndef count_exact_type_records(\n    dataset_jsonl: str | Path,\n    *,\n    required_type: str = EXACT_REQUIRED_TYPE,\n    first_n_ids: int = 5,\n) -> dict[str, Any]:\n    """Count matching records with a single streaming pass and O(1) document memory."""\n    total = 0\n    matching = 0\n    first_ids: list[str] = []\n    for source_index, record in enumerate(iter_jsonl(dataset_jsonl)):\n        total += 1\n        if not is_exact_type(record, required_type):\n            continue\n        matching += 1\n        if len(first_ids) < max(0, int(first_n_ids)):\n            first_ids.append(document_id(record, source_index))\n    return {\n        "total_records": total,\n        "matching_records": matching,\n        "required_type": required_type,\n        "first_matching_ids": first_ids,\n    }\n\n\ndef iter_prepared_documents_streaming(\n    *,\n    dataset_jsonl: str | Path,\n    task_guidance_path: str | Path,\n    batch_root: str | Path,\n    run_all_documents: bool,\n    smoke_document_limit: int = 5,\n    required_type: str = EXACT_REQUIRED_TYPE,\n    start_index: int = 0,\n) -> Iterator[PreparedDocument]:\n    """Yield one prepared matching document at a time.\n\n    The caller controls how far this generator is advanced. The bounded launcher\n    advances it only when a document-worker slot is available, so the parent\n    process never constructs a list of the full dev split.\n    """\n    dataset_jsonl = Path(dataset_jsonl).resolve()\n    task_guidance_path = Path(task_guidance_path).resolve()\n    batch_root = Path(batch_root).resolve()\n    selected_root = batch_root / "selected_documents"\n    selected_root.mkdir(parents=True, exist_ok=True)\n    task_guidance = read_json(task_guidance_path)\n\n    selection_jsonl = batch_root / "selection_documents.jsonl"\n    selection_jsonl.parent.mkdir(parents=True, exist_ok=True)\n    selection_jsonl.write_text("", encoding="utf-8")\n\n    selected_count = 0\n    matching_index = 0\n    for source_index, record in enumerate(iter_jsonl(dataset_jsonl)):\n        if not is_exact_type(record, required_type):\n            continue\n        if matching_index < int(start_index):\n            matching_index += 1\n            continue\n        if not run_all_documents and selected_count >= int(smoke_document_limit):\n            break\n\n        doc_id = document_id(record, source_index)\n        slug = safe_slug(doc_id)\n        selection_index = selected_count\n        doc_root = selected_root / f"{selection_index:06d}_{slug}"\n        doc_root.mkdir(parents=True, exist_ok=True)\n\n        # Gold is stripped before the native pipeline input is serialized.\n        input_record = strip_gold(record, task_guidance)\n        input_line = json.dumps(input_record, ensure_ascii=False, separators=(",", ":"))\n        gold_line = json.dumps(record, ensure_ascii=False, separators=(",", ":"))\n        input_path = doc_root / "input.jsonl"\n        gold_path = doc_root / "gold.jsonl"\n        input_path.write_text(input_line + "\\n", encoding="utf-8")\n        gold_path.write_text(gold_line + "\\n", encoding="utf-8")\n\n        prepared = PreparedDocument(\n            selection_index=selection_index,\n            source_index=source_index,\n            document_id=doc_id,\n            title=str(record.get("title")) if record.get("title") is not None else None,\n            slug=slug,\n            input_jsonl=str(input_path),\n            gold_jsonl=str(gold_path),\n            run_dir=str(batch_root / "document_runs" / f"{selection_index:06d}_{slug}"),\n            input_sha256=sha256_bytes((input_line + "\\n").encode("utf-8")),\n            gold_sha256=sha256_bytes((gold_line + "\\n").encode("utf-8")),\n        )\n        append_jsonl(selection_jsonl, asdict(prepared))\n        yield prepared\n\n        selected_count += 1\n        matching_index += 1\n\n\ndef _load_parent_resumable_result(\n    job: dict[str, Any],\n    config: BatchRunConfig,\n) -> dict[str, Any] | None:\n    if not config.resume_completed:\n        return None\n    run_dir = Path(job["run_dir"]).resolve()\n    result_path = run_dir / "document_result.json"\n    if not result_path.is_file():\n        return None\n    try:\n        previous = read_json(result_path)\n        fingerprint = base._scientific_fingerprint(job, config)\n    except Exception:\n        return None\n    if previous.get("status") != "completed":\n        return None\n    if previous.get("scientific_fingerprint") != fingerprint:\n        return None\n    resumed = dict(previous)\n    resumed["status"] = "skipped_completed"\n    resumed["resumed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")\n    return resumed\n\n\ndef _persist_parent_failure(job: dict[str, Any], config: BatchRunConfig, exc: BaseException) -> dict[str, Any]:\n    run_dir = Path(job["run_dir"]).resolve()\n    run_dir.mkdir(parents=True, exist_ok=True)\n    failure = {\n        "status": "failed_parent_process",\n        "batch_version": base.BATCH_VERSION,\n        "orchestrator_version": ORCHESTRATOR_VERSION,\n        "scientific_fingerprint": base._scientific_fingerprint(job, config),\n        "selection_index": job["selection_index"],\n        "source_index": job["source_index"],\n        "document_id": job["document_id"],\n        "title": job.get("title"),\n        "slug": job["slug"],\n        "run_dir": str(run_dir),\n        "transient": False,\n        "error_type": type(exc).__name__,\n        "error": str(exc),\n        "traceback": traceback.format_exc(),\n        "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(run_dir / "document_failure.json", failure)\n    return failure\n\n\ndef _event_from_result(\n    result: dict[str, Any],\n    *,\n    completed: int,\n    total: int,\n    started: float,\n) -> dict[str, Any]:\n    return {\n        "completed": completed,\n        "total": total,\n        "status": result.get("status"),\n        "document_id": result.get("document_id"),\n        "pipeline_seconds": result.get("pipeline_seconds"),\n        "pipeline_reused": result.get("pipeline_reused", False),\n        "attempt": result.get("attempt"),\n        "transient": result.get("transient"),\n        "error_type": result.get("error_type"),\n        "error": result.get("error"),\n        "elapsed_batch_seconds": round(time.time() - started, 3),\n        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n\n\ndef _print_progress(event: dict[str, Any], *, progress_every: int) -> None:\n    completed = int(event["completed"])\n    total = int(event["total"])\n    status = str(event.get("status") or "")\n    should_print = (\n        completed == 1\n        or completed == total\n        or completed % max(1, int(progress_every)) == 0\n        or status.startswith("failed")\n    )\n    if not should_print:\n        return\n    pipeline = event.get("pipeline_seconds")\n    pipeline_text = f"{pipeline}s" if pipeline is not None else "n/a"\n    suffix = " | reused saved pipeline" if event.get("pipeline_reused") else ""\n    if status.startswith("failed"):\n        suffix += (\n            f" | {event.get(\'error_type\')}: {event.get(\'error\')}"\n            f" | transient={event.get(\'transient\')}"\n        )\n    print(\n        f"[{completed}/{total}] {status}: {event.get(\'document_id\')} "\n        f"| pipeline={pipeline_text}{suffix}"\n    )\n\n\ndef run_documents_bounded_streaming(\n    *,\n    documents: Iterator[PreparedDocument],\n    expected_total: int,\n    config: BatchRunConfig,\n    api_key: str,\n    batch_root: str | Path,\n) -> dict[str, Any]:\n    """Run with at most ``document_workers`` submitted jobs in parent memory."""\n    if not api_key:\n        raise ValueError("OPENROUTER_API_KEY is not set")\n    if not 1 <= int(config.document_workers) <= 5:\n        raise ValueError("document_workers must be between 1 and 5")\n    if int(config.layer_workers) < 1:\n        raise ValueError("layer_workers must be positive")\n\n    batch_root = Path(batch_root).resolve()\n    batch_root.mkdir(parents=True, exist_ok=True)\n    events_path = batch_root / "batch_events.jsonl"\n    events_path.write_text("", encoding="utf-8")\n\n    started = time.time()\n    prepared = 0\n    completed = 0\n    completed_ok = 0\n    failed = 0\n    resumed = 0\n    exhausted = False\n    config_dict = asdict(config)\n    pending: dict[Any, dict[str, Any]] = {}\n\n    def record_result(result: dict[str, Any]) -> None:\n        nonlocal completed, completed_ok, failed, resumed\n        completed += 1\n        status = str(result.get("status") or "")\n        if status in {"completed", "skipped_completed"}:\n            completed_ok += 1\n        else:\n            failed += 1\n        if status == "skipped_completed" or result.get("pipeline_reused"):\n            resumed += 1\n        event = _event_from_result(\n            result,\n            completed=completed,\n            total=expected_total,\n            started=started,\n        )\n        append_jsonl(events_path, event)\n        _print_progress(event, progress_every=config.progress_every)\n\n    context = mp.get_context("spawn")\n    with ProcessPoolExecutor(max_workers=config.document_workers, mp_context=context) as pool:\n        while not exhausted or pending:\n            # Advance the JSONL generator only while a worker slot is available.\n            while not exhausted and len(pending) < int(config.document_workers):\n                try:\n                    prepared_document = next(documents)\n                except StopIteration:\n                    exhausted = True\n                    break\n                prepared += 1\n                job = asdict(prepared_document)\n\n                parent_resume = _load_parent_resumable_result(job, config)\n                if parent_resume is not None:\n                    record_result(parent_resume)\n                    continue\n\n                future = pool.submit(base._worker_run_document, job, config_dict, api_key)\n                pending[future] = job\n\n            if not pending:\n                continue\n\n            done, _ = wait(tuple(pending), return_when=FIRST_COMPLETED)\n            for future in done:\n                job = pending.pop(future)\n                try:\n                    result = future.result()\n                except BaseException as exc:\n                    result = _persist_parent_failure(job, config, exc)\n                record_result(result)\n\n    return {\n        "expected_total": expected_total,\n        "documents_prepared": prepared,\n        "documents_finished": completed,\n        "documents_completed_or_resumed": completed_ok,\n        "documents_failed": failed,\n        "documents_resumed": resumed,\n        "max_pending_documents": int(config.document_workers),\n        "elapsed_seconds": round(time.time() - started, 3),\n    }\n\n\ndef _metric_accumulator() -> dict[str, int]:\n    return {"predicted": 0, "gold": 0, "true_positive": 0, "false_positive": 0, "false_negative": 0}\n\n\ndef _add_metrics(acc: dict[str, int], row: dict[str, Any]) -> None:\n    for key in acc:\n        acc[key] += int(row.get(key, 0) or 0)\n\n\ndef _finish_metrics(acc: dict[str, int]) -> dict[str, Any]:\n    predicted = acc["predicted"]\n    gold = acc["gold"]\n    tp = acc["true_positive"]\n    precision = tp / predicted if predicted else 0.0\n    recall = tp / gold if gold else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {**acc, "precision": precision, "recall": recall, "f1": f1}\n\n\ndef _write_csv_rows(path: Path, rows: list[dict[str, Any]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        path.write_text("", encoding="utf-8")\n        return\n    fieldnames: list[str] = []\n    seen: set[str] = set()\n    for row in rows:\n        for key in row:\n            if key not in seen:\n                seen.add(key)\n                fieldnames.append(key)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef aggregate_batch_results_streaming(\n    *,\n    batch_root: str | Path,\n    relation_catalog_path: str | Path | None = None,\n) -> dict[str, Any]:\n    """Aggregate per-document files without loading document results into a list."""\n    batch_root = Path(batch_root).resolve()\n    selection_path = batch_root / "selection_documents.jsonl"\n    if not selection_path.is_file():\n        raise FileNotFoundError(f"Missing selection stream: {selection_path}")\n\n    analysis_root = batch_root / "aggregate_analysis"\n    analysis_root.mkdir(parents=True, exist_ok=True)\n    per_document_csv = analysis_root / "per_document_metrics.csv"\n    per_document_jsonl = analysis_root / "document_results_compact.jsonl"\n    predictions_jsonl = analysis_root / "predictions.jsonl"\n    failures_jsonl = analysis_root / "failed_documents.jsonl"\n\n    per_document_fields = [\n        "selection_index", "source_index", "document_id", "title", "status",\n        "pipeline_seconds", "wall_seconds", "relation_predicted", "relation_gold",\n        "relation_tp", "relation_fp", "relation_fn", "relation_precision",\n        "relation_recall", "relation_f1", "entity_precision", "entity_recall",\n        "entity_f1", "endpoint_precision", "endpoint_recall", "endpoint_f1", "run_dir",\n    ]\n\n    relation_acc = _metric_accumulator()\n    entity_acc = _metric_accumulator()\n    endpoint_acc = _metric_accumulator()\n    macro_relation_sums = {"precision": 0.0, "recall": 0.0, "f1": 0.0}\n    macro_entity_sums = {"precision": 0.0, "recall": 0.0, "f1": 0.0}\n    relation_counts: dict[str, dict[str, int]] = {}\n    failure_counts: dict[str, int] = {}\n    layer_buckets: dict[int, dict[str, Any]] = {}\n    pipeline_seconds_values: list[float] = []  # numeric only; no document payloads\n    failed_preview: list[dict[str, Any]] = []\n\n    requested = 0\n    completed_count = 0\n    failed_count = 0\n\n    with (\n        per_document_csv.open("w", encoding="utf-8", newline="") as csv_handle,\n        per_document_jsonl.open("w", encoding="utf-8") as compact_handle,\n        predictions_jsonl.open("w", encoding="utf-8") as predictions_handle,\n        failures_jsonl.open("w", encoding="utf-8") as failures_handle,\n    ):\n        csv_writer = csv.DictWriter(csv_handle, fieldnames=per_document_fields, extrasaction="ignore")\n        csv_writer.writeheader()\n\n        for job in iter_jsonl(selection_path):\n            requested += 1\n            run_dir = Path(job["run_dir"]).resolve()\n            result_path = run_dir / "document_result.json"\n            failure_path = run_dir / "document_failure.json"\n\n            result: dict[str, Any] | None = None\n            if result_path.is_file():\n                candidate = read_json(result_path)\n                if candidate.get("status") == "completed":\n                    result = candidate\n\n            if result is None:\n                failed_count += 1\n                if failure_path.is_file():\n                    failure = read_json(failure_path)\n                else:\n                    failure = {\n                        "status": "missing_result",\n                        "selection_index": job.get("selection_index"),\n                        "source_index": job.get("source_index"),\n                        "document_id": job.get("document_id"),\n                        "title": job.get("title"),\n                        "run_dir": str(run_dir),\n                        "error_type": "MissingResult",\n                        "error": "No completed document_result.json or document_failure.json was found.",\n                    }\n                failures_handle.write(json.dumps(failure, ensure_ascii=False, default=_json_default) + "\\n")\n                if len(failed_preview) < 50:\n                    failed_preview.append(failure)\n                continue\n\n            completed_count += 1\n            rel = result["relation_metrics"]\n            ent = result["entity_metrics"]["entity_inventory"]\n            endpoint = result["entity_metrics"]["relation_endpoint_inventory"]\n            _add_metrics(relation_acc, rel)\n            _add_metrics(entity_acc, ent)\n            _add_metrics(endpoint_acc, endpoint)\n            for key in macro_relation_sums:\n                macro_relation_sums[key] += float(rel.get(key, 0.0) or 0.0)\n                macro_entity_sums[key] += float(ent.get(key, 0.0) or 0.0)\n\n            pipeline_seconds = float(result.get("pipeline_seconds") or 0.0)\n            pipeline_seconds_values.append(pipeline_seconds)\n            row = {\n                "selection_index": result.get("selection_index"),\n                "source_index": result.get("source_index"),\n                "document_id": result.get("document_id"),\n                "title": result.get("title"),\n                "status": result.get("status"),\n                "pipeline_seconds": result.get("pipeline_seconds"),\n                "wall_seconds": result.get("wall_seconds"),\n                "relation_predicted": rel.get("predicted"),\n                "relation_gold": rel.get("gold"),\n                "relation_tp": rel.get("true_positive"),\n                "relation_fp": rel.get("false_positive"),\n                "relation_fn": rel.get("false_negative"),\n                "relation_precision": rel.get("precision"),\n                "relation_recall": rel.get("recall"),\n                "relation_f1": rel.get("f1"),\n                "entity_precision": ent.get("precision"),\n                "entity_recall": ent.get("recall"),\n                "entity_f1": ent.get("f1"),\n                "endpoint_precision": endpoint.get("precision"),\n                "endpoint_recall": endpoint.get("recall"),\n                "endpoint_f1": endpoint.get("f1"),\n                "run_dir": result.get("run_dir"),\n            }\n            csv_writer.writerow(row)\n            compact_handle.write(json.dumps(row, ensure_ascii=False, default=_json_default) + "\\n")\n\n            for prediction in result.get("predictions", []):\n                predictions_handle.write(json.dumps({\n                    "document_id": result.get("document_id"),\n                    "title": result.get("title"),\n                    **prediction,\n                }, ensure_ascii=False, default=_json_default) + "\\n")\n\n            for bucket in ("tp", "fp", "fn"):\n                for triple in rel.get(bucket, []):\n                    if len(triple) != 3:\n                        continue\n                    relation_id = str(triple[1])\n                    counts = relation_counts.setdefault(relation_id, {"tp": 0, "fp": 0, "fn": 0})\n                    counts[bucket] += 1\n\n            for reason, count in (result.get("failure_counts") or {}).items():\n                failure_counts[str(reason)] = failure_counts.get(str(reason), 0) + int(count)\n\n            for layer_row in result.get("cumulative_evaluation", []):\n                index = int(layer_row.get("layer_index", -1))\n                bucket = layer_buckets.setdefault(index, {\n                    "layer_index": index,\n                    "layer_name": layer_row.get("layer_name"),\n                    "predicted": 0,\n                    "gold": 0,\n                    "true_positive": 0,\n                    "false_positive": 0,\n                    "false_negative": 0,\n                })\n                for key in ["predicted", "gold", "true_positive", "false_positive", "false_negative"]:\n                    bucket[key] += int(layer_row.get(key, 0) or 0)\n\n    labels: dict[str, str] = {}\n    if relation_catalog_path and Path(relation_catalog_path).is_file():\n        catalog = read_json(relation_catalog_path)\n        for relation in catalog.get("properties", []):\n            relation_id = str(relation.get("property_id") or relation.get("id") or "")\n            if relation_id:\n                labels[relation_id] = str(relation.get("label") or "")\n\n    per_relation: list[dict[str, Any]] = []\n    for relation_id, counts in sorted(relation_counts.items()):\n        predicted = counts["tp"] + counts["fp"]\n        gold = counts["tp"] + counts["fn"]\n        precision = counts["tp"] / predicted if predicted else 0.0\n        recall = counts["tp"] / gold if gold else 0.0\n        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n        per_relation.append({\n            "relation_id": relation_id,\n            "label": labels.get(relation_id, ""),\n            "predicted": predicted,\n            "gold": gold,\n            "true_positive": counts["tp"],\n            "false_positive": counts["fp"],\n            "false_negative": counts["fn"],\n            "precision": precision,\n            "recall": recall,\n            "f1": f1,\n        })\n\n    cumulative: list[dict[str, Any]] = []\n    for index in sorted(layer_buckets):\n        row = layer_buckets[index]\n        precision = row["true_positive"] / row["predicted"] if row["predicted"] else 0.0\n        recall = row["true_positive"] / row["gold"] if row["gold"] else 0.0\n        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n        cumulative.append({**row, "precision": precision, "recall": recall, "f1": f1})\n\n    micro_relation = _finish_metrics(relation_acc)\n    micro_entity = _finish_metrics(entity_acc)\n    micro_endpoint = _finish_metrics(endpoint_acc)\n    macro_relation = {\n        key: macro_relation_sums[key] / completed_count if completed_count else 0.0\n        for key in macro_relation_sums\n    }\n    macro_entity = {\n        key: macro_entity_sums[key] / completed_count if completed_count else 0.0\n        for key in macro_entity_sums\n    }\n    total_pipeline_seconds = sum(pipeline_seconds_values)\n\n    summary = {\n        "batch_version": base.BATCH_VERSION,\n        "orchestrator_version": ORCHESTRATOR_VERSION,\n        "required_type_key": "type",\n        "required_type_value": EXACT_REQUIRED_TYPE,\n        "documents_requested": requested,\n        "documents_completed_or_resumed": completed_count,\n        "documents_failed": failed_count,\n        "micro_relation": micro_relation,\n        "macro_relation": macro_relation,\n        "micro_entity_inventory": micro_entity,\n        "macro_entity_inventory": macro_entity,\n        "micro_relation_endpoint_inventory": micro_endpoint,\n        "total_document_pipeline_seconds": total_pipeline_seconds,\n        "mean_document_pipeline_seconds": total_pipeline_seconds / completed_count if completed_count else 0.0,\n        "median_document_pipeline_seconds": statistics.median(pipeline_seconds_values) if pipeline_seconds_values else 0.0,\n        "first_failure_counts": failure_counts,\n        "per_document_metrics_csv": str(per_document_csv),\n        "failed_documents_jsonl": str(failures_jsonl),\n        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n\n    write_json(analysis_root / "batch_summary.json", summary)\n    write_json(analysis_root / "per_relation_metrics.json", per_relation)\n    write_json(analysis_root / "cumulative_layer_micro_evaluation.json", cumulative)\n    _write_csv_rows(analysis_root / "per_relation_metrics.csv", per_relation)\n    _write_csv_rows(analysis_root / "cumulative_layer_micro_evaluation.csv", cumulative)\n\n    return {\n        "summary": summary,\n        "per_relation": per_relation,\n        "cumulative": cumulative,\n        "failed_preview": failed_preview,\n        "paths": {\n            "per_document_csv": str(per_document_csv),\n            "per_document_jsonl": str(per_document_jsonl),\n            "predictions_jsonl": str(predictions_jsonl),\n            "failed_documents_jsonl": str(failures_jsonl),\n            "batch_summary": str(analysis_root / "batch_summary.json"),\n        },\n    }\n\n\ndef run_batch_streaming(\n    *,\n    dataset_jsonl: str | Path,\n    task_guidance_path: str | Path,\n    batch_root: str | Path,\n    run_all_documents: bool,\n    smoke_document_limit: int,\n    start_index: int,\n    config: BatchRunConfig,\n    api_key: str,\n    matching_record_count: int | None = None,\n    required_type: str = EXACT_REQUIRED_TYPE,\n) -> dict[str, Any]:\n    batch_root = Path(batch_root).resolve()\n    batch_root.mkdir(parents=True, exist_ok=True)\n\n    if matching_record_count is None:\n        matching_record_count = int(count_exact_type_records(\n            dataset_jsonl,\n            required_type=required_type,\n            first_n_ids=0,\n        )["matching_records"])\n    available = max(0, int(matching_record_count) - int(start_index))\n    selected_count = available if run_all_documents else min(int(smoke_document_limit), available)\n    if selected_count <= 0:\n        raise ValueError(\n            f"No records with exact key/value type={required_type!r} remain after start_index={start_index}."\n        )\n\n    selection_manifest = {\n        "orchestrator_version": ORCHESTRATOR_VERSION,\n        "scientific_batch_version": base.BATCH_VERSION,\n        "dataset_jsonl": str(Path(dataset_jsonl).resolve()),\n        "dataset_sha256": sha256_file(dataset_jsonl),\n        "task_guidance_path": str(Path(task_guidance_path).resolve()),\n        "task_guidance_sha256": sha256_file(task_guidance_path),\n        "filter": {"key": "type", "value": required_type, "fallback_to_split": False},\n        "run_all_documents": run_all_documents,\n        "smoke_document_limit": smoke_document_limit,\n        "start_index": start_index,\n        "matching_record_count": matching_record_count,\n        "selected_count": selected_count,\n        "selection_documents_jsonl": str(batch_root / "selection_documents.jsonl"),\n        "memory_policy": {\n            "full_document_list_materialized": False,\n            "maximum_pending_document_jobs": int(config.document_workers),\n        },\n        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(batch_root / "selection_manifest.json", selection_manifest)\n\n    invocation = {\n        **selection_manifest,\n        "config": {**asdict(config), "api_key": "NOT_STORED"},\n        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(batch_root / "batch_invocation.json", invocation)\n\n    documents = iter_prepared_documents_streaming(\n        dataset_jsonl=dataset_jsonl,\n        task_guidance_path=task_guidance_path,\n        batch_root=batch_root,\n        run_all_documents=run_all_documents,\n        smoke_document_limit=smoke_document_limit,\n        required_type=required_type,\n        start_index=start_index,\n    )\n    scheduler = run_documents_bounded_streaming(\n        documents=documents,\n        expected_total=selected_count,\n        config=config,\n        api_key=api_key,\n        batch_root=batch_root,\n    )\n    aggregate = aggregate_batch_results_streaming(\n        batch_root=batch_root,\n        relation_catalog_path=config.relation_catalog_path,\n    )\n\n    invocation["completed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")\n    invocation["scheduler"] = scheduler\n    invocation["documents_completed_or_resumed"] = aggregate["summary"]["documents_completed_or_resumed"]\n    invocation["documents_failed"] = aggregate["summary"]["documents_failed"]\n    write_json(batch_root / "batch_invocation.json", invocation)\n    return {"scheduler": scheduler, **aggregate}\n'}
FROZEN_HELPER_SHA256 = {'docred_native_ablation.py': '3fd93c36722c0ed1b036e545d9da49f40cc9c81aaf0552d3111483e044836221', 'docred_native_ablation_v3.py': 'c89775b5aa598855d16972a65d51d5e1d8ad78ec5ba32710336722f3ab873fad', 'docred_native_ablation_v4.py': 'd6773361c4a40f48d7d6a85662c8f32f625cb1bf46f11924a8b1ba80fbdebf76', 'docred_native_ablation_v5.py': 'b8d5ca10eeed329ba8fa38a54bfe4fe82a38b23b53a4d77f36387169e3bdde91', 'docred_native_batch_v6.py': '3fcf7322fbf78bab1d38d77dbb45f521636acf8085a092151bc231c5c79e7850', 'docred_native_batch_v6_1.py': '6b3b8c38b8ed9a5aca0c5d17981def8d4cdf26682abf1cea6c1016c8618f4f11', 'docred_native_batch_v6_2_dev_streaming.py': '0ec362065f8554b0ffed49bb477a322179c6ae69f2d24dcd81b728476446c500'}

restored = []
verified = []
backed_up = []

for filename, source in FROZEN_HELPER_SOURCES.items():
    target = TOOLS_DIR / filename
    expected_sha = FROZEN_HELPER_SHA256[filename]
    source_sha = hashlib.sha256(source.encode("utf-8")).hexdigest()
    assert source_sha == expected_sha, (filename, source_sha, expected_sha)

    if target.is_file():
        current_sha = hashlib.sha256(target.read_bytes()).hexdigest()
        if current_sha == expected_sha:
            verified.append(filename)
            continue

        # Preserve a differently-versioned local experiment helper before
        # restoring the exact frozen v6.2 version used by this benchmark.
        backup = target.with_suffix(target.suffix + f".pre_v62_backup_{current_sha[:12]}")
        if not backup.exists():
            backup.write_bytes(target.read_bytes())
        backed_up.append((filename, str(backup), current_sha))

    target.write_text(source, encoding="utf-8")
    actual_sha = hashlib.sha256(target.read_bytes()).hexdigest()
    assert actual_sha == expected_sha, (filename, actual_sha, expected_sha)
    restored.append(filename)

print("Frozen helper verification/restoration:")
for name in verified:
    print("  VERIFIED", name, FROZEN_HELPER_SHA256[name][:12])
for name in restored:
    print("  RESTORED", name, FROZEN_HELPER_SHA256[name][:12])
for name, backup, old_sha in backed_up:
    print("  BACKUP  ", name, old_sha[:12], "->", backup)

# Compile all restored helpers before importing any of them.
for filename in FROZEN_HELPER_SOURCES:
    path = TOOLS_DIR / filename
    compile(path.read_text(encoding="utf-8"), str(path), "exec")
print("All frozen helper sources compile: OK")

# Avoid stale imports if a prior failed notebook was run in this kernel.
for module_name in [
    "docred_native_batch_v6_2_dev_streaming",
    "docred_native_batch_v6_1",
    "docred_native_batch_v6",
    "docred_native_ablation_v5",
    "docred_native_ablation_v4",
    "docred_native_ablation_v3",
    "docred_native_ablation",
]:
    sys.modules.pop(module_name, None)

from docred_native_batch_v6_2_dev_streaming import (
    BatchRunConfig,
    aggregate_batch_results_streaming,
    count_exact_type_records,
    read_json,
    run_batch_streaming,
)

mp.freeze_support()

print("\nPROJECT_ROOT =", PROJECT_ROOT)
print("TOOLS_DIR    =", TOOLS_DIR)
print("Frozen DocRED v6.2 toolchain import: OK")
print("No API calls made.")


AssertionError: ('docred_native_ablation_v5.py', 'b9591689be0b61cd546512117b862feea43d8ce8d43f0a022cc0b44460d46762', 'b8d5ca10eeed329ba8fa38a54bfe4fe82a38b23b53a4d77f36387169e3bdde91')

## Frozen DocRED paths and execution configuration

These values match the original v6.2 full-dev runner. The same batch root is critical: this is what lets the scheduler recognize and skip the already completed 978 documents.


In [ ]:
RUN_ALL_DEV_DOCUMENTS = True
EXACT_TYPE_VALUE = "dev"
START_DEV_INDEX = 0

# Only used by the helper API when run_all_documents=False.
SMOKE_DEV_DOCUMENT_LIMIT = 5

DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
    ],
)

# MUST stay identical to the original v6.2 full run.
BATCH_ROOT = NOTEBOOK_DIR / "runs/docred_native_v5_1_dev_streaming"

ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v5.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/docred_task_guidance_v5_1_frozen.json"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"

# Original v6.2 orchestration.
DOCUMENT_WORKERS = 4
LAYER_WORKERS = 16

REASONING_EFFORT = "minimal"
MAX_TOKENS = 4096
REQUEST_TIMEOUT = 120

# These are the critical resume settings.
RESUME_COMPLETED = True
RETRY_FAILED_DOCUMENTS = True

# Same retry behavior as the original runner.
DOCUMENT_ATTEMPTS = 2
RETRY_BACKOFF_SECONDS = 8.0
DOCUMENT_LAUNCH_STAGGER_SECONDS = 0.75
VERBOSE_DOCUMENTS = False
PROGRESS_EVERY = 1

# Paid execution switch.
RUN_RETRY = True

print("BATCH_ROOT =", BATCH_ROOT)
print("MODEL =", MODEL_NAME)
print("DOCUMENT_WORKERS =", DOCUMENT_WORKERS)
print("LAYER_WORKERS =", LAYER_WORKERS)
print("RESUME_COMPLETED =", RESUME_COMPLETED)
print("RETRY_FAILED_DOCUMENTS =", RETRY_FAILED_DOCUMENTS)


## Zero-cost scientific and filesystem preflight


In [ ]:
required = [
    DATASET_JSONL,
    ONTOLOGY_PATH,
    ONTOLOGY_ORIGINAL,
    RELATION_CATALOG,
    RELATION_ALIASES,
    PROFILE_PATH,
    GUIDANCE_PATH,
    TASK_GUIDANCE_PATH,
]

missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

if not BATCH_ROOT.is_dir():
    raise FileNotFoundError(
        "The original DocRED batch root does not exist. "
        "This notebook is resume-only and refuses to create a new experiment:\n"
        f"{BATCH_ROOT}"
    )

profile = read_json(PROFILE_PATH)
task_guidance = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)

corpus_scan = count_exact_type_records(
    DATASET_JSONL,
    required_type=EXACT_TYPE_VALUE,
    first_n_ids=5,
)

matching_records = int(corpus_scan["matching_records"])

print("JSONL records:", corpus_scan["total_records"])
print('Records with exact type="dev":', matching_records)
print("Ontology properties:", catalog["property_count"])
print("Allowed relation IDs:", len(task_guidance["allowed_relation_ids"]))
print("Frozen profile:", profile["profile_name"])

assert EXACT_TYPE_VALUE == "dev"
assert matching_records == 998, (
    "Expected the same 998 exact type=dev records from the original run.",
    matching_records,
)
assert catalog["property_count"] == 96
assert len(task_guidance["allowed_relation_ids"]) == 96

# Anti-cheating / gold-isolation invariants from the frozen profile.
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False

# Freeze the execution shell.
assert DOCUMENT_WORKERS == 4
assert LAYER_WORKERS == 16
assert RESUME_COMPLETED is True
assert RETRY_FAILED_DOCUMENTS is True

print("\nPreflight: OK")
print("No API calls made.")


## Rebuild CURRENT aggregate from saved artifacts — zero API calls

This determines the exact current number of successful and failed documents before any retry.

At the original stopping point this should show **978 completed / 20 failed**.  
The notebook also remains safe if you rerun it after recovering some of those failures.


In [ ]:
before = aggregate_batch_results_streaming(
    batch_root=BATCH_ROOT,
    relation_catalog_path=RELATION_CATALOG,
)

before_summary = before["summary"]

before_requested = int(before_summary["documents_requested"])
before_completed = int(before_summary["documents_completed_or_resumed"])
before_failed = int(before_summary["documents_failed"])

print("CURRENT DOCRED STATE")
print("====================")
print("requested:", before_requested)
print("completed/resumed:", before_completed)
print("failed:", before_failed)

assert before_requested == 998, before_requested
assert before_completed + before_failed == before_requested, (
    before_completed,
    before_failed,
    before_requested,
)
assert before_completed >= 978, (
    "This resume notebook expects the previous full run or a later partial retry.",
    before_completed,
)
assert 0 <= before_failed <= 20, before_failed

failed_path = Path(before["paths"]["failed_documents_jsonl"])
print("Failures JSONL:", failed_path)

def read_jsonl(path: Path):
    rows = []
    if not path.is_file():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

failed_rows_before = read_jsonl(failed_path)

# The aggregate count is authoritative. The failure artifact should agree.
assert len(failed_rows_before) == before_failed, (
    len(failed_rows_before),
    before_failed,
)

if failed_rows_before:
    failed_df = pd.DataFrame(failed_rows_before)
    columns = [
        c for c in [
            "selection_index",
            "source_index",
            "document_id",
            "title",
            "status",
            "error_type",
            "error",
            "run_dir",
        ]
        if c in failed_df.columns
    ]
    display(failed_df[columns])
else:
    print("No failed documents remain.")

display(Markdown(
    f"**Current state:** {before_completed}/998 complete; "
    f"**{before_failed} failed documents eligible for retry**.  \n"
    "The completed documents are protected by `resume_completed=True`."
))


## Paid retry cell — ONLY failed/unresolved documents are eligible

`run_batch_streaming` still scans the dev JSONL to recover source records, but the existing batch state causes completed records to be resumed/skipped.

It does **not** pay to rerun the already-completed documents.


In [ ]:
if before_failed == 0:
    print("DocRED is already 998/998. Paid retry skipped.")
    batch = before

elif not RUN_RETRY:
    print(
        f"RUN_RETRY=False: {before_failed} failed document(s) remain. "
        "No API calls made."
    )
    batch = before

else:
    API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    config = BatchRunConfig(
        project_root=str(PROJECT_ROOT),
        ontology_path=str(ONTOLOGY_PATH),
        profile_path=str(PROFILE_PATH),
        guidance_path=str(GUIDANCE_PATH),
        relation_catalog_path=str(RELATION_CATALOG),
        relation_aliases_path=str(RELATION_ALIASES),
        model_name=MODEL_NAME,
        host=OPENROUTER_HOST,
        document_workers=DOCUMENT_WORKERS,
        layer_workers=LAYER_WORKERS,
        reasoning_effort=REASONING_EFFORT,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        resume_completed=RESUME_COMPLETED,
        retry_failed_documents=RETRY_FAILED_DOCUMENTS,
        document_attempts=DOCUMENT_ATTEMPTS,
        retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
        launch_stagger_seconds=DOCUMENT_LAUNCH_STAGGER_SECONDS,
        verbose_documents=VERBOSE_DOCUMENTS,
        progress_every=PROGRESS_EVERY,
    )

    print(
        f"Retrying unresolved DocRED documents only: {before_failed} currently failed.\n"
        f"Completed documents protected: {before_completed}."
    )

    batch = run_batch_streaming(
        dataset_jsonl=DATASET_JSONL,
        task_guidance_path=TASK_GUIDANCE_PATH,
        batch_root=BATCH_ROOT,
        run_all_documents=True,
        smoke_document_limit=SMOKE_DEV_DOCUMENT_LIMIT,
        start_index=0,
        config=config,
        api_key=API_KEY,
        matching_record_count=matching_records,
        required_type=EXACT_TYPE_VALUE,
    )

    print("Retry invocation finished.")


## Rebuild final aggregate from disk — zero additional API calls

This deliberately rebuilds the aggregate again from persisted document artifacts instead of trusting only the in-memory scheduler result.


In [ ]:
final_batch = aggregate_batch_results_streaming(
    batch_root=BATCH_ROOT,
    relation_catalog_path=RELATION_CATALOG,
)

summary = final_batch["summary"]

requested = int(summary["documents_requested"])
completed = int(summary["documents_completed_or_resumed"])
failed = int(summary["documents_failed"])

print("FINAL/CURRENT DOCRED STATE")
print("==========================")
print("requested:", requested)
print("completed/resumed:", completed)
print("failed:", failed)

assert requested == 998
assert completed + failed == requested

final_metrics = {
    "documents_requested": requested,
    "documents_completed_or_resumed": completed,
    "documents_failed": failed,

    "relation_micro_precision": summary["micro_relation"]["precision"],
    "relation_micro_recall": summary["micro_relation"]["recall"],
    "relation_micro_f1": summary["micro_relation"]["f1"],

    "relation_macro_precision": summary["macro_relation"]["precision"],
    "relation_macro_recall": summary["macro_relation"]["recall"],
    "relation_macro_f1": summary["macro_relation"]["f1"],

    "entity_micro_precision": summary["micro_entity_inventory"]["precision"],
    "entity_micro_recall": summary["micro_entity_inventory"]["recall"],
    "entity_micro_f1": summary["micro_entity_inventory"]["f1"],

    "endpoint_micro_precision": summary["micro_relation_endpoint_inventory"]["precision"],
    "endpoint_micro_recall": summary["micro_relation_endpoint_inventory"]["recall"],
    "endpoint_micro_f1": summary["micro_relation_endpoint_inventory"]["f1"],

    "mean_pipeline_seconds": summary["mean_document_pipeline_seconds"],
    "median_pipeline_seconds": summary["median_document_pipeline_seconds"],
}

display(pd.DataFrame([final_metrics]))

print("\nRELATION MICRO")
print(
    f"P={summary['micro_relation']['precision']:.9f} | "
    f"R={summary['micro_relation']['recall']:.9f} | "
    f"F1={summary['micro_relation']['f1']:.9f}"
)

print("\nENTITY MICRO")
print(
    f"P={summary['micro_entity_inventory']['precision']:.9f} | "
    f"R={summary['micro_entity_inventory']['recall']:.9f} | "
    f"F1={summary['micro_entity_inventory']['f1']:.9f}"
)

print("\nRELATION ENDPOINT MICRO")
print(
    f"P={summary['micro_relation_endpoint_inventory']['precision']:.9f} | "
    f"R={summary['micro_relation_endpoint_inventory']['recall']:.9f} | "
    f"F1={summary['micro_relation_endpoint_inventory']['f1']:.9f}"
)

failed_rows_after = read_jsonl(Path(final_batch["paths"]["failed_documents_jsonl"]))

if failed_rows_after:
    print(f"\nStill failed: {len(failed_rows_after)}")
    failed_after_df = pd.DataFrame(failed_rows_after)
    columns = [
        c for c in [
            "selection_index",
            "source_index",
            "document_id",
            "title",
            "status",
            "error_type",
            "error",
            "run_dir",
        ]
        if c in failed_after_df.columns
    ]
    display(failed_after_df[columns])
    print(
        "\nRerun this notebook to retry ONLY those remaining failures. "
        "Newly recovered documents will be skipped next time."
    )
else:
    print("\nSUCCESS: DocRED is now 998/998 with zero failed documents.")


## Exact relation counts + final exports

This reads the aggregate files generated by the existing DocRED evaluator, including TP/FP/FN/predicted/gold relation totals where available.


In [ ]:
per_document_path = Path(final_batch["paths"]["per_document_csv"])
per_document = (
    pd.read_csv(per_document_path)
    if per_document_path.is_file()
    else pd.DataFrame()
)

cumulative_path = BATCH_ROOT / "aggregate_analysis/cumulative_layer_micro_evaluation.csv"
cumulative = (
    pd.read_csv(cumulative_path)
    if cumulative_path.is_file()
    else pd.DataFrame()
)

if not cumulative.empty:
    final_layer = cumulative.sort_values("layer_index").iloc[-1]
    print("FINAL LAYER RELATION COUNTS")
    for key in [
        "predicted",
        "gold",
        "true_positive",
        "false_positive",
        "false_negative",
        "precision",
        "recall",
        "f1",
    ]:
        if key in final_layer:
            print(f"{key}: {final_layer[key]}")

if not per_document.empty:
    print("\nPER-DOCUMENT RUNTIME")
    if "pipeline_seconds" in per_document.columns:
        print("documents:", len(per_document))
        print("mean:", float(per_document["pipeline_seconds"].mean()))
        print("median:", float(per_document["pipeline_seconds"].median()))
        print("min:", float(per_document["pipeline_seconds"].min()))
        print("max:", float(per_document["pipeline_seconds"].max()))

FINAL_REPORT_PATH = BATCH_ROOT / "aggregate_analysis/docred_final_resume20_report.json"
FINAL_REPORT_PATH.write_text(
    json.dumps(
        {
            "model": MODEL_NAME,
            "batch_root": str(BATCH_ROOT),
            "retry_started_from": {
                "completed": before_completed,
                "failed": before_failed,
            },
            "final": final_metrics,
            "remaining_failed_documents": failed_rows_after,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nSaved final report:", FINAL_REPORT_PATH)
print("Batch summary:", final_batch["paths"]["batch_summary"])
print("Per-document CSV:", final_batch["paths"]["per_document_csv"])
print("Per-relation CSV:", BATCH_ROOT / "aggregate_analysis/per_relation_metrics.csv")
print("Cumulative layer CSV:", cumulative_path)
print("Failures JSONL:", final_batch["paths"]["failed_documents_jsonl"])
